In [1]:
import os

os.environ["PYTORCH_MIOPEN_SUGGEST_NHWC_BATCHNORM"] = "0"
os.environ["MIOPEN_FIND_MODE"] = "FAST"

import torch

print(torch.__version__)
print(torch.version.hip)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.9.1+rocm7.2.1
7.2.53211-158bd99533
True
AMD Radeon RX 9060 XT


In [2]:
from __future__ import annotations

# =============================================================================
# CONFIGURACIÓN DE RECURSOS — Ryzen 5 5600X, 16 GB RAM y GPU AMD de 16 GB
# Debe definirse antes de importar NumPy, PyTorch, GDAL o Rasterio.
# =============================================================================
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("GDAL_CACHEMAX", "512")
# Evita la ruta NHWC de BatchNorm en MIOpen. El codificador se convierte
# además a GroupNorm para trabajar de forma estable con lotes pequeños.
os.environ.setdefault("PYTORCH_MIOPEN_SUGGEST_NHWC_BATCHNORM", "0")
os.environ.setdefault("MIOPEN_FIND_MODE", "FAST")
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:128",
)

import gc
import json
import os
import random
import re
import shutil
import time
import warnings
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import models
from tqdm import tqdm

# =============================================================================
# 1. RUTAS — MODIFICAR ÚNICAMENTE ESTA SECCIÓN
# =============================================================================
DATASET_ROOT = Path(r"C:\Users\Ivan sua\anaconda3\Dataset")
METADATA_DIR = DATASET_ROOT / "metadata"
RUNS_DIR = Path(r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion")

# Revisa ambos niveles en orden. Cuando un nivel ya tiene resultados finales,
# se omite automáticamente y se continúa con el siguiente.
LEVELS_TO_TRAIN = ["nivel_1", "nivel_2"]
SKIP_COMPLETED_LEVELS = True

CREATE_NEW_RUNS = False  # Reanuda la última ejecución disponible
RESUME_BY_LEVEL = {"nivel_1": None, "nivel_2": None}

REQUIRE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Uso controlado del Ryzen 5 5600X sin saturar la RAM.
torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_float32_matmul_precision("high")
SEED = 42
DETERMINISTIC = False

# Misma unidad de salida y mismas métricas que U-Net.
EPOCHS = 120
# Configuración estable para 16 GB de VRAM.
BATCH_SIZE_BY_LEVEL = {"nivel_1": 1, "nivel_2": 1}
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
OPTIMIZER_TYPE = "adamw"
# Ejecutar preferiblemente como archivo .py desde Anaconda Prompt.
NUM_WORKERS = 0
PREFETCH_FACTOR = None
PIN_MEMORY = False
PERSISTENT_WORKERS = False
EARLY_STOPPING_PATIENCE = 20
USE_AMP = False
GRADIENT_ACCUMULATION_BY_LEVEL = {"nivel_1": 4, "nivel_2": 4}

# Pausas de estabilización para reducir la carga sostenida del controlador.
EPOCH_COOLDOWN_SECONDS = 3
LEVEL_COOLDOWN_SECONDS = 20
PROGRESS_UPDATE_EVERY = 20

ENCODER_PRETRAINED = True
FREEZE_ENCODER = False
DECODER_DROPOUT = 0.10

BAND_INDEXES = (1, 2, 3, 4)
EXPECTED_CHIP_BANDS = 4
REFLECTANCE_SCALE = 10000.0
REFLECTANCE_ADD_OFFSET = 0.0
USE_RASTER_SCALE_OFFSET = True
AUTO_SCALE_REFLECTANCE = True
STATS_SAMPLE_SIZE = 300

MASK_NODATA_CODE = 0
IGNORE_INDEX = -100
LOSS_TYPE = "ce_dice"
DICE_WEIGHT = 0.7
USE_CLASS_WEIGHTS = True
USE_WEIGHTED_SAMPLER = False
USE_AUGMENTATION = True

SAVE_EVERY_EPOCH = False
SAVE_TEST_PREDICTIONS = False
MAX_TEST_PREDICTIONS_TO_SAVE = 20
SAVE_REPORT_PLOTS = False
PLOT_AFTER_TRAINING = False


CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}

CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}

LEVEL_FOLDER_BY_NAME = {
    "nivel_1": "Nivel_1",
    "nivel_2": "Nivel_2",
    "metadata": "metadata",
}

def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(f"Nivel no reconocido: {level_name}")

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = DETERMINISTIC
        torch.backends.cudnn.benchmark = not DETERMINISTIC


def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(f"Objeto no serializable: {type(value).__name__}")


def save_json(data, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False, default=json_default)


def safe_name(text: str) -> str:
    normalized = (
        str(text)
        .translate(str.maketrans("áéíóúÁÉÍÓÚñÑ", "aeiouAEIOUnN"))
    )
    normalized = re.sub(r"[^A-Za-z0-9]+", "_", normalized)
    return re.sub(r"_+", "_", normalized).strip("_").lower()


def resolve_case_insensitive(path: Path) -> Path:
    if path.exists():
        return path

    parts = path.parts
    if not parts:
        return path

    current = Path(parts[0])
    for part in parts[1:]:
        if current.exists() and current.is_dir():
            try:
                match = next(
                    (item.name for item in current.iterdir() if item.name.lower() == part.lower()),
                    part,
                )
            except OSError:
                match = part
            current = current / match
        else:
            current = current / part
    return current


def resolve_dataset_path(value) -> str:
    if pd.isna(value):
        return ""

    raw = os.path.expandvars(os.path.expanduser(str(value).strip().strip('"').strip("'")))
    if not raw:
        return ""

    direct = resolve_case_insensitive(Path(raw))
    if direct.exists():
        return str(direct)

    parts = [part for part in raw.replace("\\", "/").split("/") if part not in {"", "."}]
    lower_parts = [part.lower() for part in parts]

    if "dataset" in lower_parts:
        index = len(lower_parts) - 1 - lower_parts[::-1].index("dataset")
        candidate = DATASET_ROOT.joinpath(*parts[index + 1 :])
    else:
        candidate = DATASET_ROOT.joinpath(*parts)

    relative_parts = list(candidate.relative_to(DATASET_ROOT).parts)
    if relative_parts:
        relative_parts[0] = LEVEL_FOLDER_BY_NAME.get(relative_parts[0].lower(), relative_parts[0])
    return str(resolve_case_insensitive(DATASET_ROOT.joinpath(*relative_parts)))


def get_splits_dir(level_name: str) -> Path:
    candidates = [
        METADATA_DIR / "splits" / level_name,
        METADATA_DIR / "splits" / LEVEL_FOLDER_BY_NAME.get(level_name, level_name),
    ]
    return next((path for path in candidates if path.exists()), candidates[0])


def find_split_file(splits_dir: Path, *names: str) -> Path:
    if not splits_dir.exists():
        raise FileNotFoundError(f"No existe la carpeta de splits: {splits_dir}")

    available = {item.name.lower(): item for item in splits_dir.iterdir() if item.is_file()}
    for name in names:
        if name.lower() in available:
            return available[name.lower()]

    raise FileNotFoundError(f"No se encontró ninguno de {names} en {splits_dir}")


def read_split_csv(csv_path: Path, split_name: str) -> pd.DataFrame:
    if not csv_path.exists() or csv_path.stat().st_size == 0:
        raise FileNotFoundError(f"CSV inexistente o vacío: {csv_path}")

    df = pd.read_csv(csv_path)
    columns = {str(column).strip().lower(): column for column in df.columns}

    image_column = next(
        (columns[name] for name in ("image_path", "image", "img_path", "img") if name in columns),
        None,
    )
    mask_column = next(
        (columns[name] for name in ("mask_path", "label_path", "mask", "label") if name in columns),
        None,
    )

    if image_column is None or mask_column is None:
        raise ValueError(
            f"El split {csv_path} debe contener image_path y mask_path; "
            f"columnas disponibles: {list(df.columns)}"
        )

    df = df.rename(columns={image_column: "image_path", mask_column: "mask_path"})
    df = df.dropna(subset=["image_path", "mask_path"]).copy()
    df["image_path"] = df["image_path"].apply(resolve_dataset_path)
    df["mask_path"] = df["mask_path"].apply(resolve_dataset_path)

    duplicate_count = int(df.duplicated(subset=["image_path", "mask_path"]).sum())
    if duplicate_count:
        warnings.warn(
            f"{split_name}: se eliminaron {duplicate_count} pares duplicados.",
            RuntimeWarning,
        )
        df = df.drop_duplicates(subset=["image_path", "mask_path"], keep="first")

    missing_images = [path for path in df["image_path"] if not Path(path).exists()]
    missing_masks = [path for path in df["mask_path"] if not Path(path).exists()]
    if missing_images or missing_masks:
        raise FileNotFoundError(
            f"{split_name}: faltan {len(missing_images)} imágenes y {len(missing_masks)} máscaras."
        )

    if df.empty:
        raise ValueError(f"El split {split_name} quedó vacío después de la validación.")

    return df.reset_index(drop=True)


def normalized_pair_set(df: pd.DataFrame) -> set[tuple[str, str]]:
    return {
        (
            str(Path(image)).replace("\\", "/").lower(),
            str(Path(mask)).replace("\\", "/").lower(),
        )
        for image, mask in zip(df["image_path"], df["mask_path"])
    }


def lineage_set(df: pd.DataFrame) -> set[str]:
    columns = {str(column).strip().lower(): column for column in df.columns}
    for candidate in ("source_chip_id", "original_chip_id", "chip_id"):
        if candidate in columns:
            values = df[columns[candidate]].dropna().astype(str).str.strip()
            return {value.lower() for value in values if value}
    return set()


def validate_split_disjointness(train: pd.DataFrame, val: pd.DataFrame, test: pd.DataFrame) -> None:
    pair_sets = {
        "train": normalized_pair_set(train),
        "val": normalized_pair_set(val),
        "test": normalized_pair_set(test),
    }
    pair_overlaps = {
        "train-val": pair_sets["train"] & pair_sets["val"],
        "train-test": pair_sets["train"] & pair_sets["test"],
        "val-test": pair_sets["val"] & pair_sets["test"],
    }

    lineage_sets = {
        "train": lineage_set(train),
        "val": lineage_set(val),
        "test": lineage_set(test),
    }
    lineage_overlaps = {
        "train-val": lineage_sets["train"] & lineage_sets["val"],
        "train-test": lineage_sets["train"] & lineage_sets["test"],
        "val-test": lineage_sets["val"] & lineage_sets["test"],
    }

    conflicts = {name: len(values) for name, values in pair_overlaps.items() if values}
    lineage_conflicts = {
        name: len(values)
        for name, values in lineage_overlaps.items()
        if values
    }

    if conflicts or lineage_conflicts:
        details = []
        if conflicts:
            details.append(
                "pares repetidos: "
                + ", ".join(f"{name}={count}" for name, count in conflicts.items())
            )
        if lineage_conflicts:
            details.append(
                "chips originales compartidos entre splits: "
                + ", ".join(f"{name}={count}" for name, count in lineage_conflicts.items())
            )
        raise ValueError(
            "Se detectó fuga de información entre entrenamiento, validación y prueba; "
            + "; ".join(details)
            + ". Los aumentos de un mismo chip deben permanecer en un único split."
        )


def validate_image_mask_pairs(
    dataframes: dict[str, pd.DataFrame],
    class_dict: dict[int, str],
) -> tuple[int, tuple[int, int]]:
    expected_shape = None
    observed_classes = set()

    for split_name, dataframe in dataframes.items():
        for image_path, mask_path in tqdm(
            zip(dataframe["image_path"], dataframe["mask_path"]),
            total=len(dataframe),
            desc=f"Validando {split_name}",
        ):
            with rasterio.open(image_path) as image_src, rasterio.open(mask_path) as mask_src:
                if image_src.count != EXPECTED_CHIP_BANDS:
                    raise ValueError(
                        f"{image_path} tiene {image_src.count} bandas; se esperaban {EXPECTED_CHIP_BANDS}."
                    )
                if max(BAND_INDEXES) > image_src.count:
                    raise ValueError(f"Índices de banda inválidos para {image_path}.")
                if (image_src.height, image_src.width) != (mask_src.height, mask_src.width):
                    raise ValueError(f"Dimensiones incompatibles: {image_path} y {mask_path}.")
                if image_src.crs != mask_src.crs:
                    raise ValueError(f"CRS incompatible: {image_path} y {mask_path}.")
                if not image_src.transform.almost_equals(mask_src.transform):
                    raise ValueError(f"Transformación espacial incompatible: {image_path} y {mask_path}.")

                shape = (image_src.height, image_src.width)
                if expected_shape is None:
                    expected_shape = shape
                elif shape != expected_shape:
                    raise ValueError(
                        f"Todos los chips de un nivel deben tener el mismo tamaño; "
                        f"se detectaron {expected_shape} y {shape}."
                    )

                mask = mask_src.read(1)
                observed_classes.update(int(value) for value in np.unique(mask))

    allowed = set(class_dict)
    unexpected = observed_classes - allowed
    if unexpected:
        raise ValueError(f"Las máscaras contienen códigos no definidos: {sorted(unexpected)}.")

    train_classes = set()
    for mask_path in dataframes["train"]["mask_path"]:
        with rasterio.open(mask_path) as src:
            train_classes.update(int(value) for value in np.unique(src.read(1)))
    missing_train = (allowed - {MASK_NODATA_CODE}) - train_classes
    if missing_train:
        raise ValueError(
            f"Clases sin píxeles en entrenamiento: {sorted(missing_train)}."
        )

    return EXPECTED_CHIP_BANDS, expected_shape


def read_splits(level_name: str, class_dict: dict[int, str]):
    splits_dir = get_splits_dir(level_name)
    train = read_split_csv(
        find_split_file(splits_dir, "train_samples.csv", "train.csv"),
        "train",
    )
    val = read_split_csv(
        find_split_file(splits_dir, "val_samples.csv", "val.csv"),
        "val",
    )
    test = read_split_csv(
        find_split_file(splits_dir, "test_samples.csv", "test.csv"),
        "test",
    )

    validate_split_disjointness(train, val, test)
    band_count, chip_shape = validate_image_mask_pairs(
        {"train": train, "val": val, "test": test},
        class_dict,
    )
    return train, val, test, splits_dir, band_count, chip_shape

def read_reflectance(path: str | Path, band_indexes=None):
    band_indexes = BAND_INDEXES if band_indexes is None else band_indexes

    with rasterio.open(path) as src:
        array = src.read(indexes=band_indexes).astype(np.float32)
        nodata = src.nodata
        scales = np.asarray(
            [src.scales[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]
        offsets = np.asarray(
            [src.offsets[index - 1] for index in band_indexes],
            dtype=np.float32,
        )[:, None, None]

    finite = np.isfinite(array)
    if nodata is not None and np.isfinite(nodata):
        finite &= array != nodata

    valid_pixel = finite.all(axis=0)
    array = np.where(finite, array, np.nan)

    finite_values = array[np.isfinite(array)]
    if finite_values.size == 0:
        raise ValueError(f"La imagen no contiene valores válidos: {path}")

    has_raster_transform = (
        USE_RASTER_SCALE_OFFSET
        and (
            not np.allclose(scales, 1.0)
            or not np.allclose(offsets, 0.0)
        )
    )
    if has_raster_transform:
        array = array * scales + offsets
    elif AUTO_SCALE_REFLECTANCE and float(np.nanpercentile(finite_values, 99.5)) > 2.0:
        array = (
            array + float(REFLECTANCE_ADD_OFFSET)
        ) / float(REFLECTANCE_SCALE)

    return np.clip(array, 0.0, 1.0).astype(np.float32), valid_pixel


class LULCSegmentationDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        code_to_idx: dict[int, int],
        means: list[float],
        stds: list[float],
        augment: bool = False,
    ):
        self.df = dataframe.reset_index(drop=True)
        self.code_to_idx = code_to_idx
        self.means = np.asarray(means, dtype=np.float32)[:, None, None]
        self.stds = np.asarray(stds, dtype=np.float32)[:, None, None]
        self.augment = augment

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, index: int):
        row = self.df.iloc[index]
        image, valid_pixel = read_reflectance(row["image_path"])

        with rasterio.open(row["mask_path"]) as src:
            original_mask = src.read(1).astype(np.int64)

        if image.shape[1:] != original_mask.shape:
            raise ValueError(
                f"Dimensiones incompatibles: {row['image_path']} y {row['mask_path']}."
            )

        mask = np.full(original_mask.shape, IGNORE_INDEX, dtype=np.int64)
        for original_code, internal_index in self.code_to_idx.items():
            mask[(original_mask == original_code) & valid_pixel] = internal_index

        if self.augment:
            if random.random() < 0.5:
                image = np.flip(image, axis=2)
                mask = np.flip(mask, axis=1)
            if random.random() < 0.5:
                image = np.flip(image, axis=1)
                mask = np.flip(mask, axis=0)
            rotation = random.randint(0, 3)
            if rotation:
                image = np.rot90(image, rotation, axes=(1, 2))
                mask = np.rot90(mask, rotation, axes=(0, 1))

        valid = np.isfinite(image).all(axis=0) & (mask != IGNORE_INDEX)
        image = np.nan_to_num(image, nan=0.0)
        image = (image - self.means) / np.maximum(self.stds, 1e-6)
        image[:, ~valid] = 0.0

        return {
            "image": torch.from_numpy(np.ascontiguousarray(image)).float(),
            "mask": torch.from_numpy(np.ascontiguousarray(mask)).long(),
            "image_path": row["image_path"],
            "mask_path": row["mask_path"],
        }


def compute_image_stats(
    dataframe: pd.DataFrame,
    sample_size: int | None = None,
) -> tuple[list[float], list[float]]:
    sample_size = STATS_SAMPLE_SIZE if sample_size is None else sample_size
    paths = dataframe["image_path"].tolist()

    if sample_size and len(paths) > sample_size:
        rng = np.random.default_rng(SEED)
        paths = rng.choice(paths, size=sample_size, replace=False).tolist()

    channel_sum = None
    channel_squared_sum = None
    channel_count = None

    for path in tqdm(paths, desc="Estadísticas radiométricas"):
        array, _ = read_reflectance(path)
        flat = array.reshape(array.shape[0], -1)
        valid = np.isfinite(flat)

        sums = np.nansum(flat, axis=1)
        squared_sums = np.nansum(flat * flat, axis=1)
        counts = valid.sum(axis=1)

        if channel_sum is None:
            channel_sum = sums
            channel_squared_sum = squared_sums
            channel_count = counts
        else:
            channel_sum += sums
            channel_squared_sum += squared_sums
            channel_count += counts

    if channel_sum is None or np.any(channel_count == 0):
        raise RuntimeError("No fue posible calcular las estadísticas radiométricas.")

    means = channel_sum / channel_count
    variances = channel_squared_sum / channel_count - means**2
    stds = np.sqrt(np.maximum(variances, 1e-12))
    return means.astype(float).tolist(), stds.astype(float).tolist()


def compute_class_pixel_counts(
    dataframe: pd.DataFrame,
    code_to_idx: dict[int, int],
) -> np.ndarray:
    counts = np.zeros(len(code_to_idx), dtype=np.int64)

    iterator = dataframe[["image_path", "mask_path"]].itertuples(index=False)
    for image_path, mask_path in tqdm(
        iterator,
        total=len(dataframe),
        desc="Píxeles por clase",
    ):
        _, valid_pixel = read_reflectance(image_path)
        with rasterio.open(mask_path) as src:
            mask = src.read(1)

        if mask.shape != valid_pixel.shape:
            raise ValueError(
                f"Dimensiones incompatibles al contar clases: {image_path} y {mask_path}."
            )

        for original_code, internal_index in code_to_idx.items():
            counts[internal_index] += int(
                ((mask == original_code) & valid_pixel).sum()
            )

    return counts


def make_class_weights(counts: np.ndarray) -> torch.Tensor:
    if np.any(counts == 0):
        missing = np.where(counts == 0)[0].tolist()
        raise ValueError(f"Clases internas sin píxeles de entrenamiento: {missing}")

    frequencies = counts / counts.sum()
    weights = 1.0 / np.sqrt(frequencies)
    weights /= weights.mean()
    return torch.tensor(weights, dtype=torch.float32)


def build_weighted_sampler(dataframe: pd.DataFrame):
    if not USE_WEIGHTED_SAMPLER:
        return None
    if "class_code" not in dataframe.columns:
        warnings.warn(
            "No existe class_code; se omite el muestreo balanceado por chip.",
            RuntimeWarning,
        )
        return None

    codes = pd.to_numeric(dataframe["class_code"], errors="coerce").fillna(-1).astype(int)
    counts = codes.value_counts().to_dict()
    sample_weights = codes.map(lambda code: 1.0 / counts[int(code)]).to_numpy(dtype=np.float64)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    return WeightedRandomSampler(
        torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )


def save_dataset_summary(
    train: pd.DataFrame,
    val: pd.DataFrame,
    test: pd.DataFrame,
    pixel_counts: np.ndarray,
    idx_to_code: dict[int, int],
    class_dict: dict[int, str],
    output_dir: Path,
) -> None:
    output_dir = ensure_dir(output_dir)

    split_summary = pd.DataFrame(
        {
            "split": ["train", "val", "test"],
            "chips": [len(train), len(val), len(test)],
        }
    )
    split_summary["percentage"] = (
        split_summary["chips"] / split_summary["chips"].sum() * 100.0
    )
    split_summary.to_csv(
        output_dir / "dataset_splits.csv",
        index=False,
        encoding="utf-8-sig",
    )

    pixel_summary = pd.DataFrame(
        {
            "class_index": np.arange(len(pixel_counts)),
            "class_code": [idx_to_code[index] for index in range(len(pixel_counts))],
            "class_name": [
                class_dict[idx_to_code[index]]
                for index in range(len(pixel_counts))
            ],
            "pixels_train": pixel_counts,
        }
    )
    pixel_summary["percentage_train"] = (
        pixel_summary["pixels_train"] / max(pixel_summary["pixels_train"].sum(), 1) * 100.0
    )
    pixel_summary.to_csv(
        output_dir / "dataset_pixels_por_clase.csv",
        index=False,
        encoding="utf-8-sig",
    )

class ConvNormAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        groups = min(32, out_channels)
        while out_channels % groups != 0:
            groups -= 1
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = ConvNormAct(in_channels + skip_channels, out_channels, dropout)

    def forward(self, decoder, skip):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([decoder, skip], dim=1))



def _valid_group_count(channels: int, maximum_groups: int = 32) -> int:
    """Selecciona el mayor número de grupos que divide exactamente los canales."""
    for groups in (maximum_groups, 16, 8, 4, 2, 1):
        if groups <= channels and channels % groups == 0:
            return groups
    return 1


def replace_batchnorm_with_groupnorm(module: nn.Module) -> nn.Module:
    """
    Sustituye recursivamente BatchNorm2d por GroupNorm.

    Esta conversión evita la operación BatchNorm de MIOpen que produjo
    miopenStatusUnknownError y es más estable con batch size 1–4.
    Los parámetros afines de los pesos preentrenados se conservan.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            replacement = nn.GroupNorm(
                num_groups=_valid_group_count(child.num_features),
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(child.weight)
                    replacement.bias.copy_(child.bias)
            setattr(module, name, replacement)
        else:
            replace_batchnorm_with_groupnorm(child)
    return module


def adapt_first_convolution(old_conv: nn.Conv2d, in_channels: int, pretrained_loaded: bool) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        if pretrained_loaded:
            # Entrada Sentinel-2: B2, B3, B4, B8. Los pesos ImageNet vienen en RGB.
            if in_channels >= 3:
                new_conv.weight[:, 0] = old_conv.weight[:, 2]  # B2 <- azul
                new_conv.weight[:, 1] = old_conv.weight[:, 1]  # B3 <- verde
                new_conv.weight[:, 2] = old_conv.weight[:, 0]  # B4 <- rojo
                if in_channels > 3:
                    mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
                    new_conv.weight[:, 3:] = mean_weight.repeat(1, in_channels - 3, 1, 1)
            else:
                new_conv.weight[:, :in_channels] = old_conv.weight[:, :in_channels]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
    return new_conv


class ResNet50UNet(nn.Module):
    """Segmentación semántica con ResNet-50 como codificador y decodificador tipo U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        pretrained: bool = True,
        freeze_encoder: bool = False,
        decoder_dropout: float = 0.10,
    ):
        super().__init__()
        pretrained_loaded = False
        weights = None
        if pretrained:
            try:
                weights = models.ResNet50_Weights.DEFAULT
                encoder = models.resnet50(weights=weights)
                pretrained_loaded = True
            except Exception as exc:
                warnings.warn(
                    f"No fue posible cargar pesos ImageNet de ResNet-50 ({exc}); se inicializa sin pesos.",
                    RuntimeWarning,
                )
                encoder = models.resnet50(weights=None)
        else:
            encoder = models.resnet50(weights=None)

        encoder.conv1 = adapt_first_convolution(
            encoder.conv1,
            in_channels,
            pretrained_loaded,
        )
        encoder = replace_batchnorm_with_groupnorm(encoder)

        self.conv1 = encoder.conv1
        self.bn1 = encoder.bn1
        self.relu = encoder.relu
        self.maxpool = encoder.maxpool
        self.layer1 = encoder.layer1   # 256 canales, 1/4
        self.layer2 = encoder.layer2   # 512 canales, 1/8
        self.layer3 = encoder.layer3   # 1024 canales, 1/16
        self.layer4 = encoder.layer4   # 2048 canales, 1/32

        self.decoder4 = DecoderBlock(2048, 1024, 512, decoder_dropout)
        self.decoder3 = DecoderBlock(512, 512, 256, decoder_dropout)
        self.decoder2 = DecoderBlock(256, 256, 128, decoder_dropout)
        self.decoder1 = DecoderBlock(128, 64, 64, decoder_dropout)
        self.final_refine = ConvNormAct(64, 32, decoder_dropout)
        self.segmentation_head = nn.Conv2d(32, num_classes, kernel_size=1)
        self.pretrained_loaded = pretrained_loaded

        if freeze_encoder:
            for module in (
                self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4
            ):
                for parameter in module.parameters():
                    parameter.requires_grad = False

    def forward(self, x):
        input_size = x.shape[-2:]
        skip0 = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(skip0)
        skip1 = self.layer1(x)
        skip2 = self.layer2(skip1)
        skip3 = self.layer3(skip2)
        x = self.layer4(skip3)

        x = self.decoder4(x, skip3)
        x = self.decoder3(x, skip2)
        x = self.decoder2(x, skip1)
        x = self.decoder1(x, skip0)
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        x = self.final_refine(x)
        return self.segmentation_head(x)


def build_segmentation_model(
    in_channels: int,
    num_classes: int,
    pretrained: bool = ENCODER_PRETRAINED,
    freeze_encoder: bool = FREEZE_ENCODER,
):
    model = ResNet50UNet(
        in_channels=in_channels,
        num_classes=num_classes,
        pretrained=pretrained,
        freeze_encoder=freeze_encoder,
        decoder_dropout=DECODER_DROPOUT,
    )
    return model, bool(model.pretrained_loaded)

class DiceLoss(nn.Module):
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX, smooth: float = 1.0):
        super().__init__()
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.smooth = smooth

    def forward(self, logits, target):
        probabilities = torch.softmax(logits, dim=1)
        valid = target != self.ignore_index
        losses = []

        for class_index in range(self.num_classes):
            if class_index == self.ignore_index:
                continue

            truth = ((target == class_index) & valid).float()
            if truth.sum() == 0:
                continue

            prediction = probabilities[:, class_index] * valid
            intersection = (prediction * truth).sum()
            denominator = prediction.sum() + truth.sum()
            dice = (2.0 * intersection + self.smooth) / (denominator + self.smooth)
            losses.append(1.0 - dice)

        return torch.stack(losses).mean() if losses else logits.sum() * 0.0


class CombinedSegmentationLoss(nn.Module):
    def __init__(
        self,
        num_classes: int,
        class_weights: torch.Tensor | None,
    ):
        super().__init__()
        self.cross_entropy = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=IGNORE_INDEX,
        )
        self.dice = DiceLoss(num_classes, IGNORE_INDEX)

    def forward(self, logits, target):
        ce_loss = self.cross_entropy(logits, target)
        if LOSS_TYPE == "weighted_ce":
            return ce_loss
        if LOSS_TYPE == "ce_dice":
            return ce_loss + DICE_WEIGHT * self.dice(logits, target)
        raise ValueError(f"LOSS_TYPE no reconocido: {LOSS_TYPE}")


class ConfusionAccumulator:
    def __init__(self, num_classes: int, ignore_index: int = IGNORE_INDEX):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.matrix = torch.zeros((num_classes, num_classes), dtype=torch.int64)

    @torch.no_grad()
    def update(self, logits, target) -> None:
        prediction = logits.argmax(dim=1)
        valid = (
            (target != self.ignore_index)
            & (target >= 0)
            & (target < self.num_classes)
            & (prediction >= 0)
            & (prediction < self.num_classes)
        )
        encoded = self.num_classes * target[valid] + prediction[valid]
        counts = torch.bincount(
            encoded.detach().cpu(),
            minlength=self.num_classes**2,
        )
        self.matrix += counts.reshape(self.num_classes, self.num_classes)


def metrics_from_confusion(matrix: np.ndarray, ignore_index: int = IGNORE_INDEX):
    matrix = matrix.astype(np.float64)
    true_positive = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    total = matrix.sum()

    precision = np.divide(
        true_positive,
        predicted,
        out=np.zeros_like(true_positive),
        where=predicted > 0,
    )
    recall = np.divide(
        true_positive,
        support,
        out=np.zeros_like(true_positive),
        where=support > 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) > 0,
    )
    iou = np.divide(
        true_positive,
        support + predicted - true_positive,
        out=np.zeros_like(true_positive),
        where=(support + predicted - true_positive) > 0,
    )

    active = (support > 0) & (np.arange(len(support)) != ignore_index)
    pixel_accuracy = float(true_positive.sum() / total) if total else 0.0
    expected = float((support * predicted).sum() / (total**2)) if total else 0.0
    kappa = (
        (pixel_accuracy - expected) / (1.0 - expected)
        if total and expected < 1.0
        else 0.0
    )

    summary = {
        "pixel_accuracy": pixel_accuracy,
        "macro_precision": float(precision[active].mean()) if active.any() else 0.0,
        "macro_recall": float(recall[active].mean()) if active.any() else 0.0,
        "macro_f1": float(f1[active].mean()) if active.any() else 0.0,
        "mean_iou": float(iou[active].mean()) if active.any() else 0.0,
        "kappa": float(kappa),
    }
    per_class = pd.DataFrame(
        {
            "class_code": np.arange(len(support)),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
            "support": support.astype(np.int64),
        }
    )
    return summary, per_class


def amp_enabled() -> bool:
    return bool(USE_AMP and DEVICE.type == "cuda")


def autocast_context():
    if amp_enabled():
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=amp_enabled())
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=amp_enabled())


def train_one_epoch(model, loader, optimizer, criterion, scaler, epoch, num_classes, gradient_accumulation_steps):
    model.train()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        loader,
        desc=f"Entrenamiento {epoch}",
        leave=False,
        mininterval=5.0,
        dynamic_ncols=False,
    )
    for step, batch in enumerate(progress, start=1):
        images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
        masks = batch["mask"].to(DEVICE, non_blocking=PIN_MEMORY)

        group_start = (
            (step - 1) // gradient_accumulation_steps
        ) * gradient_accumulation_steps
        group_size = min(
            gradient_accumulation_steps,
            len(loader) - group_start,
        )

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)
            scaled_loss = loss / group_size

        if amp_enabled():
            scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        should_step = (
            step % gradient_accumulation_steps == 0
            or step == len(loader)
        )
        if should_step:
            if amp_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        valid_pixels = int((masks != IGNORE_INDEX).sum().item())
        total_loss += float(loss.item()) * max(valid_pixels, 1)
        total_valid_pixels += valid_pixels
        accumulator.update(logits.detach(), masks)

        if step % PROGRESS_UPDATE_EVERY == 0 or step == len(loader):
            metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
            progress.set_postfix(
                loss=f"{total_loss / max(total_valid_pixels, 1):.4f}",
                mIoU=f"{metrics['mean_iou']:.4f}",
                refresh=False,
            )

        del images, masks, logits, loss, scaled_loss

    progress.close()
    metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
    metrics["loss"] = total_loss / max(total_valid_pixels, 1)
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion, split_name, num_classes):
    model.eval()
    accumulator = ConfusionAccumulator(num_classes)
    total_loss = 0.0
    total_valid_pixels = 0

    progress = tqdm(
        loader,
        desc=split_name,
        leave=False,
        mininterval=5.0,
        dynamic_ncols=False,
    )
    for step, batch in enumerate(progress, start=1):
        images = batch["image"].to(DEVICE, non_blocking=PIN_MEMORY)
        masks = batch["mask"].to(DEVICE, non_blocking=PIN_MEMORY)

        with autocast_context():
            logits = model(images)
            loss = criterion(logits, masks)

        valid_pixels = int((masks != IGNORE_INDEX).sum().item())
        total_loss += float(loss.item()) * max(valid_pixels, 1)
        total_valid_pixels += valid_pixels
        accumulator.update(logits, masks)

        if step % PROGRESS_UPDATE_EVERY == 0 or step == len(loader):
            metrics, _ = metrics_from_confusion(accumulator.matrix.numpy())
            progress.set_postfix(
                loss=f"{total_loss / max(total_valid_pixels, 1):.4f}",
                mIoU=f"{metrics['mean_iou']:.4f}",
                refresh=False,
            )

        del images, masks, logits, loss

    progress.close()
    metrics, per_class = metrics_from_confusion(accumulator.matrix.numpy())
    metrics["loss"] = total_loss / max(total_valid_pixels, 1)
    return metrics, per_class, accumulator.matrix.numpy()

def atomic_torch_save(payload, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, path)


def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scheduler,
    scaler,
    config,
    validation_metrics,
    best_value,
    best_epoch,
    patience,
):
    """
    Guarda un checkpoint liviano.

    Se omiten los estados de AdamW, scheduler y GradScaler para evitar el
    aumento brusco de RAM que puede cerrar el kernel al terminar una época.
    Al reanudar se recuperan los pesos y la época; el optimizador se reinicia.
    """
    cpu_state = {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }
    atomic_torch_save(
        {
            "epoch": int(epoch),
            "model_state_dict": cpu_state,
            "config": config,
            "validation_metrics": validation_metrics,
            "best_val_iou": float(best_value),
            "best_epoch": int(best_epoch),
            "patience": int(patience),
            "saved_at": datetime.now().isoformat(timespec="seconds"),
            "checkpoint_type": "model_only",
        },
        path,
    )
    del cpu_state
    gc.collect()




def load_checkpoint_if_needed(
    last_checkpoint,
    model,
    optimizer,
    scheduler,
    scaler,
    auto_resume,
):
    if not auto_resume or not Path(last_checkpoint).exists():
        return 1, float("-inf"), 0, 0

    try:
        checkpoint = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(last_checkpoint, map_location="cpu")

    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    if checkpoint.get("checkpoint_type") == "model_only":
        warnings.warn(
            "Se reanudaron los pesos del modelo, pero AdamW, scheduler y "
            "GradScaler comienzan con estado nuevo para reducir el consumo de RAM.",
            RuntimeWarning,
        )

    for key, target in (
        ("optimizer_state_dict", optimizer),
        ("scheduler_state_dict", scheduler),
    ):
        state = checkpoint.get(key)
        if state:
            try:
                target.load_state_dict(state)
            except Exception as exc:
                warnings.warn(f"No se restauró {key}: {exc}", RuntimeWarning)

    scaler_state = checkpoint.get("scaler_state_dict")
    if scaler is not None and isinstance(scaler_state, dict) and scaler_state:
        try:
            scaler.load_state_dict(scaler_state)
        except Exception as exc:
            warnings.warn(f"No se restauró GradScaler: {exc}", RuntimeWarning)

    return (
        int(checkpoint.get("epoch", 0)) + 1,
        float(checkpoint.get("best_val_iou", float("-inf"))),
        int(checkpoint.get("best_epoch", 0)),
        int(checkpoint.get("patience", 0)),
    )


def plot_history(history: pd.DataFrame, output_dir: Path) -> None:
    if history.empty:
        return

    output_dir = ensure_dir(output_dir)
    pairs = [
        ("train_loss", "val_loss", "Pérdida"),
        ("train_pixel_accuracy", "val_pixel_accuracy", "Exactitud por píxel"),
        ("train_mean_iou", "val_mean_iou", "IoU media"),
        ("train_macro_f1", "val_macro_f1", "F1 macro"),
        ("train_kappa", "val_kappa", "Kappa"),
    ]

    for train_column, val_column, title in pairs:
        plt.figure(figsize=(9, 5))
        if train_column in history:
            plt.plot(history["epoch"], history[train_column], label="Entrenamiento")
        if val_column in history:
            plt.plot(history["epoch"], history[val_column], label="Validación")
        plt.xlabel("Época")
        plt.ylabel(title)
        plt.title(title)
        plt.grid(True)
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / f"{safe_name(title)}.png", dpi=180)
        plt.close()


def plot_confusion_matrix(matrix, idx_to_name, path):
    labels = list(range(matrix.shape[0]))
    names = [idx_to_name[index] for index in labels]

    plt.figure(figsize=(10, 8))
    plt.imshow(matrix, interpolation="nearest")
    plt.colorbar()
    ticks = np.arange(len(labels))
    plt.xticks(ticks, names, rotation=90)
    plt.yticks(ticks, names)
    plt.xlabel("Predicción")
    plt.ylabel("Referencia")
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()


def save_test_predictions(
    model,
    loader,
    output_dir,
    max_items,
    idx_to_code: dict[int, int],
):
    output_dir = ensure_dir(output_dir)
    model.eval()
    saved = 0
    lookup = np.asarray(
        [idx_to_code[index] for index in range(len(idx_to_code))],
        dtype=np.uint8,
    )

    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicciones de prueba"):
            images = batch["image"].to(DEVICE)
            with autocast_context():
                internal_predictions = model(images).argmax(dim=1).cpu().numpy()

            for index, internal_prediction in enumerate(internal_predictions):
                if saved >= max_items:
                    return

                image_path = batch["image_path"][index]
                prediction = lookup[internal_prediction]
                _, valid_pixel = read_reflectance(image_path)
                prediction = prediction.copy()
                prediction[~valid_pixel] = MASK_NODATA_CODE

                with rasterio.open(image_path) as src:
                    profile = src.profile.copy()
                profile.update(
                    count=1,
                    dtype="uint8",
                    nodata=MASK_NODATA_CODE,
                    compress="lzw",
                )

                output = output_dir / f"prediccion_test_{saved:04d}.tif"
                with rasterio.open(output, "w", **profile) as dst:
                    dst.write(prediction, 1)
                saved += 1


def write_markdown_report(summary, config, path):
    lines = ["# Informe de entrenamiento ResNet50-UNet\n", "## Configuración\n"]
    for key, value in config.items():
        lines.append(f"- **{key}**: {value}")

    lines.append("\n## Resultados de prueba\n")
    for key, value in summary.items():
        formatted = f"{float(value):.6f}" if isinstance(value, (int, float, np.integer, np.floating)) else str(value)
        lines.append(f"- **{key}**: {formatted}")

    path = Path(path)
    ensure_dir(path.parent)
    path.write_text("\n".join(lines), encoding="utf-8")


def find_completed_run(level_name: str):
    """
    Determina si un nivel ya terminó.

    Se considera completado cuando existe:
    1. LEVEL_COMPLETED.json, o
    2. reports/resumen_metricas_test.json, o
    3. un checkpoint cuya época llegó al máximo o cuya paciencia alcanzó
       el criterio de early stopping.
    """
    if not RUNS_DIR.exists():
        return None

    candidates = []

    for run_dir in RUNS_DIR.glob(f"ResNet50Seg_{level_name}_*"):
        marker = run_dir / "LEVEL_COMPLETED.json"
        report = run_dir / "reports" / "resumen_metricas_test.json"
        checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"

        completed = marker.exists() or report.exists()

        if not completed and checkpoint.exists():
            try:
                try:
                    state = torch.load(
                        checkpoint,
                        map_location="cpu",
                        weights_only=False,
                    )
                except TypeError:
                    state = torch.load(
                        checkpoint,
                        map_location="cpu",
                    )

                checkpoint_epoch = int(state.get("epoch", 0))
                checkpoint_patience = int(state.get("patience", 0))

                completed = (
                    checkpoint_epoch >= EPOCHS
                    or checkpoint_patience >= EARLY_STOPPING_PATIENCE
                )

                del state
                gc.collect()

            except Exception as error:
                print(
                    f"No fue posible revisar el checkpoint de {run_dir.name}: "
                    f"{error}"
                )

        if completed:
            evidence_times = [
                path.stat().st_mtime
                for path in (marker, report, checkpoint)
                if path.exists()
            ]
            modified = max(evidence_times) if evidence_times else 0
            candidates.append((modified, run_dir))

    if not candidates:
        return None

    return max(candidates, key=lambda item: item[0])[1]


def mark_level_completed(
    level_name: str,
    run_dir: Path,
    best_epoch: int,
    best_val_iou: float,
):
    marker = {
        "completed": True,
        "level": level_name,
        "run_dir": str(run_dir),
        "best_epoch": int(best_epoch),
        "best_val_iou": float(best_val_iou),
        "completed_at": datetime.now().isoformat(timespec="seconds"),
    }
    save_json(marker, Path(run_dir) / "LEVEL_COMPLETED.json")


def build_run_dir(level_name: str):
    forced = RESUME_BY_LEVEL.get(level_name)

    if forced:
        experiment_name = forced
        auto_resume = True
    elif CREATE_NEW_RUNS:
        experiment_name = f"ResNet50Seg_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        auto_resume = False
    else:
        candidates = []
        if RUNS_DIR.exists():
            for run_dir in RUNS_DIR.glob(f"ResNet50Seg_{level_name}_*"):
                checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
                if checkpoint.exists():
                    candidates.append((checkpoint.stat().st_mtime, run_dir.name))

        if candidates:
            experiment_name = max(candidates)[1]
            auto_resume = True
        else:
            experiment_name = f"ResNet50Seg_{level_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
            auto_resume = False

    run_dir = ensure_dir(RUNS_DIR / experiment_name)
    checkpoint_dir = ensure_dir(run_dir / "checkpoints")
    report_dir = ensure_dir(run_dir / "reports")
    prediction_dir = ensure_dir(run_dir / "test_predictions")
    return run_dir, checkpoint_dir, report_dir, prediction_dir, experiment_name, auto_resume


def train_level(level_name: str):
    batch_size = int(BATCH_SIZE_BY_LEVEL[level_name])
    gradient_accumulation_steps = int(
        GRADIENT_ACCUMULATION_BY_LEVEL[level_name]
    )
    print(
        f"\n{level_name}: batch={batch_size}, "
        f"acumulación={gradient_accumulation_steps}, "
        f"batch efectivo={batch_size * gradient_accumulation_steps}"
    )
    class_dict = get_class_dict(level_name)
    thematic_codes = sorted(code for code in class_dict if code != MASK_NODATA_CODE)
    code_to_idx = {code: index for index, code in enumerate(thematic_codes)}
    idx_to_code = {index: code for code, index in code_to_idx.items()}
    idx_to_name = {
        index: class_dict[code]
        for index, code in idx_to_code.items()
    }
    num_classes = len(code_to_idx)

    train_df, val_df, test_df, splits_dir, band_count, chip_shape = read_splits(
        level_name,
        class_dict,
    )

    means, stds = compute_image_stats(train_df)
    pixel_counts = compute_class_pixel_counts(train_df, code_to_idx)
    class_weights = (
        make_class_weights(pixel_counts).to(DEVICE)
        if USE_CLASS_WEIGHTS
        else None
    )
    sampler = build_weighted_sampler(train_df)

    if USE_CLASS_WEIGHTS and USE_WEIGHTED_SAMPLER:
        warnings.warn(
            "Se están aplicando simultáneamente pesos de clase y muestreo balanceado.",
            RuntimeWarning,
        )

    train_loader = DataLoader(
        LULCSegmentationDataset(
            train_df,
            code_to_idx,
            means,
            stds,
            augment=USE_AUGMENTATION,
        ),
        batch_size=batch_size,
        shuffle=sampler is None,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )
    val_loader = DataLoader(
        LULCSegmentationDataset(val_df, code_to_idx, means, stds, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )
    test_loader = DataLoader(
        LULCSegmentationDataset(test_df, code_to_idx, means, stds, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS and NUM_WORKERS > 0,
        prefetch_factor=PREFETCH_FACTOR if NUM_WORKERS > 0 else None,
    )

    model, pretrained_loaded = build_segmentation_model(
        in_channels=len(BAND_INDEXES),
        num_classes=num_classes,
        pretrained=ENCODER_PRETRAINED,
        freeze_encoder=FREEZE_ENCODER,
    )
    model = model.to(DEVICE)

    print("Validando la arquitectura en la GPU...")
    first_batch = next(iter(train_loader))["image"].to(
        DEVICE,
        non_blocking=PIN_MEMORY,
    )
    model.eval()
    with torch.inference_mode():
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP and DEVICE.type == "cuda",
        ):
            output = model(first_batch[:1])
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    model.train()
    print("Arquitectura validada. Iniciando entrenamiento.")

    expected_output = (1, num_classes, first_batch.shape[-2], first_batch.shape[-1])
    if tuple(output.shape) != expected_output:
        raise RuntimeError(
            f"Salida inesperada del modelo: {tuple(output.shape)}; "
            f"se esperaba {expected_output}."
        )

    del first_batch, output
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    if OPTIMIZER_TYPE == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )
    elif OPTIMIZER_TYPE == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=LEARNING_RATE,
            momentum=0.99,
            weight_decay=WEIGHT_DECAY,
        )
    else:
        raise ValueError(f"OPTIMIZER_TYPE no reconocido: {OPTIMIZER_TYPE}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=8,
        min_lr=1e-6,
    )
    criterion = CombinedSegmentationLoss(num_classes, class_weights)
    scaler = make_grad_scaler()

    (
        run_dir,
        checkpoint_dir,
        report_dir,
        prediction_dir,
        experiment_name,
        auto_resume,
    ) = build_run_dir(level_name)

    config = {
        "model": "ResNet50-UNet",
        "encoder": "resnet50",
        "task": "segmentacion_semantica",
        "level": level_name,
        "dataset_root": str(DATASET_ROOT),
        "splits_dir": str(splits_dir),
        "run_dir": str(run_dir),
        "experiment_name": experiment_name,
        "band_indexes": list(BAND_INDEXES),
        "band_order": ["B2", "B3", "B4", "B8"],
        "reflectance_scale": REFLECTANCE_SCALE,
        "reflectance_add_offset": REFLECTANCE_ADD_OFFSET,
        "use_raster_scale_offset": USE_RASTER_SCALE_OFFSET,
        "auto_scale_reflectance": AUTO_SCALE_REFLECTANCE,
        "band_count": band_count,
        "chip_shape": chip_shape,
        "num_classes": num_classes,
        "mask_nodata_code": MASK_NODATA_CODE,
        "ignore_index": IGNORE_INDEX,
        "code_to_idx": code_to_idx,
        "idx_to_code": idx_to_code,
        "idx_to_name": idx_to_name,
        "means": means,
        "stds": stds,
        "encoder_pretrained_requested": ENCODER_PRETRAINED,
        "encoder_pretrained_loaded": pretrained_loaded,
        "freeze_encoder": FREEZE_ENCODER,
        "decoder_dropout": DECODER_DROPOUT,
        "optimizer": OPTIMIZER_TYPE,
        "loss": LOSS_TYPE,
        "dice_weight": DICE_WEIGHT,
        "epochs": EPOCHS,
        "batch_size": batch_size,
        "gradient_accumulation_steps": gradient_accumulation_steps,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "use_class_weights": USE_CLASS_WEIGHTS,
        "use_weighted_sampler": USE_WEIGHTED_SAMPLER,
        "use_augmentation": USE_AUGMENTATION,
        "seed": SEED,
    }
    save_json(config, run_dir / "config.json")
    save_dataset_summary(
        train_df,
        val_df,
        test_df,
        pixel_counts,
        idx_to_code,
        class_dict,
        report_dir / "dataset",
    )

    history_path = run_dir / "history.csv"
    history = (
        pd.read_csv(history_path).to_dict("records")
        if history_path.exists() and history_path.stat().st_size > 0
        else []
    )

    last_checkpoint = checkpoint_dir / "last_checkpoint.pt"
    start_epoch, best_value, best_epoch, patience = load_checkpoint_if_needed(
        last_checkpoint,
        model,
        optimizer,
        scheduler,
        scaler,
        auto_resume,
    )

    for epoch in range(start_epoch, EPOCHS + 1):
        train_metrics = train_one_epoch(
            model,
            train_loader,
            optimizer,
            criterion,
            scaler,
            epoch,
            num_classes,
            gradient_accumulation_steps,
        )
        val_metrics, _, _ = evaluate(
            model,
            val_loader,
            criterion,
            f"Validación {level_name} época {epoch}",
            num_classes,
        )

        scheduler.step(val_metrics["mean_iou"])
        improved = val_metrics["mean_iou"] > best_value

        if improved:
            best_value = val_metrics["mean_iou"]
            best_epoch = epoch
            patience = 0
        else:
            patience += 1

        row = {"epoch": epoch, "learning_rate": optimizer.param_groups[0]["lr"]}
        row.update({f"train_{key}": value for key, value in train_metrics.items()})
        row.update({f"val_{key}": value for key, value in val_metrics.items()})
        history.append(row)

        history_df = (
            pd.DataFrame(history)
            .drop_duplicates(subset=["epoch"], keep="last")
            .sort_values("epoch")
        )
        history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

        print("Guardando checkpoint liviano...")
        save_checkpoint(
            last_checkpoint,
            epoch,
            model,
            optimizer,
            scheduler,
            scaler,
            config,
            val_metrics,
            best_value,
            best_epoch,
            patience,
        )

        if improved:
            shutil.copy2(last_checkpoint, run_dir / "best_model.pt")

        if SAVE_EVERY_EPOCH:
            shutil.copy2(
                last_checkpoint,
                checkpoint_dir / f"checkpoint_epoch_{epoch:03d}.pt",
            )

        print(
            f"[{level_name} | época {epoch}] "
            f"train_loss={train_metrics['loss']:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_mIoU={val_metrics['mean_iou']:.4f} "
            f"mejor_mIoU={best_value:.4f} "
            f"paciencia={patience}/{EARLY_STOPPING_PATIENCE}"
        )

        if DEVICE.type == "cuda":
            allocated = torch.cuda.memory_allocated() / 1024**3
            reserved = torch.cuda.memory_reserved() / 1024**3
            peak = torch.cuda.max_memory_allocated() / 1024**3
            print(
                f"VRAM: asignada={allocated:.2f} GB, "
                f"reservada={reserved:.2f} GB, pico={peak:.2f} GB"
            )

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        if EPOCH_COOLDOWN_SECONDS > 0:
            print(
                f"Pausa de estabilización: {EPOCH_COOLDOWN_SECONDS} segundos."
            )
            time.sleep(EPOCH_COOLDOWN_SECONDS)

        if patience >= EARLY_STOPPING_PATIENCE:
            break

    if PLOT_AFTER_TRAINING:
        final_history_df = (
            pd.DataFrame(history)
            .drop_duplicates(subset=["epoch"], keep="last")
            .sort_values("epoch")
        )
        plot_history(final_history_df, report_dir)

    best_path = run_dir / "best_model.pt"
    checkpoint_path = best_path if best_path.exists() else last_checkpoint
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"No se generó un checkpoint en {run_dir}")

    try:
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    except TypeError:
        checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)

    test_metrics, per_class, confusion = evaluate(
        model,
        test_loader,
        criterion,
        f"Prueba {level_name}",
        num_classes,
    )

    per_class = per_class.rename(columns={"class_code": "class_index"})
    per_class["class_code"] = per_class["class_index"].map(idx_to_code)
    per_class["class_name"] = per_class["class_index"].map(idx_to_name)
    per_class.to_csv(
        report_dir / "metricas_por_clase_test.csv",
        index=False,
        encoding="utf-8-sig",
    )

    labels = list(range(num_classes))
    names = [idx_to_name[index] for index in labels]
    pd.DataFrame(
        confusion,
        index=names,
        columns=names,
    ).to_csv(
        report_dir / "matriz_confusion_test.csv",
        encoding="utf-8-sig",
    )

    if SAVE_REPORT_PLOTS:
        plot_confusion_matrix(
            confusion,
            idx_to_name,
            report_dir / "matriz_confusion_test.png",
        )

    if SAVE_TEST_PREDICTIONS:
        save_test_predictions(
            model,
            test_loader,
            prediction_dir,
            MAX_TEST_PREDICTIONS_TO_SAVE,
            idx_to_code,
        )

    summary = {
        **test_metrics,
        "train_level": level_name,
        "run_dir": str(run_dir),
        "best_epoch": int(best_epoch),
        "best_val_iou": float(best_value),
        "completed": True,
    }
    save_json(summary, report_dir / "resumen_metricas_test.json")
    mark_level_completed(
        level_name=level_name,
        run_dir=run_dir,
        best_epoch=best_epoch,
        best_val_iou=best_value,
    )
    write_markdown_report(
        summary,
        config,
        report_dir / "informe_entrenamiento_resnet50_segmentacion.md",
    )

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary



def run_all_levels():
    """
    Ejecuta los niveles directamente en la misma celda.

    No usa analizadores de argumentos, procesos externos ni salidas forzadas.
    """
    set_seed(SEED)
    ensure_dir(RUNS_DIR)

    summaries = []

    for position, level_name in enumerate(LEVELS_TO_TRAIN, start=1):
        print("\n" + "=" * 80)
        print(
            f"REVISANDO {level_name} "
            f"({position}/{len(LEVELS_TO_TRAIN)})"
        )
        print("=" * 80)

        completed_run = (
            find_completed_run(level_name)
            if SKIP_COMPLETED_LEVELS
            else None
        )

        if completed_run is not None:
            print(
                f"{level_name} ya está terminado en:\n{completed_run}\n"
                "Se omitirá y se continuará con el siguiente nivel."
            )
            summaries.append(
                {
                    "train_level": level_name,
                    "status": "already_completed",
                    "run_dir": str(completed_run),
                }
            )
            continue

        print(
            f"{level_name} no tiene una ejecución final completa. "
            "El entrenamiento se iniciará o se reanudará desde el último "
            "checkpoint disponible."
        )

        try:
            summary = train_level(level_name)
            summary["status"] = "completed"
            summaries.append(summary)

        except KeyboardInterrupt:
            print(
                f"\nEntrenamiento interrumpido durante {level_name}. "
                "El último checkpoint guardado se conserva."
            )
            summaries.append(
                {
                    "train_level": level_name,
                    "status": "interrupted",
                }
            )
            break

        except Exception as error:
            print(
                f"\nSe produjo un error durante {level_name}: {error}\n"
                "El último checkpoint guardado se conserva."
            )
            traceback.print_exc()
            summaries.append(
                {
                    "train_level": level_name,
                    "status": "error",
                    "error": str(error),
                }
            )
            break

        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            torch.cuda.empty_cache()

        if LEVEL_COOLDOWN_SECONDS > 0:
            print(
                f"\n{level_name} terminó. Pausa de "
                f"{LEVEL_COOLDOWN_SECONDS} segundos antes de continuar."
            )
            time.sleep(LEVEL_COOLDOWN_SECONDS)

    summary_df = pd.DataFrame(summaries)
    output = RUNS_DIR / (
        "resumen_resnet50_segmentacion_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    )
    summary_df.to_csv(
        output,
        index=False,
        encoding="utf-8-sig",
    )

    print("\n" + "=" * 80)
    print("RESUMEN GENERAL")
    print("=" * 80)
    print(summary_df)
    print(f"\nArchivo de resumen: {output}")

    return summary_df


def main():
    if REQUIRE_CUDA and DEVICE.type != "cuda":
        raise RuntimeError(
            "CUDA/ROCm no está disponible; cambia REQUIRE_CUDA a False "
            "para ejecutar en CPU."
        )

    print("=" * 80)
    print("RESNET-50 PARA SEGMENTACIÓN SEMÁNTICA")
    print("EJECUCIÓN DIRECTA EN UNA CELDA DE JUPYTER")
    print("=" * 80)
    print(f"Dispositivo: {DEVICE}")
    print(f"PyTorch: {torch.__version__}")
    print(f"ROCm/HIP: {torch.version.hip}")
    print(f"Hilos PyTorch: {torch.get_num_threads()}")
    print(f"DataLoader workers: {NUM_WORKERS}")

    if DEVICE.type == "cuda":
        print(f"Acelerador: {torch.cuda.get_device_name(0)}")
        total_vram = (
            torch.cuda.get_device_properties(0).total_memory / 1024**3
        )
        print(f"VRAM disponible: {total_vram:.2f} GB")

    return run_all_levels()


# Ejecución directa en una sola celda de Jupyter.
summary_df = main()


RESNET-50 PARA SEGMENTACIÓN SEMÁNTICA
EJECUCIÓN DIRECTA EN UNA CELDA DE JUPYTER
Dispositivo: cuda
PyTorch: 2.9.1+rocm7.2.1
ROCm/HIP: 7.2.53211-158bd99533
Hilos PyTorch: 4
DataLoader workers: 0
Acelerador: AMD Radeon RX 9060 XT
VRAM disponible: 15.92 GB

REVISANDO nivel_1 (1/2)
nivel_1 ya está terminado en:
C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion\ResNet50Seg_nivel_1_20260730_191354
Se omitirá y se continuará con el siguiente nivel.

REVISANDO nivel_2 (2/2)
nivel_2 no tiene una ejecución final completa. El entrenamiento se iniciará o se reanudará desde el último checkpoint disponible.

nivel_2: batch=1, acumulación=4, batch efectivo=4


Píxeles por clase: 100%|██████████| 3727/3727 [00:46<00:00, 80.26it/s]


Validando la arquitectura en la GPU...
Arquitectura validada. Iniciando entrenamiento.


Guardando checkpoint liviano...
[nivel_2 | época 1] train_loss=2.6740 val_loss=2.3130 val_mIoU=0.0962 mejor_mIoU=0.0962 paciencia=0/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.


Guardando checkpoint liviano...
[nivel_2 | época 2] train_loss=2.1921 val_loss=2.0697 val_mIoU=0.1151 mejor_mIoU=0.1151 paciencia=0/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.


...

Guardando checkpoint liviano...
[nivel_2 | época 117] train_loss=0.5883 val_loss=1.3258 val_mIoU=0.5028 mejor_mIoU=0.5101 paciencia=12/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.


Guardando checkpoint liviano...
[nivel_2 | época 118] train_loss=0.5840 val_loss=1.3246 val_mIoU=0.5037 mejor_mIoU=0.5101 paciencia=13/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.


Guardando checkpoint liviano...
[nivel_2 | época 119] train_loss=0.5857 val_loss=1.3234 val_mIoU=0.5042 mejor_mIoU=0.5101 paciencia=14/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.


Guardando checkpoint liviano...
[nivel_2 | época 120] train_loss=0.5812 val_loss=1.3235 val_mIoU=0.5014 mejor_mIoU=0.5101 paciencia=15/20
VRAM: asignada=0.50 GB, reservada=0.86 GB, pico=0.83 GB
Pausa de estabilización: 3 segundos.



nivel_2 terminó. Pausa de 20 segundos antes de continuar.

RESUMEN GENERAL
  train_level             status  \
0     nivel_1  already_completed   
1     nivel_2          completed   

                                             run_dir  pixel_accuracy  \
0  C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_S...             NaN   
1  C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_S...        0.729329   

   macro_precision  macro_recall  macro_f1  mean_iou     kappa      loss  \
0              NaN           NaN       NaN       NaN       NaN       NaN   
1         0.622153      0.661933  0.636794  0.486643  0.644405  1.339446   

   best_epoch  best_val_iou completed  
0         NaN           NaN       NaN  
1       105.0      0.510065      True  

Archivo de resumen: C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion\resumen_resnet50_segmentacion_20260804_033734.csv


In [1]:
from __future__ import annotations

# =============================================================================
# CONFIGURACIÓN DE RECURSOS — Ryzen 5 5600X, 16 GB RAM y GPU AMD de 16 GB
# Debe definirse antes de importar NumPy, PyTorch, GDAL o Rasterio.
# =============================================================================
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("GDAL_CACHEMAX", "512")
# Evita la ruta NHWC de BatchNorm en MIOpen. El codificador se convierte
# además a GroupNorm para trabajar de forma estable con lotes pequeños.
os.environ.setdefault("PYTORCH_MIOPEN_SUGGEST_NHWC_BATCHNORM", "0")
os.environ.setdefault("MIOPEN_FIND_MODE", "FAST")
os.environ.setdefault(
    "PYTORCH_ALLOC_CONF",
    "expandable_segments:True,max_split_size_mb:128",
)

import gc
import json
import math
import os
import shutil
import traceback
import warnings
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from torchvision import models
from tqdm import tqdm

# =============================================================================
# 1. CONFIGURACIÓN
# =============================================================================
MODEL_NAME = "ResNet50-UNet"
# =============================================================================
# 1. RUTAS — MODIFICAR ÚNICAMENTE ESTA SECCIÓN
# =============================================================================
MODEL_ROOT = Path(r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion")
INPUT_RASTER = Path(r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif")
OUTPUT_DIR = Path(r"C:\Users\Ivan sua\anaconda3\Predicciones_ResNet50_Segmentacion")

# Se procesan en este orden dentro de una sola celda:
# primero nivel 1 y después nivel 2.
LEVELS_TO_PREDICT = ["nivel_1", "nivel_2"]

# None selecciona automáticamente la ejecución final más reciente.
# También puede escribirse el nombre exacto de una carpeta de entrenamiento.
RUN_BY_LEVEL = {"nivel_1": None, "nivel_2": None}

REFERENCE_BY_LEVEL = {
    "nivel_1": Path(r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel1.tif"),
    "nivel_2": Path(r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel2.tif"),
}
# Reutiliza una predicción final existente y continúa con el siguiente nivel.
# Cambiar a False cuando se necesite regenerar los GeoTIFF.
REUSE_COMPLETED_PREDICTIONS = True

# Para la predicción final se recomienda usar únicamente best_model.pt.
# Cambiar a True solo para permitir last_checkpoint.pt cuando no exista
# un mejor modelo guardado.
ALLOW_LAST_CHECKPOINT = False

# Máscara opcional con 1 para píxeles válidos y 0 para nubes, sombras o exclusiones.
VALIDITY_MASK_RASTER = None

REQUIRE_CUDA = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_num_threads(4)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_float32_matmul_precision("high")
USE_AMP = False  # Mayor estabilidad en Windows/ROCm
TILE_SIZE_BY_LEVEL = {"nivel_1": 256, "nivel_2": 256}
OVERLAP_BY_LEVEL = {"nivel_1": 32, "nivel_2": 32}
RESUME_PREDICTION = True
FLUSH_EVERY_TILES = 5
EMPTY_CACHE_EVERY_TILES = 10
SAVE_CONFUSION_PLOT = False
OUTPUT_NODATA = 0
TEMP_DIR = OUTPUT_DIR / "_temporales"

# El código evalúa todos los píxeles válidos en la zona de coincidencia entre
# la predicción y el ráster de referencia. Cuando la referencia corresponde
# a un año diferente de la imagen, los resultados representan concordancia
# espacial con esa referencia histórica y no exactitud temática del año 2025.
EVALUATION_NOTE = (
    "Evaluación censal de todos los píxeles válidos coincidentes. "
    "Si la referencia es de un año diferente, las métricas expresan "
    "concordancia espacial con la referencia histórica."
)

CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}
CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}


def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(level_name)


# =============================================================================
# 2. ARQUITECTURA
# =============================================================================

class ConvNormAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        groups = min(32, out_channels)
        while out_channels % groups != 0:
            groups -= 1
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = ConvNormAct(in_channels + skip_channels, out_channels, dropout)

    def forward(self, decoder, skip):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([decoder, skip], dim=1))



def _valid_group_count(channels: int, maximum_groups: int = 32) -> int:
    """Selecciona el mayor número de grupos que divide exactamente los canales."""
    for groups in (maximum_groups, 16, 8, 4, 2, 1):
        if groups <= channels and channels % groups == 0:
            return groups
    return 1


def replace_batchnorm_with_groupnorm(module: nn.Module) -> nn.Module:
    """
    Sustituye recursivamente BatchNorm2d por GroupNorm.

    Esta conversión evita la operación BatchNorm de MIOpen que produjo
    miopenStatusUnknownError y es más estable con batch size 1–4.
    Los parámetros afines de los pesos preentrenados se conservan.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            replacement = nn.GroupNorm(
                num_groups=_valid_group_count(child.num_features),
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(child.weight)
                    replacement.bias.copy_(child.bias)
            setattr(module, name, replacement)
        else:
            replace_batchnorm_with_groupnorm(child)
    return module


def adapt_first_convolution(old_conv: nn.Conv2d, in_channels: int, pretrained_loaded: bool) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        if pretrained_loaded:
            # Entrada Sentinel-2: B2, B3, B4, B8. Los pesos ImageNet vienen en RGB.
            if in_channels >= 3:
                new_conv.weight[:, 0] = old_conv.weight[:, 2]  # B2 <- azul
                new_conv.weight[:, 1] = old_conv.weight[:, 1]  # B3 <- verde
                new_conv.weight[:, 2] = old_conv.weight[:, 0]  # B4 <- rojo
                if in_channels > 3:
                    mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
                    new_conv.weight[:, 3:] = mean_weight.repeat(1, in_channels - 3, 1, 1)
            else:
                new_conv.weight[:, :in_channels] = old_conv.weight[:, :in_channels]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
    return new_conv


class ResNet50UNet(nn.Module):
    """Segmentación semántica con ResNet-50 como codificador y decodificador tipo U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        pretrained: bool = True,
        freeze_encoder: bool = False,
        decoder_dropout: float = 0.10,
    ):
        super().__init__()
        pretrained_loaded = False
        weights = None
        if pretrained:
            try:
                weights = models.ResNet50_Weights.DEFAULT
                encoder = models.resnet50(weights=weights)
                pretrained_loaded = True
            except Exception as exc:
                warnings.warn(
                    f"No fue posible cargar pesos ImageNet de ResNet-50 ({exc}); se inicializa sin pesos.",
                    RuntimeWarning,
                )
                encoder = models.resnet50(weights=None)
        else:
            encoder = models.resnet50(weights=None)

        encoder.conv1 = adapt_first_convolution(
            encoder.conv1,
            in_channels,
            pretrained_loaded,
        )
        encoder = replace_batchnorm_with_groupnorm(encoder)

        self.conv1 = encoder.conv1
        self.bn1 = encoder.bn1
        self.relu = encoder.relu
        self.maxpool = encoder.maxpool
        self.layer1 = encoder.layer1   # 256 canales, 1/4
        self.layer2 = encoder.layer2   # 512 canales, 1/8
        self.layer3 = encoder.layer3   # 1024 canales, 1/16
        self.layer4 = encoder.layer4   # 2048 canales, 1/32

        self.decoder4 = DecoderBlock(2048, 1024, 512, decoder_dropout)
        self.decoder3 = DecoderBlock(512, 512, 256, decoder_dropout)
        self.decoder2 = DecoderBlock(256, 256, 128, decoder_dropout)
        self.decoder1 = DecoderBlock(128, 64, 64, decoder_dropout)
        self.final_refine = ConvNormAct(64, 32, decoder_dropout)
        self.segmentation_head = nn.Conv2d(32, num_classes, kernel_size=1)
        self.pretrained_loaded = pretrained_loaded

        if freeze_encoder:
            for module in (
                self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4
            ):
                for parameter in module.parameters():
                    parameter.requires_grad = False

    def forward(self, x):
        input_size = x.shape[-2:]
        skip0 = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(skip0)
        skip1 = self.layer1(x)
        skip2 = self.layer2(skip1)
        skip3 = self.layer3(skip2)
        x = self.layer4(skip3)

        x = self.decoder4(x, skip3)
        x = self.decoder3(x, skip2)
        x = self.decoder2(x, skip1)
        x = self.decoder1(x, skip0)
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        x = self.final_refine(x)
        return self.segmentation_head(x)


def build_segmentation_model(
    in_channels: int,
    num_classes: int,
    pretrained: bool = False,
    freeze_encoder: bool = False,
):
    model = ResNet50UNet(
        in_channels=in_channels,
        num_classes=num_classes,
        pretrained=pretrained,
        freeze_encoder=freeze_encoder,
        decoder_dropout=0.10,
    )
    return model, bool(model.pretrained_loaded)


# =============================================================================
# 3. UTILIDADES DE MODELO Y PREPROCESAMIENTO
# =============================================================================
def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data: dict, path: str | Path) -> Path:
    """
    Guarda un diccionario en JSON de forma segura.

    Primero escribe un archivo temporal y después lo reemplaza para evitar
    estados de reanudación dañados tras un cierre inesperado.
    """
    path = Path(path)
    ensure_dir(path.parent)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            indent=2,
            ensure_ascii=False,
        )

    temporary_path.replace(path)
    return path


def safe_cuda_cleanup() -> None:
    """
    Libera memoria de GPU sin ocultar el error principal cuando el
    controlador de Windows/ROCm ya se ha reiniciado.
    """
    gc.collect()

    if not torch.cuda.is_available():
        return

    try:
        torch.cuda.synchronize()
    except Exception as cleanup_error:
        warnings.warn(
            f"No fue posible sincronizar la GPU durante la limpieza: "
            f"{cleanup_error}",
            RuntimeWarning,
        )

    try:
        torch.cuda.empty_cache()
    except Exception as cleanup_error:
        warnings.warn(
            f"No fue posible liberar la caché de la GPU: {cleanup_error}",
            RuntimeWarning,
        )


def _validate_run_config(run_dir: Path, level_name: str) -> dict:
    config_path = run_dir / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(
            f"La ejecución no contiene config.json: {run_dir}"
        )

    config = load_json(config_path)

    if str(config.get("task", "")).lower() != "segmentacion_semantica":
        raise ValueError(
            f"La ejecución no corresponde a segmentación semántica: {run_dir}"
        )

    if str(config.get("level", "")).lower() != level_name:
        raise ValueError(
            f"El nivel del config.json no coincide con {level_name}: {run_dir}"
        )

    model_name = str(config.get("model", "")).lower()
    encoder_name = str(config.get("encoder", "")).lower()
    if "resnet50" not in model_name and encoder_name != "resnet50":
        raise ValueError(
            f"La ejecución no corresponde a ResNet-50: {run_dir}"
        )

    band_indexes = tuple(
        int(value)
        for value in config.get("band_indexes", [])
    )
    if len(band_indexes) != 4:
        raise ValueError(
            "La ejecución debe usar cuatro bandas B2, B3, B4 y B8. "
            f"Se encontró: {band_indexes}"
        )

    return config


def find_run_dir(level_name: str) -> Path:
    forced = RUN_BY_LEVEL.get(level_name)

    if forced:
        candidate = MODEL_ROOT / forced
        if not candidate.exists():
            raise FileNotFoundError(candidate)
        _validate_run_config(candidate, level_name)
        return candidate

    candidates = []

    for run_dir in MODEL_ROOT.glob(f"ResNet50Seg_{level_name}_*"):
        try:
            _validate_run_config(run_dir, level_name)
        except Exception:
            continue

        best_model = run_dir / "best_model.pt"
        last_checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
        completed_marker = run_dir / "LEVEL_COMPLETED.json"
        final_report = run_dir / "reports" / "resumen_metricas_test.json"

        if not best_model.exists() and not (
            ALLOW_LAST_CHECKPOINT and last_checkpoint.exists()
        ):
            continue

        completed = completed_marker.exists() or final_report.exists()
        evidence = [
            path.stat().st_mtime
            for path in (
                completed_marker,
                final_report,
                best_model,
                last_checkpoint,
            )
            if path.exists()
        ]
        modified = max(evidence) if evidence else run_dir.stat().st_mtime

        # Se priorizan las ejecuciones finalizadas y después la más reciente.
        candidates.append((1 if completed else 0, modified, run_dir))

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró un modelo de segmentación válido para "
            f"{level_name} en {MODEL_ROOT}. Verifica que exista "
            "best_model.pt y config.json."
        )

    return max(candidates, key=lambda item: (item[0], item[1]))[2]


def checkpoint_path(run_dir: Path) -> Path:
    best_model = run_dir / "best_model.pt"
    if best_model.exists():
        return best_model

    last_checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
    if ALLOW_LAST_CHECKPOINT and last_checkpoint.exists():
        warnings.warn(
            "No existe best_model.pt; se utilizará last_checkpoint.pt.",
            RuntimeWarning,
        )
        return last_checkpoint

    raise FileNotFoundError(
        f"No existe best_model.pt en {run_dir}. "
        "Finaliza el entrenamiento o cambia ALLOW_LAST_CHECKPOINT a True."
    )


def parse_mapping(mapping) -> dict[int, int]:
    return {int(key): int(value) for key, value in mapping.items()}


def load_model_for_level(level_name: str):
    run_dir = find_run_dir(level_name)
    config = load_json(run_dir / "config.json")
    idx_to_code = parse_mapping(config["idx_to_code"])
    idx_to_name = {int(k): str(v) for k, v in config["idx_to_name"].items()}
    band_indexes = tuple(int(v) for v in config.get("band_indexes", [1, 2, 3, 4]))
    num_classes = int(config["num_classes"])

    model, _ = build_segmentation_model(
        in_channels=len(band_indexes),
        num_classes=num_classes,
        pretrained=False,
        freeze_encoder=False,
    )
    selected_checkpoint = checkpoint_path(run_dir)
    try:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
        )

    state = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    gc.collect()
    expected_indices = list(range(len(idx_to_code)))
    if sorted(idx_to_code) != expected_indices:
        raise ValueError(
            "El idx_to_code del checkpoint no es consecutivo desde cero: "
            f"{idx_to_code}"
        )
    if len(set(idx_to_code.values())) != len(idx_to_code):
        raise ValueError(
            f"El checkpoint contiene códigos temáticos duplicados: {idx_to_code}"
        )
    model = model.to(DEVICE).eval()
    return model, config, run_dir, idx_to_code, idx_to_name, band_indexes


def amp_context():
    if DEVICE.type == "cuda" and USE_AMP:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


def reflectance_window(src, window: Window, band_indexes, config):
    array = src.read(indexes=band_indexes, window=window).astype(np.float32)
    finite = np.isfinite(array)
    if src.nodata is not None and np.isfinite(src.nodata):
        finite &= array != src.nodata
    valid = finite.all(axis=0)
    array = np.where(finite, array, np.nan)

    scales = np.asarray([src.scales[i - 1] for i in band_indexes], dtype=np.float32)[:, None, None]
    offsets = np.asarray([src.offsets[i - 1] for i in band_indexes], dtype=np.float32)[:, None, None]
    use_transform = bool(config.get("use_raster_scale_offset", True)) and (
        not np.allclose(scales, 1.0) or not np.allclose(offsets, 0.0)
    )
    finite_values = array[np.isfinite(array)]
    if use_transform:
        array = array * scales + offsets
    elif bool(config.get("auto_scale_reflectance", True)) and finite_values.size and float(np.nanpercentile(finite_values, 99.5)) > 2.0:
        array = (array + float(config.get("reflectance_add_offset", 0.0))) / float(config.get("reflectance_scale", 10000.0))

    array = np.clip(array, 0.0, 1.0)
    means = np.asarray(config["means"], dtype=np.float32)[:, None, None]
    stds = np.asarray(config["stds"], dtype=np.float32)[:, None, None]
    array = np.nan_to_num((array - means) / np.maximum(stds, 1e-6), nan=0.0)
    array[:, ~valid] = 0.0
    return array.astype(np.float32), valid


def pad_to_multiple(tensor: torch.Tensor, multiple: int = 32):
    height, width = tensor.shape[-2:]
    pad_h = (multiple - height % multiple) % multiple
    pad_w = (multiple - width % multiple) % multiple
    padded = F.pad(tensor, (0, pad_w, 0, pad_h), mode="replicate")
    return padded, height, width


def starts(length: int, tile_size: int, overlap: int) -> list[int]:
    if tile_size <= overlap:
        raise ValueError("TILE_SIZE debe ser mayor que OVERLAP")
    if length <= tile_size:
        return [0]
    stride = tile_size - overlap
    values = list(range(0, length - tile_size + 1, stride))
    final = length - tile_size
    if values[-1] != final:
        values.append(final)
    return values


def blend_weight(height: int, width: int) -> np.ndarray:
    wy = np.hanning(height) if height > 2 else np.ones(height)
    wx = np.hanning(width) if width > 2 else np.ones(width)
    weight = np.outer(wy, wx).astype(np.float32)
    return np.maximum(weight, 0.05)


# =============================================================================
# 4. PREDICCIÓN PÍXEL A PÍXEL
# =============================================================================
@torch.inference_mode()
def predict_level(level_name: str):
    model, config, run_dir, idx_to_code, idx_to_name, band_indexes = load_model_for_level(level_name)
    if not INPUT_RASTER.exists():
        raise FileNotFoundError(f"No existe INPUT_RASTER: {INPUT_RASTER}")

    tile_size = int(TILE_SIZE_BY_LEVEL[level_name])
    overlap = int(OVERLAP_BY_LEVEL[level_name])
    if tile_size <= overlap:
        raise ValueError("El tamaño de tesela debe ser mayor que el solapamiento.")

    level_dir = ensure_dir(OUTPUT_DIR / level_name)
    temp_dir = ensure_dir(TEMP_DIR / level_name)
    prediction_path = level_dir / f"{MODEL_NAME}_{level_name}_prediccion.tif"
    confidence_path = level_dir / f"{MODEL_NAME}_{level_name}_confianza.tif"
    completed_path = level_dir / "PREDICTION_COMPLETED.json"
    state_path = temp_dir / "estado_prediccion.json"

    if (
        REUSE_COMPLETED_PREDICTIONS
        and prediction_path.exists()
        and confidence_path.exists()
        and completed_path.exists()
    ):
        print(
            f"{level_name} ya tiene una predicción final completa. "
            "Se reutilizará para calcular las métricas."
        )
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        return (
            prediction_path,
            confidence_path,
            run_dir,
            idx_to_code,
            idx_to_name,
        )

    with rasterio.open(INPUT_RASTER) as src:
        height, width = src.height, src.width
        num_classes = len(idx_to_code)
        if max(band_indexes) > src.count:
            raise ValueError(
                f"La imagen tiene {src.count} bandas y se solicitaron {band_indexes}"
            )

        row_starts = starts(height, tile_size, overlap)
        col_starts = starts(width, tile_size, overlap)
        tile_positions = [
            (row, col)
            for row in row_starts
            for col in col_starts
        ]
        total_tiles = len(tile_positions)

        probability_file = temp_dir / "probability_sum.dat"
        weight_file = temp_dir / "weight_sum.dat"
        selected_checkpoint = checkpoint_path(run_dir)
        expected_state = {
            "height": height,
            "width": width,
            "num_classes": num_classes,
            "tile_size": tile_size,
            "overlap": overlap,
            "level": level_name,
            "input_raster": str(INPUT_RASTER.resolve()),
            "run_dir": str(run_dir.resolve()),
            "checkpoint": str(selected_checkpoint.resolve()),
            "band_indexes": list(band_indexes),
        }

        next_tile_index = 0
        can_resume = (
            RESUME_PREDICTION
            and state_path.exists()
            and probability_file.exists()
            and weight_file.exists()
        )

        if can_resume:
            try:
                state = load_json(state_path)
                compatible = all(
                    state.get(key) == value
                    for key, value in expected_state.items()
                )
                if compatible:
                    next_tile_index = int(state.get("next_tile_index", 0))
                    probability_sum = np.memmap(
                        probability_file,
                        dtype="float32",
                        mode="r+",
                        shape=(num_classes, height, width),
                    )
                    weight_sum = np.memmap(
                        weight_file,
                        dtype="float32",
                        mode="r+",
                        shape=(height, width),
                    )
                    print(
                        f"Reanudando {level_name} desde la tesela "
                        f"{next_tile_index} de {total_tiles}."
                    )
                else:
                    can_resume = False
            except Exception as exc:
                warnings.warn(
                    f"No se pudo reanudar la predicción: {exc}. Se reinicia.",
                    RuntimeWarning,
                )
                can_resume = False

        if not can_resume:
            for path in (probability_file, weight_file, state_path):
                if path.exists():
                    path.unlink()
            probability_sum = np.memmap(
                probability_file,
                dtype="float32",
                mode="w+",
                shape=(num_classes, height, width),
            )
            weight_sum = np.memmap(
                weight_file,
                dtype="float32",
                mode="w+",
                shape=(height, width),
            )
            probability_sum[:] = 0
            weight_sum[:] = 0
            probability_sum.flush()
            weight_sum.flush()
            next_tile_index = 0

        validity_vrt = None
        validity_src = None
        if VALIDITY_MASK_RASTER is not None:
            validity_src = rasterio.open(VALIDITY_MASK_RASTER)
            validity_vrt = WarpedVRT(
                validity_src,
                crs=src.crs,
                transform=src.transform,
                width=src.width,
                height=src.height,
                resampling=Resampling.nearest,
                nodata=0,
            )

        progress = tqdm(
            range(next_tile_index, total_tiles),
            total=total_tiles,
            initial=next_tile_index,
            desc=f"Predicción {level_name}",
            leave=False,
            mininterval=5.0,
            dynamic_ncols=False,
        )

        try:
            for tile_index in progress:
                row, col = tile_positions[tile_index]
                tile_h = min(tile_size, height - row)
                tile_w = min(tile_size, width - col)
                window = Window(col, row, tile_w, tile_h)

                image, valid = reflectance_window(
                    src,
                    window,
                    band_indexes,
                    config,
                )
                if validity_vrt is not None:
                    valid &= validity_vrt.read(1, window=window) > 0

                if valid.any():
                    tensor = torch.from_numpy(image).unsqueeze(0).to(
                        DEVICE,
                        non_blocking=False,
                    )
                    tensor, original_h, original_w = pad_to_multiple(tensor, 32)

                    with amp_context():
                        logits = model(tensor)
                        tile_probabilities = torch.softmax(logits, dim=1)

                    tile_probabilities = (
                        tile_probabilities[0, :, :original_h, :original_w]
                        .float()
                        .cpu()
                        .numpy()
                    )
                    weight = (
                        blend_weight(tile_h, tile_w)
                        * valid.astype(np.float32)
                    )

                    probability_sum[
                        :,
                        row:row + tile_h,
                        col:col + tile_w,
                    ] += tile_probabilities * weight[None, :, :]
                    weight_sum[
                        row:row + tile_h,
                        col:col + tile_w,
                    ] += weight

                    del tensor, logits, tile_probabilities, weight

                next_index = tile_index + 1

                if next_index % FLUSH_EVERY_TILES == 0 or next_index == total_tiles:
                    probability_sum.flush()
                    weight_sum.flush()
                    save_json(
                        {
                            **expected_state,
                            "next_tile_index": next_index,
                            "total_tiles": total_tiles,
                            "model": MODEL_NAME,
                            "level": level_name,
                        },
                        state_path,
                    )

                if (
                    DEVICE.type == "cuda"
                    and next_index % EMPTY_CACHE_EVERY_TILES == 0
                ):
                    torch.cuda.synchronize()
                    torch.cuda.empty_cache()
                    gc.collect()
        finally:
            progress.close()
            probability_sum.flush()
            weight_sum.flush()
            if validity_vrt is not None:
                validity_vrt.close()
            if validity_src is not None:
                validity_src.close()

        class_profile = src.profile.copy()
        class_profile.update(
            count=1,
            dtype="uint16",
            nodata=OUTPUT_NODATA,
            compress="lzw",
            tiled=True,
        )
        confidence_profile = src.profile.copy()
        confidence_profile.update(
            count=1,
            dtype="float32",
            nodata=0.0,
            compress="lzw",
            tiled=True,
        )
        code_array = np.asarray(
            [idx_to_code[index] for index in range(num_classes)],
            dtype=np.uint16,
        )

        with rasterio.open(
            prediction_path,
            "w",
            **class_profile,
        ) as dst_class, rasterio.open(
            confidence_path,
            "w",
            **confidence_profile,
        ) as dst_conf:
            block = 256
            for row in tqdm(
                range(0, height, block),
                desc="Escritura GeoTIFF",
            ):
                h = min(block, height - row)
                probabilities = np.asarray(
                    probability_sum[:, row:row + h, :],
                    dtype=np.float32,
                )
                weights = np.asarray(
                    weight_sum[row:row + h, :],
                    dtype=np.float32,
                )
                valid = weights > 0
                probabilities = np.divide(
                    probabilities,
                    weights[None, :, :],
                    out=np.zeros_like(probabilities),
                    where=valid[None, :, :],
                )
                indices = probabilities.argmax(axis=0)
                confidence = probabilities.max(axis=0).astype(np.float32)
                prediction = np.full(
                    (h, width),
                    OUTPUT_NODATA,
                    dtype=np.uint16,
                )
                prediction[valid] = code_array[indices[valid]]
                confidence[~valid] = 0.0
                output_window = Window(0, row, width, h)
                dst_class.write(prediction, 1, window=output_window)
                dst_conf.write(confidence, 1, window=output_window)

                del probabilities, weights, indices, confidence, prediction
                gc.collect()

        del probability_sum, weight_sum
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

        save_json(
            {
                "completed": True,
                "model": MODEL_NAME,
                "level": level_name,
                "input_raster": str(INPUT_RASTER),
                "run_dir": str(run_dir),
                "checkpoint": str(checkpoint_path(run_dir)),
                "prediction": str(prediction_path),
                "confidence": str(confidence_path),
                "tile_size": tile_size,
                "overlap": overlap,
                "band_indexes": list(band_indexes),
            },
            completed_path,
        )

        for path in (probability_file, weight_file, state_path):
            try:
                path.unlink()
            except OSError:
                pass

    return prediction_path, confidence_path, run_dir, idx_to_code, idx_to_name


# =============================================================================
# 5. MÉTRICAS GLOBALES Y POR CLASE
# =============================================================================
def metrics_from_confusion(matrix: np.ndarray):
    matrix = matrix.astype(np.float64)
    tp = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    total = matrix.sum()
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) > 0)
    iou = np.divide(tp, support + predicted - tp, out=np.zeros_like(tp), where=(support + predicted - tp) > 0)
    active = support > 0
    accuracy = float(tp.sum() / total) if total else 0.0
    expected = float((support * predicted).sum() / (total ** 2)) if total else 0.0
    kappa = float((accuracy - expected) / (1.0 - expected)) if total and expected < 1.0 else 0.0
    weights = support / total if total else np.zeros_like(support)
    global_metrics = {
        "accuracy": accuracy,
        "macro_precision": float(precision[active].mean()) if active.any() else 0.0,
        "macro_recall": float(recall[active].mean()) if active.any() else 0.0,
        "macro_f1": float(f1[active].mean()) if active.any() else 0.0,
        "mean_iou": float(iou[active].mean()) if active.any() else 0.0,
        "weighted_f1": float((f1 * weights).sum()) if total else 0.0,
        "kappa": kappa,
        "evaluated_pixels": int(total),
    }
    return global_metrics, precision, recall, f1, iou, support, predicted, tp


def evaluate_prediction(prediction_path: Path, reference_path: Path, idx_to_code, idx_to_name, output_dir: Path):
    if not reference_path.exists():
        warnings.warn(f"No existe la referencia; se omiten métricas: {reference_path}", RuntimeWarning)
        return None

    codes = np.asarray([idx_to_code[i] for i in sorted(idx_to_code)], dtype=np.int64)
    names = [idx_to_name[i] for i in sorted(idx_to_code)]
    code_to_index = {int(code): i for i, code in enumerate(codes)}
    confusion = np.zeros((len(codes), len(codes)), dtype=np.int64)
    unknown_reference = 0

    maximum_code = int(codes.max())
    lookup = np.full(maximum_code + 1, -1, dtype=np.int16)
    lookup[codes] = np.arange(len(codes), dtype=np.int16)

    with rasterio.open(prediction_path) as pred, rasterio.open(reference_path) as ref:
        with WarpedVRT(ref, crs=pred.crs, transform=pred.transform, width=pred.width, height=pred.height, resampling=Resampling.nearest, nodata=ref.nodata) as aligned:
            for _, window in pred.block_windows(1):
                y_pred = pred.read(1, window=window)
                y_true = aligned.read(1, window=window)
                valid = y_pred != OUTPUT_NODATA
                valid &= np.isfinite(y_true)
                if aligned.nodata is not None:
                    valid &= y_true != aligned.nodata
                if not valid.any():
                    continue
                true_values = y_true[valid].astype(np.int64)
                pred_values = y_pred[valid].astype(np.int64)
                known = np.isin(true_values, codes) & np.isin(pred_values, codes)
                unknown_reference += int((~np.isin(true_values, codes)).sum())
                true_values = true_values[known]
                pred_values = pred_values[known]
                if true_values.size == 0:
                    continue
                if (
                    true_values.max(initial=0) > maximum_code
                    or pred_values.max(initial=0) > maximum_code
                ):
                    raise ValueError(
                        "Se encontraron códigos fuera del rango definido "
                        "después de aplicar el filtro de clases conocidas."
                    )

                true_idx = lookup[true_values].astype(np.int64, copy=False)
                pred_idx = lookup[pred_values].astype(np.int64, copy=False)
                encoded = len(codes) * true_idx + pred_idx
                confusion += np.bincount(encoded, minlength=len(codes) ** 2).reshape(len(codes), len(codes))

    global_metrics, precision, recall, f1, iou, support, predicted, tp = metrics_from_confusion(confusion)
    global_metrics["unknown_reference_pixels"] = int(unknown_reference)
    global_metrics["evaluation_note"] = EVALUATION_NOTE

    missing_reference = [
        int(codes[index])
        for index, value in enumerate(support)
        if int(value) == 0
    ]
    global_metrics["classes_without_reference_pixels"] = missing_reference

    class_metrics = pd.DataFrame({
        "class_code": codes,
        "class_name": names,
        "support_reference": support.astype(np.int64),
        "support_prediction": predicted.astype(np.int64),
        "true_positive": tp.astype(np.int64),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
    })
    confusion_df = pd.DataFrame(confusion, index=names, columns=names)

    row_totals = confusion.sum(axis=1, keepdims=True)
    confusion_normalized = np.divide(
        confusion,
        row_totals,
        out=np.zeros_like(confusion, dtype=np.float64),
        where=row_totals > 0,
    )
    confusion_normalized_df = pd.DataFrame(
        confusion_normalized,
        index=names,
        columns=names,
    )

    ensure_dir(output_dir)
    class_metrics.to_csv(
        output_dir / "metricas_por_clase_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )
    confusion_df.to_csv(
        output_dir / "matriz_confusion_prediccion.csv",
        encoding="utf-8-sig",
    )
    confusion_normalized_df.to_csv(
        output_dir / "matriz_confusion_normalizada_prediccion.csv",
        encoding="utf-8-sig",
    )
    with (
        output_dir / "metricas_globales_prediccion.json"
    ).open("w", encoding="utf-8") as file:
        json.dump(global_metrics, file, indent=2, ensure_ascii=False)
    try:
        with pd.ExcelWriter(output_dir / "metricas_prediccion.xlsx", engine="openpyxl") as writer:
            pd.DataFrame([global_metrics]).to_excel(
                writer,
                sheet_name="globales",
                index=False,
            )
            class_metrics.to_excel(
                writer,
                sheet_name="por_clase",
                index=False,
            )
            confusion_df.to_excel(
                writer,
                sheet_name="confusion",
            )
            confusion_normalized_df.to_excel(
                writer,
                sheet_name="confusion_normalizada",
            )
    except Exception as exc:
        warnings.warn(f"No se pudo crear Excel: {exc}", RuntimeWarning)

    if missing_reference:
        warnings.warn(
            "La muestra de referencia no contiene píxeles para las clases: "
            f"{missing_reference}. Sus métricas no son interpretables.",
            RuntimeWarning,
        )

    if SAVE_CONFUSION_PLOT:
        figure, axis = plt.subplots(figsize=(10, 9))
        image = axis.imshow(confusion, interpolation="nearest")
        axis.set_title(f"Matriz de confusión - {MODEL_NAME}")
        axis.set_xlabel("Predicción")
        axis.set_ylabel("Referencia")
        axis.set_xticks(range(len(names)), names, rotation=90, fontsize=7)
        axis.set_yticks(range(len(names)), names, fontsize=7)
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
        figure.tight_layout()
        figure.savefig(
            output_dir / "matriz_confusion_prediccion.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.close(figure)
    return global_metrics


# =============================================================================
# 6. EJECUCIÓN
# =============================================================================
def main():
    if REQUIRE_CUDA and DEVICE.type != "cuda":
        raise RuntimeError("CUDA/ROCm no está disponible; cambia REQUIRE_CUDA a False para CPU.")
    ensure_dir(OUTPUT_DIR)
    print(f"Modelo: {MODEL_NAME}")
    print(f"Dispositivo: {DEVICE}")
    print(f"PyTorch: {torch.__version__}")
    print(f"ROCm/HIP: {torch.version.hip}")
    print(f"Teselas por nivel: {TILE_SIZE_BY_LEVEL}")
    print(f"Solapamiento por nivel: {OVERLAP_BY_LEVEL}")
    if DEVICE.type == "cuda":
        print(f"Acelerador: {torch.cuda.get_device_name(0)}")
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"VRAM disponible: {total_vram:.2f} GB")

    summaries = []

    for position, level_name in enumerate(
        LEVELS_TO_PREDICT,
        start=1,
    ):
        print("\n" + "=" * 80)
        print(
            f"PREDICCIÓN {position}/{len(LEVELS_TO_PREDICT)}: "
            f"{level_name}"
        )
        print("=" * 80)

        try:
            (
                prediction_path,
                confidence_path,
                run_dir,
                idx_to_code,
                idx_to_name,
            ) = predict_level(level_name)

            metrics = evaluate_prediction(
                prediction_path,
                REFERENCE_BY_LEVEL[level_name],
                idx_to_code,
                idx_to_name,
                OUTPUT_DIR / level_name / "metricas",
            )

            summaries.append({
                "model": MODEL_NAME,
                "level": level_name,
                "status": "completed",
                "run_dir": str(run_dir),
                "prediction": str(prediction_path),
                "confidence": str(confidence_path),
                **(metrics or {}),
            })

        except KeyboardInterrupt:
            print(
                f"\nLa ejecución fue interrumpida durante {level_name}. "
                "El estado temporal se conserva para reanudar."
            )
            summaries.append({
                "model": MODEL_NAME,
                "level": level_name,
                "status": "interrupted",
            })
            break

        except Exception as error:
            print(
                f"\nNo fue posible completar {level_name}: {error}"
            )
            traceback.print_exc()
            summaries.append({
                "model": MODEL_NAME,
                "level": level_name,
                "status": "error",
                "error": str(error),
            })
            print(
                "El error quedó registrado. Se liberarán los recursos y "
                "se intentará continuar con el siguiente nivel."
            )

        finally:
            safe_cuda_cleanup()

        if level_name != LEVELS_TO_PREDICT[-1]:
            print(
                f"\nProceso de {level_name} finalizado. "
                "Se continuará con nivel_2."
            )

    summary_df = pd.DataFrame(summaries)
    summary_csv = OUTPUT_DIR / "resumen_predicciones.csv"
    summary_excel = OUTPUT_DIR / "resumen_predicciones.xlsx"

    summary_df.to_csv(
        summary_csv,
        index=False,
        encoding="utf-8-sig",
    )

    try:
        summary_df.to_excel(
            summary_excel,
            index=False,
            engine="openpyxl",
        )
    except Exception as error:
        warnings.warn(
            f"No se pudo crear el resumen Excel: {error}",
            RuntimeWarning,
        )

    print("\n" + "=" * 80)
    print("RESUMEN DE PREDICCIÓN Y EVALUACIÓN")
    print("=" * 80)
    print(summary_df)
    print(f"\nResumen CSV: {summary_csv}")

    return summary_df


# Ejecución directa al copiar todo el TXT en una sola celda de Jupyter.
summary_df = main()


Modelo: ResNet50-UNet
Dispositivo: cuda
PyTorch: 2.9.1+rocm7.2.1
ROCm/HIP: 7.2.53211-158bd99533
Teselas por nivel: {'nivel_1': 256, 'nivel_2': 256}
Solapamiento por nivel: {'nivel_1': 32, 'nivel_2': 32}
Acelerador: AMD Radeon RX 9060 XT
VRAM disponible: 15.92 GB

PREDICCIÓN 1/2: nivel_1


Escritura GeoTIFF: 100%|██████████| 20/20 [00:03<00:00,  6.06it/s]   



Proceso de nivel_1 finalizado. Se continuará con nivel_2.

PREDICCIÓN 2/2: nivel_2
nivel_2 ya tiene una predicción final completa. Se reutilizará para calcular las métricas.

RESUMEN DE PREDICCIÓN Y EVALUACIÓN
           model    level     status  \
0  ResNet50-UNet  nivel_1  completed   
1  ResNet50-UNet  nivel_2  completed   

                                             run_dir  \
0  C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_S...   
1  C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_S...   

                                          prediction  \
0  C:\Users\Ivan sua\anaconda3\Predicciones_ResNe...   
1  C:\Users\Ivan sua\anaconda3\Predicciones_ResNe...   

                                          confidence  accuracy  \
0  C:\Users\Ivan sua\anaconda3\Predicciones_ResNe...  0.096341   
1  C:\Users\Ivan sua\anaconda3\Predicciones_ResNe...  0.045393   

   macro_precision  macro_recall  macro_f1  mean_iou  weighted_f1     kappa  \
0         0.156085      0.084149  0.074859  0.040404

C:\Users\Ivan sua\AppData\Local\Temp\ipykernel_2672\4111688554.py:1130: RuntimeWarning: La muestra de referencia no contiene píxeles para las clases: [13]. Sus métricas no son interpretables.
  warnings.warn(


In [1]:

# =============================================================================
# DIAGNÓSTICO INTEGRAL DE PREDICCIONES RESNET-50
# Segmentación semántica LULC — Nivel 1 y Nivel 2
#
# EJECUCIÓN:
# 1. Copia todo este archivo en UNA sola celda de Jupyter.
# 2. Ajusta únicamente las rutas de la sección 1.
# 3. Ejecuta la celda.
#
# El código valida:
# - correspondencia entre índices internos y códigos temáticos;
# - distribución de clases y posible colapso del modelo;
# - alineación espacial, CRS, resolución, extensión y transformación;
# - desplazamientos de varios píxeles entre predicción y referencia;
# - orden, escala y normalización de las cuatro bandas;
# - consistencia de config.json, checkpoint y métricas del entrenamiento;
# - posibles duplicados entre train, validation y test;
# - diferencia entre métricas internas y predicción externa;
# - Kappa observado, acuerdo esperado, desacuerdo de cantidad y asignación;
# - remapeo óptimo de códigos para detectar permutaciones;
# - confianza en píxeles correctos e incorrectos;
# - ausencia de categorías en la referencia;
# - diferencia temporal entre imagen y referencia.
#
# No usa __file__, argparse, subprocess ni SystemExit.
# =============================================================================

from __future__ import annotations

import gc
import json
import math
import traceback
import warnings
from collections import defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform_bounds

try:
    from scipy.optimize import linear_sum_assignment
    SCIPY_AVAILABLE = True
except Exception:
    linear_sum_assignment = None
    SCIPY_AVAILABLE = False


# =============================================================================
# 1. RUTAS Y PARÁMETROS QUE DEBES REVISAR
# =============================================================================

# Carpeta donde quedaron las predicciones de ambos niveles.
PREDICTION_ROOT = Path(
    r"C:\Users\Ivan sua\anaconda3\Predicciones_ResNet50_Segmentacion"
)

# Puede dejarse en None porque el código intenta recuperarla desde
# PREDICTION_COMPLETED.json.
INPUT_RASTER = None
# Ejemplo:
# INPUT_RASTER = Path(r"D:\Imagenes\Sentinel2_Almeidas_2025.tif")

# Puede dejarse en None si PREDICTION_COMPLETED.json conserva run_dir.
MODEL_ROOT = None
# Ejemplo:
# MODEL_ROOT = Path(r"D:\Modelos_ResNet50_Segmentacion")

# Rásteres de referencia. Estas rutas sí deben existir.
REFERENCE_BY_LEVEL = {
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel1.tif"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel2.tif"
    ),
}

# Rutas manuales opcionales. None activa la búsqueda automática.
PREDICTION_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}
CONFIDENCE_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}
RUN_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}

# Máscara opcional de validez: valor > 0 = evaluar.
VALIDITY_MASK_BY_LEVEL = {
    "nivel_1": None,
    "nivel_2": None,
}

# Fechas para interpretar correctamente la evaluación.
IMAGE_YEAR = 2025
REFERENCE_YEAR_BY_LEVEL = {
    "nivel_1": 2022,
    "nivel_2": 2022,
}

# Carpeta de resultados del diagnóstico.
DIAGNOSTIC_OUTPUT = (
    PREDICTION_ROOT / "DIAGNOSTICO_INTEGRAL_RESNET50"
)

LEVELS = ("nivel_1", "nivel_2")

# Evaluación completa de todos los píxeles válidos.
FULL_PIXEL_EVALUATION = True

# Diagnóstico de desplazamiento espacial.
SHIFT_MAX_PIXELS = 5
SHIFT_MAX_DIMENSION = 700

# Muestreo radiométrico de la imagen multibanda.
RADIOMETRY_MAX_DIMENSION = 600

# Número máximo de chips por split para revisar su georreferenciación.
MAX_CHIPS_FOR_SPATIAL_SPLIT_CHECK = 500

# Umbrales orientativos para emitir alertas.
MAPPING_GAIN_ALERT = 0.10
SHIFT_GAIN_ALERT = 0.03
RADIOMETRIC_MEAN_SHIFT_ALERT = 2.0
EXTERNAL_METRIC_DROP_ALERT = 0.20
DOMINANT_CLASS_ALERT = 0.70
HIGH_CONFIDENCE_THRESHOLD = 0.80

# Diagnóstico 100 % CPU. No carga checkpoints con PyTorch ni inicializa ROCm/GPU.
CPU_ONLY_DIAGNOSTIC = True

# Fuerza recolección de memoria después de bloques intensivos.
AGGRESSIVE_GC = True


# =============================================================================
# 2. CLASES TEMÁTICAS ESPERADAS
# =============================================================================

DEFAULT_CLASSES = {
    "nivel_1": {
        1: "2. Territorios agrícolas",
        2: "3. Bosques y áreas seminaturales",
        3: "5. Superficies de agua",
        4: "1. Territorios artificializados",
        5: "4. Áreas húmedas",
    },
    "nivel_2": {
        1: "2.4 Áreas agrícolas heterogéneas",
        2: "3.2 Vegetación herbácea o arbustiva",
        3: "3.1 Bosques",
        4: "2.3 Pastos",
        5: "5.1 Aguas continentales",
        6: "1.1 Zonas urbanizadas",
        7: "1.3 Extracción minera y escombreras",
        8: "4.1 Áreas húmedas continentales",
        9: "1.2 Zonas industriales, comerciales y redes",
        10: "1.4 Zonas verdes artificializadas no agrícolas",
        11: "2.1 Cultivos transitorios",
        12: "2.2 Cultivos permanentes",
        13: "3.3 Áreas abiertas con poca o ninguna vegetación",
    },
}


# =============================================================================
# 3. FUNCIONES GENERALES
# =============================================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_json(path: str | Path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        with path.open("r", encoding="utf-8") as file:
            return json.load(file)
    except Exception as error:
        warnings.warn(
            f"No se pudo leer {path}: {error}",
            RuntimeWarning,
        )
        return default


def to_serializable(value):
    if isinstance(value, dict):
        return {
            str(key): to_serializable(item)
            for key, item in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [to_serializable(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value) if not isinstance(value, str) else False:
        return None
    return value


def save_json(data, path: str | Path) -> Path:
    path = Path(path)
    ensure_dir(path.parent)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as file:
        json.dump(
            to_serializable(data),
            file,
            indent=2,
            ensure_ascii=False,
        )
    temporary.replace(path)
    return path


def normalize_int_dict(data) -> dict[int, object]:
    if not isinstance(data, dict):
        return {}
    result = {}
    for key, value in data.items():
        try:
            result[int(key)] = value
        except Exception:
            continue
    return result


def safe_float(value, default=np.nan) -> float:
    try:
        return float(value)
    except Exception:
        return float(default)


def find_first_existing(paths) -> Path | None:
    for path in paths:
        if path is not None and Path(path).exists():
            return Path(path)
    return None


def severity_rank(value: str) -> int:
    return {
        "CRÍTICO": 0,
        "ALTO": 1,
        "MEDIO": 2,
        "BAJO": 3,
        "INFORMATIVO": 4,
        "CORRECTO": 5,
    }.get(str(value).upper(), 99)


def status_label(condition: bool) -> str:
    return "CORRECTO" if condition else "REVISAR"


def dataframe_or_empty(records, columns=None) -> pd.DataFrame:
    if isinstance(records, pd.DataFrame):
        return records.copy()
    if records:
        return pd.DataFrame(records)
    return pd.DataFrame(columns=columns or [])


# =============================================================================
# 4. LOCALIZACIÓN DE PREDICCIONES, CONFIANZA Y ENTRENAMIENTOS
# =============================================================================

def prediction_level_dir(level_name: str) -> Path:
    return PREDICTION_ROOT / level_name


def find_prediction_marker(level_name: str) -> dict:
    marker = prediction_level_dir(level_name) / "PREDICTION_COMPLETED.json"
    return load_json(marker, default={}) or {}


def find_prediction_path(level_name: str) -> Path:
    forced = PREDICTION_BY_LEVEL.get(level_name)
    if forced is not None:
        path = Path(forced)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    marker = find_prediction_marker(level_name)
    marker_path = marker.get("prediction")
    candidates = []

    if marker_path:
        candidates.append(Path(marker_path))

    level_dir = prediction_level_dir(level_name)
    candidates.extend(
        sorted(
            level_dir.glob(f"*{level_name}*prediccion*.tif"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )
    candidates.extend(
        sorted(
            level_dir.glob("*prediccion*.tif"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

    path = find_first_existing(candidates)
    if path is None:
        raise FileNotFoundError(
            f"No se encontró la predicción de {level_name} en {level_dir}"
        )
    return path


def find_confidence_path(level_name: str) -> Path | None:
    forced = CONFIDENCE_BY_LEVEL.get(level_name)
    if forced is not None:
        path = Path(forced)
        return path if path.exists() else None

    marker = find_prediction_marker(level_name)
    candidates = []

    marker_path = marker.get("confidence")
    if marker_path:
        candidates.append(Path(marker_path))

    level_dir = prediction_level_dir(level_name)
    candidates.extend(
        sorted(
            level_dir.glob(f"*{level_name}*confianza*.tif"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )
    candidates.extend(
        sorted(
            level_dir.glob("*confianza*.tif"),
            key=lambda path: path.stat().st_mtime,
            reverse=True,
        )
    )

    return find_first_existing(candidates)


def valid_run_config(run_dir: Path, level_name: str) -> bool:
    config = load_json(run_dir / "config.json", default={}) or {}
    model = str(config.get("model", "")).lower()
    encoder = str(config.get("encoder", "")).lower()
    task = str(config.get("task", "")).lower()
    level = str(config.get("level", "")).lower()

    return (
        level == level_name
        and task == "segmentacion_semantica"
        and (
            "resnet50" in model
            or encoder == "resnet50"
        )
    )


def find_run_dir(level_name: str) -> Path:
    forced = RUN_BY_LEVEL.get(level_name)
    if forced is not None:
        path = Path(forced)
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    marker = find_prediction_marker(level_name)
    marker_run = marker.get("run_dir")
    if marker_run and Path(marker_run).exists():
        return Path(marker_run)

    roots = []
    if MODEL_ROOT is not None:
        roots.append(Path(MODEL_ROOT))

    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for run_dir in root.glob(f"ResNet50Seg_{level_name}_*"):
            if not valid_run_config(run_dir, level_name):
                continue

            evidence = [
                run_dir / "LEVEL_COMPLETED.json",
                run_dir / "reports" / "resumen_metricas_test.json",
                run_dir / "best_model.pt",
                run_dir / "checkpoints" / "last_checkpoint.pt",
            ]
            existing = [path for path in evidence if path.exists()]
            if not existing:
                continue

            completed = (
                (run_dir / "LEVEL_COMPLETED.json").exists()
                or (
                    run_dir
                    / "reports"
                    / "resumen_metricas_test.json"
                ).exists()
            )
            modified = max(path.stat().st_mtime for path in existing)
            candidates.append(
                (1 if completed else 0, modified, run_dir)
            )

    if not candidates:
        raise FileNotFoundError(
            f"No se pudo localizar la ejecución de entrenamiento para "
            f"{level_name}. Define MODEL_ROOT o RUN_BY_LEVEL."
        )

    return max(
        candidates,
        key=lambda item: (item[0], item[1]),
    )[2]


def resolve_input_raster(level_name: str) -> Path | None:
    if INPUT_RASTER is not None:
        path = Path(INPUT_RASTER)
        return path if path.exists() else None

    marker = find_prediction_marker(level_name)
    marker_input = marker.get("input_raster")
    if marker_input and Path(marker_input).exists():
        return Path(marker_input)

    return None


# =============================================================================
# 5. CONFIGURACIÓN, CLASES Y ENTRENAMIENTO
# =============================================================================

def load_level_context(level_name: str) -> dict:
    prediction_path = find_prediction_path(level_name)
    confidence_path = find_confidence_path(level_name)
    reference_path = Path(REFERENCE_BY_LEVEL[level_name])
    run_dir = find_run_dir(level_name)
    config_path = run_dir / "config.json"
    config = load_json(config_path, default={}) or {}

    idx_to_code = normalize_int_dict(config.get("idx_to_code"))
    idx_to_name = normalize_int_dict(config.get("idx_to_name"))
    code_to_idx = normalize_int_dict(config.get("code_to_idx"))

    if idx_to_code:
        ordered_indexes = sorted(idx_to_code)
        codes = [
            int(idx_to_code[index])
            for index in ordered_indexes
        ]
        names = [
            str(
                idx_to_name.get(
                    index,
                    DEFAULT_CLASSES[level_name].get(
                        int(idx_to_code[index]),
                        f"Clase {idx_to_code[index]}",
                    ),
                )
            )
            for index in ordered_indexes
        ]
    elif code_to_idx:
        ordered = sorted(
            (
                int(index),
                int(code),
            )
            for code, index in code_to_idx.items()
        )
        codes = [code for index, code in ordered]
        names = [
            DEFAULT_CLASSES[level_name].get(
                code,
                f"Clase {code}",
            )
            for code in codes
        ]
    else:
        codes = sorted(DEFAULT_CLASSES[level_name])
        names = [
            DEFAULT_CLASSES[level_name][code]
            for code in codes
        ]

    input_raster = resolve_input_raster(level_name)

    return {
        "level": level_name,
        "prediction_path": prediction_path,
        "confidence_path": confidence_path,
        "reference_path": reference_path,
        "validity_mask": (
            Path(VALIDITY_MASK_BY_LEVEL[level_name])
            if VALIDITY_MASK_BY_LEVEL[level_name] is not None
            else None
        ),
        "run_dir": run_dir,
        "config_path": config_path,
        "config": config,
        "codes": np.asarray(codes, dtype=np.int64),
        "names": names,
        "class_name_by_code": dict(zip(codes, names)),
        "input_raster": input_raster,
    }


def validate_model_configuration(context: dict) -> pd.DataFrame:
    level_name = context["level"]
    config = context["config"]
    run_dir = context["run_dir"]
    codes = context["codes"].tolist()

    checks = []

    def add(
        item,
        observed,
        expected,
        correct,
        recommendation,
    ):
        checks.append(
            {
                "componente": item,
                "valor_observado": str(observed),
                "valor_esperado": str(expected),
                "estado": status_label(bool(correct)),
                "recomendacion": recommendation,
            }
        )

    add(
        "Tarea",
        config.get("task"),
        "segmentacion_semantica",
        str(config.get("task", "")).lower()
        == "segmentacion_semantica",
        "El checkpoint debe corresponder a segmentación píxel a píxel.",
    )
    add(
        "Modelo",
        config.get("model"),
        "ResNet50-UNet",
        "resnet50" in str(config.get("model", "")).lower(),
        "Verificar que no se haya seleccionado el antiguo clasificador por chip.",
    )
    add(
        "Codificador",
        config.get("encoder"),
        "resnet50",
        str(config.get("encoder", "")).lower() == "resnet50",
        "El codificador debe ser ResNet-50.",
    )
    add(
        "Nivel",
        config.get("level"),
        level_name,
        str(config.get("level", "")).lower() == level_name,
        "No mezclar el modelo del nivel 1 con el nivel 2.",
    )

    band_indexes = [
        int(value)
        for value in config.get("band_indexes", [])
    ]
    band_order = [
        str(value).upper()
        for value in config.get("band_order", [])
    ]
    add(
        "Índices de bandas",
        band_indexes,
        [1, 2, 3, 4],
        band_indexes == [1, 2, 3, 4],
        "La predicción debe leer las mismas cuatro posiciones del entrenamiento.",
    )
    add(
        "Orden espectral",
        band_order,
        ["B2", "B3", "B4", "B8"],
        band_order == ["B2", "B3", "B4", "B8"],
        "Confirmar que la cuarta banda sea B8 y no B5.",
    )

    means = config.get("means", [])
    stds = config.get("stds", [])
    add(
        "Medias de entrenamiento",
        means,
        "4 valores",
        isinstance(means, list) and len(means) == 4,
        "Las medias deben calcularse solo con los chips de entrenamiento.",
    )
    add(
        "Desviaciones de entrenamiento",
        stds,
        "4 valores positivos",
        (
            isinstance(stds, list)
            and len(stds) == 4
            and all(safe_float(value, 0) > 0 for value in stds)
        ),
        "Las desviaciones deben corresponder a B2, B3, B4 y B8.",
    )

    add(
        "Función de pérdida",
        config.get("loss"),
        "ce_dice",
        str(config.get("loss", "")).lower() == "ce_dice",
        "Confirmar CrossEntropy + Dice Loss.",
    )
    add(
        "Muestreo ponderado",
        config.get("use_weighted_sampler"),
        False,
        config.get("use_weighted_sampler") is False,
        "En segmentación multiclase por píxel debe permanecer desactivado.",
    )
    add(
        "Pesos por clase",
        config.get("use_class_weights"),
        True,
        config.get("use_class_weights") is True,
        "Mantener los pesos por clase para reducir el sesgo por frecuencia.",
    )

    expected_codes = sorted(DEFAULT_CLASSES[level_name])
    add(
        "Códigos del modelo",
        sorted(codes),
        expected_codes,
        sorted(codes) == expected_codes,
        "Revisar idx_to_code y code_to_idx; un intercambio puede volver negativo el Kappa.",
    )

    add(
        "Mejor checkpoint",
        run_dir / "best_model.pt",
        "Archivo existente",
        (run_dir / "best_model.pt").exists(),
        "La predicción final debe usar best_model.pt.",
    )
    add(
        "Informe de prueba interna",
        run_dir / "reports" / "resumen_metricas_test.json",
        "Archivo existente",
        (
            run_dir
            / "reports"
            / "resumen_metricas_test.json"
        ).exists(),
        "Permite comparar el desempeño interno con la imagen completa.",
    )

    return pd.DataFrame(checks)


def load_training_information(context: dict) -> dict:
    run_dir = context["run_dir"]
    report_dir = run_dir / "reports"

    test_report = load_json(
        report_dir / "resumen_metricas_test.json",
        default={},
    ) or {}

    history_path = run_dir / "history.csv"
    history = (
        pd.read_csv(history_path)
        if history_path.exists()
        else pd.DataFrame()
    )

    dataset_splits_path = (
        report_dir / "dataset" / "dataset_splits.csv"
    )
    dataset_pixels_path = (
        report_dir
        / "dataset"
        / "dataset_pixels_por_clase.csv"
    )

    dataset_splits = (
        pd.read_csv(dataset_splits_path)
        if dataset_splits_path.exists()
        else pd.DataFrame()
    )
    dataset_pixels = (
        pd.read_csv(dataset_pixels_path)
        if dataset_pixels_path.exists()
        else pd.DataFrame()
    )

    # Para este diagnóstico no se abre best_model.pt.
    # Las métricas se recuperan de config.json, history.csv y
    # reports/resumen_metricas_test.json.
    checkpoint_metadata = {}
    checkpoint_path = run_dir / "best_model.pt"

    if checkpoint_path.exists():
        stat = checkpoint_path.stat()
        checkpoint_metadata = {
            "exists": True,
            "path": str(checkpoint_path),
            "size_mb": float(stat.st_size / (1024 ** 2)),
            "modified_at": datetime.fromtimestamp(
                stat.st_mtime
            ).isoformat(timespec="seconds"),
            "note": (
                "Checkpoint no cargado para mantener el diagnóstico "
                "completamente independiente de PyTorch/ROCm."
            ),
        }
    else:
        checkpoint_metadata = {
            "exists": False,
            "path": str(checkpoint_path),
        }

    best_history = {}
    if not history.empty:
        if "val_mean_iou" in history.columns:
            row = history.loc[
                history["val_mean_iou"].astype(float).idxmax()
            ]
            best_history = row.to_dict()
        else:
            best_history = history.iloc[-1].to_dict()

    return {
        "test_report": test_report,
        "history": history,
        "best_history": best_history,
        "dataset_splits": dataset_splits,
        "dataset_pixels": dataset_pixels,
        "checkpoint_metadata": checkpoint_metadata,
    }


# =============================================================================
# 6. METADATOS Y ALINEACIÓN ESPACIAL
# =============================================================================

def raster_metadata(path: Path, label: str) -> dict:
    with rasterio.open(path) as src:
        return {
            "archivo": label,
            "ruta": str(path),
            "crs": str(src.crs),
            "width": int(src.width),
            "height": int(src.height),
            "bandas": int(src.count),
            "dtype": str(src.dtypes[0]),
            "nodata": src.nodata,
            "pixel_x": float(src.transform.a),
            "pixel_y": float(src.transform.e),
            "origen_x": float(src.transform.c),
            "origen_y": float(src.transform.f),
            "bounds_left": float(src.bounds.left),
            "bounds_bottom": float(src.bounds.bottom),
            "bounds_right": float(src.bounds.right),
            "bounds_top": float(src.bounds.top),
            "transform": str(src.transform),
            "descripciones_bandas": str(src.descriptions),
            "escalas": str(src.scales),
            "offsets": str(src.offsets),
        }


def intersection_percentage(bounds_a, bounds_b) -> float:
    left = max(bounds_a.left, bounds_b.left)
    bottom = max(bounds_a.bottom, bounds_b.bottom)
    right = min(bounds_a.right, bounds_b.right)
    top = min(bounds_a.top, bounds_b.top)

    width = max(0.0, right - left)
    height = max(0.0, top - bottom)
    intersection = width * height

    area_a = max(
        0.0,
        (bounds_a.right - bounds_a.left)
        * (bounds_a.top - bounds_a.bottom),
    )
    if area_a <= 0:
        return 0.0
    return intersection / area_a * 100.0


def spatial_alignment_diagnostic(
    prediction_path: Path,
    reference_path: Path,
) -> tuple[pd.DataFrame, dict]:
    if not reference_path.exists():
        raise FileNotFoundError(reference_path)

    with rasterio.open(prediction_path) as pred, rasterio.open(
        reference_path
    ) as ref:
        same_crs = pred.crs == ref.crs
        same_shape = (
            pred.width == ref.width
            and pred.height == ref.height
        )
        same_transform = pred.transform.almost_equals(
            ref.transform,
            precision=1e-9,
        )
        same_resolution = (
            math.isclose(
                abs(pred.transform.a),
                abs(ref.transform.a),
                rel_tol=0,
                abs_tol=1e-9,
            )
            and math.isclose(
                abs(pred.transform.e),
                abs(ref.transform.e),
                rel_tol=0,
                abs_tol=1e-9,
            )
        )

        if same_crs:
            ref_bounds_pred_crs = ref.bounds
        else:
            transformed = transform_bounds(
                ref.crs,
                pred.crs,
                *ref.bounds,
                densify_pts=21,
            )
            ref_bounds_pred_crs = rasterio.coords.BoundingBox(
                *transformed
            )

        overlap_percent = intersection_percentage(
            pred.bounds,
            ref_bounds_pred_crs,
        )

        pixel_offset_col = np.nan
        pixel_offset_row = np.nan
        if same_crs:
            col, row = (~pred.transform) * (
                ref.bounds.left,
                ref.bounds.top,
            )
            pixel_offset_col = float(col)
            pixel_offset_row = float(row)

        rows = [
            {
                "prueba": "CRS idéntico",
                "resultado": same_crs,
                "detalle": (
                    f"Predicción: {pred.crs} | Referencia: {ref.crs}"
                ),
            },
            {
                "prueba": "Dimensiones idénticas",
                "resultado": same_shape,
                "detalle": (
                    f"Predicción: {pred.width}×{pred.height} | "
                    f"Referencia: {ref.width}×{ref.height}"
                ),
            },
            {
                "prueba": "Transformación afín idéntica",
                "resultado": same_transform,
                "detalle": (
                    f"Predicción: {pred.transform} | "
                    f"Referencia: {ref.transform}"
                ),
            },
            {
                "prueba": "Resolución idéntica",
                "resultado": same_resolution,
                "detalle": (
                    f"Predicción: {pred.res} | Referencia: {ref.res}"
                ),
            },
            {
                "prueba": "Cobertura espacial común",
                "resultado": overlap_percent > 99.0,
                "detalle": f"{overlap_percent:.4f} % de la predicción",
            },
            {
                "prueba": "Desplazamiento de origen en columnas",
                "resultado": (
                    abs(pixel_offset_col) < 1e-6
                    if np.isfinite(pixel_offset_col)
                    else False
                ),
                "detalle": pixel_offset_col,
            },
            {
                "prueba": "Desplazamiento de origen en filas",
                "resultado": (
                    abs(pixel_offset_row) < 1e-6
                    if np.isfinite(pixel_offset_row)
                    else False
                ),
                "detalle": pixel_offset_row,
            },
        ]

        summary = {
            "same_crs": same_crs,
            "same_shape": same_shape,
            "same_transform": same_transform,
            "same_resolution": same_resolution,
            "overlap_percent": overlap_percent,
            "pixel_offset_col": pixel_offset_col,
            "pixel_offset_row": pixel_offset_row,
        }

    return pd.DataFrame(rows), summary


# =============================================================================
# 7. MÉTRICAS, DISTRIBUCIÓN Y KAPPA
# =============================================================================

def metrics_from_confusion(
    confusion: np.ndarray,
    codes: np.ndarray,
    names: list[str],
) -> tuple[dict, pd.DataFrame, pd.DataFrame]:
    confusion = confusion.astype(np.float64)
    total = confusion.sum()

    row_support = confusion.sum(axis=1)
    column_support = confusion.sum(axis=0)
    true_positive = np.diag(confusion)

    precision = np.divide(
        true_positive,
        column_support,
        out=np.zeros_like(true_positive),
        where=column_support > 0,
    )
    recall = np.divide(
        true_positive,
        row_support,
        out=np.zeros_like(true_positive),
        where=row_support > 0,
    )
    f1 = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(true_positive),
        where=(precision + recall) > 0,
    )
    union = row_support + column_support - true_positive
    iou = np.divide(
        true_positive,
        union,
        out=np.zeros_like(true_positive),
        where=union > 0,
    )

    valid_reference_classes = row_support > 0
    accuracy = (
        true_positive.sum() / total
        if total > 0
        else np.nan
    )

    reference_proportion = np.divide(
        row_support,
        total,
        out=np.zeros_like(row_support),
        where=total > 0,
    )
    predicted_proportion = np.divide(
        column_support,
        total,
        out=np.zeros_like(column_support),
        where=total > 0,
    )

    expected_agreement = float(
        np.sum(
            reference_proportion
            * predicted_proportion
        )
    )
    if total > 0 and expected_agreement < 1:
        kappa = (
            accuracy - expected_agreement
        ) / (1 - expected_agreement)
    else:
        kappa = np.nan

    quantity_disagreement = float(
        0.5
        * np.sum(
            np.abs(
                reference_proportion
                - predicted_proportion
            )
        )
    )
    total_disagreement = (
        1 - accuracy
        if np.isfinite(accuracy)
        else np.nan
    )
    allocation_disagreement = (
        max(
            0.0,
            total_disagreement - quantity_disagreement,
        )
        if np.isfinite(total_disagreement)
        else np.nan
    )

    macro_precision = (
        precision[valid_reference_classes].mean()
        if valid_reference_classes.any()
        else np.nan
    )
    macro_recall = (
        recall[valid_reference_classes].mean()
        if valid_reference_classes.any()
        else np.nan
    )
    macro_f1 = (
        f1[valid_reference_classes].mean()
        if valid_reference_classes.any()
        else np.nan
    )
    mean_iou = (
        iou[valid_reference_classes].mean()
        if valid_reference_classes.any()
        else np.nan
    )
    weighted_f1 = (
        np.sum(f1 * reference_proportion)
        if total > 0
        else np.nan
    )

    globals_dict = {
        "evaluated_pixels": int(total),
        "accuracy": float(accuracy),
        "observed_agreement_po": float(accuracy),
        "expected_agreement_pe": expected_agreement,
        "kappa": float(kappa),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "mean_iou": float(mean_iou),
        "weighted_f1": float(weighted_f1),
        "quantity_disagreement": quantity_disagreement,
        "allocation_disagreement": allocation_disagreement,
        "total_disagreement": float(total_disagreement),
    }

    per_class = pd.DataFrame(
        {
            "class_code": codes.astype(int),
            "class_name": names,
            "reference_pixels": row_support.astype(np.int64),
            "predicted_pixels": column_support.astype(np.int64),
            "true_positive": true_positive.astype(np.int64),
            "reference_percentage": reference_proportion * 100,
            "predicted_percentage": predicted_proportion * 100,
            "percentage_difference": (
                predicted_proportion
                - reference_proportion
            )
            * 100,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
        }
    )

    normalized = np.divide(
        confusion,
        row_support[:, None],
        out=np.zeros_like(confusion),
        where=row_support[:, None] > 0,
    )
    normalized_df = pd.DataFrame(
        normalized,
        index=[
            f"{code} - {name}"
            for code, name in zip(codes, names)
        ],
        columns=[
            f"{code} - {name}"
            for code, name in zip(codes, names)
        ],
    )

    return globals_dict, per_class, normalized_df


def evaluate_prediction_censally(
    context: dict,
) -> dict:
    prediction_path = context["prediction_path"]
    reference_path = context["reference_path"]
    confidence_path = context["confidence_path"]
    validity_path = context["validity_mask"]
    codes = context["codes"]
    names = context["names"]

    class_count = len(codes)
    code_to_index = {
        int(code): index
        for index, code in enumerate(codes)
    }

    maximum_code = int(max(codes))
    lookup = np.full(
        maximum_code + 1,
        -1,
        dtype=np.int16,
    )
    for code, index in code_to_index.items():
        lookup[code] = index

    confusion = np.zeros(
        (class_count, class_count),
        dtype=np.int64,
    )

    counters = defaultdict(int)

    confidence_bins = np.linspace(0, 1, 11)
    confidence_count = np.zeros(10, dtype=np.int64)
    confidence_correct = np.zeros(10, dtype=np.int64)
    confidence_sum = np.zeros(10, dtype=np.float64)
    confidence_correct_sum = 0.0
    confidence_wrong_sum = 0.0
    confidence_correct_count = 0
    confidence_wrong_count = 0
    high_confidence_wrong = 0

    with rasterio.open(prediction_path) as pred, rasterio.open(
        reference_path
    ) as ref_src:
        reference_vrt = WarpedVRT(
            ref_src,
            crs=pred.crs,
            transform=pred.transform,
            width=pred.width,
            height=pred.height,
            resampling=Resampling.nearest,
            nodata=ref_src.nodata,
        )

        confidence_src = None
        confidence_vrt = None
        if confidence_path is not None and confidence_path.exists():
            confidence_src = rasterio.open(confidence_path)
            confidence_vrt = WarpedVRT(
                confidence_src,
                crs=pred.crs,
                transform=pred.transform,
                width=pred.width,
                height=pred.height,
                resampling=Resampling.bilinear,
                nodata=confidence_src.nodata,
            )

        validity_src = None
        validity_vrt = None
        if validity_path is not None and validity_path.exists():
            validity_src = rasterio.open(validity_path)
            validity_vrt = WarpedVRT(
                validity_src,
                crs=pred.crs,
                transform=pred.transform,
                width=pred.width,
                height=pred.height,
                resampling=Resampling.nearest,
                nodata=0,
            )

        try:
            # Se usa el generador directamente para no almacenar todas
            # las ventanas del ráster simultáneamente en memoria.
            for _, window in pred.block_windows(1):
                predicted = pred.read(
                    1,
                    window=window,
                ).astype(np.int64)

                reference = reference_vrt.read(
                    1,
                    window=window,
                ).astype(np.int64)

                counters["total_overlap_pixels"] += predicted.size

                valid = np.ones(
                    predicted.shape,
                    dtype=bool,
                )

                if pred.nodata is not None:
                    valid &= predicted != pred.nodata
                    counters["prediction_nodata_pixels"] += int(
                        np.count_nonzero(
                            predicted == pred.nodata
                        )
                    )

                if ref_src.nodata is not None:
                    valid &= reference != ref_src.nodata
                    counters["reference_nodata_pixels"] += int(
                        np.count_nonzero(
                            reference == ref_src.nodata
                        )
                    )

                if validity_vrt is not None:
                    validity = validity_vrt.read(
                        1,
                        window=window,
                    )
                    valid &= validity > 0
                    counters["validity_excluded_pixels"] += int(
                        np.count_nonzero(validity <= 0)
                    )

                predicted_known = (
                    (predicted >= 0)
                    & (predicted <= maximum_code)
                )
                reference_known = (
                    (reference >= 0)
                    & (reference <= maximum_code)
                )

                predicted_index = np.full(
                    predicted.shape,
                    -1,
                    dtype=np.int16,
                )
                reference_index = np.full(
                    reference.shape,
                    -1,
                    dtype=np.int16,
                )

                predicted_index[predicted_known] = lookup[
                    predicted[predicted_known]
                ]
                reference_index[reference_known] = lookup[
                    reference[reference_known]
                ]

                unknown_pred = (
                    valid
                    & (predicted_index < 0)
                )
                unknown_ref = (
                    valid
                    & (reference_index < 0)
                )

                counters["unknown_prediction_pixels"] += int(
                    unknown_pred.sum()
                )
                counters["unknown_reference_pixels"] += int(
                    unknown_ref.sum()
                )

                metric_valid = (
                    valid
                    & (predicted_index >= 0)
                    & (reference_index >= 0)
                )

                if not metric_valid.any():
                    continue

                true_values = reference_index[
                    metric_valid
                ].astype(np.int64)
                pred_values = predicted_index[
                    metric_valid
                ].astype(np.int64)

                encoded = (
                    true_values * class_count
                    + pred_values
                )
                confusion += np.bincount(
                    encoded,
                    minlength=class_count * class_count,
                ).reshape(class_count, class_count)

                if confidence_vrt is not None:
                    confidence = confidence_vrt.read(
                        1,
                        window=window,
                    ).astype(np.float32)
                    confidence_values = np.clip(
                        confidence[metric_valid],
                        0,
                        1,
                    )
                    correct = true_values == pred_values

                    bin_index = np.minimum(
                        (
                            confidence_values * 10
                        ).astype(np.int64),
                        9,
                    )

                    confidence_count += np.bincount(
                        bin_index,
                        minlength=10,
                    )
                    confidence_correct += np.bincount(
                        bin_index,
                        weights=correct.astype(np.int64),
                        minlength=10,
                    ).astype(np.int64)
                    confidence_sum += np.bincount(
                        bin_index,
                        weights=confidence_values,
                        minlength=10,
                    )

                    if correct.any():
                        confidence_correct_sum += float(
                            confidence_values[correct].sum()
                        )
                        confidence_correct_count += int(
                            correct.sum()
                        )
                    if (~correct).any():
                        confidence_wrong_sum += float(
                            confidence_values[~correct].sum()
                        )
                        confidence_wrong_count += int(
                            (~correct).sum()
                        )
                        high_confidence_wrong += int(
                            np.count_nonzero(
                                confidence_values[~correct]
                                >= HIGH_CONFIDENCE_THRESHOLD
                            )
                        )

                del (
                    predicted,
                    reference,
                    predicted_index,
                    reference_index,
                    true_values,
                    pred_values,
                    encoded,
                    valid,
                    metric_valid,
                    predicted_known,
                    reference_known,
                    unknown_pred,
                    unknown_ref,
                )

                if AGGRESSIVE_GC:
                    gc.collect()
        finally:
            reference_vrt.close()
            if confidence_vrt is not None:
                confidence_vrt.close()
            if confidence_src is not None:
                confidence_src.close()
            if validity_vrt is not None:
                validity_vrt.close()
            if validity_src is not None:
                validity_src.close()

    (
        globals_dict,
        per_class,
        confusion_normalized,
    ) = metrics_from_confusion(
        confusion,
        codes,
        names,
    )

    globals_dict.update(
        {
            key: int(value)
            for key, value in counters.items()
        }
    )

    absent_reference = per_class.loc[
        per_class["reference_pixels"] == 0,
        "class_code",
    ].astype(int).tolist()
    absent_prediction = per_class.loc[
        per_class["predicted_pixels"] == 0,
        "class_code",
    ].astype(int).tolist()

    globals_dict[
        "classes_without_reference_pixels"
    ] = absent_reference
    globals_dict[
        "classes_without_predicted_pixels"
    ] = absent_prediction

    confusion_df = pd.DataFrame(
        confusion,
        index=[
            f"{code} - {name}"
            for code, name in zip(codes, names)
        ],
        columns=[
            f"{code} - {name}"
            for code, name in zip(codes, names)
        ],
    )

    confidence_rows = []
    expected_calibration_error = 0.0
    total_confidence = confidence_count.sum()

    for index in range(10):
        count = int(confidence_count[index])
        lower = confidence_bins[index]
        upper = confidence_bins[index + 1]

        mean_confidence = (
            confidence_sum[index] / count
            if count > 0
            else np.nan
        )
        bin_accuracy = (
            confidence_correct[index] / count
            if count > 0
            else np.nan
        )

        if count > 0 and total_confidence > 0:
            expected_calibration_error += (
                count / total_confidence
            ) * abs(bin_accuracy - mean_confidence)

        confidence_rows.append(
            {
                "intervalo": f"[{lower:.1f}, {upper:.1f}]",
                "pixel_count": count,
                "mean_confidence": mean_confidence,
                "accuracy_in_bin": bin_accuracy,
                "calibration_gap": (
                    bin_accuracy - mean_confidence
                    if count > 0
                    else np.nan
                ),
            }
        )

    confidence_df = pd.DataFrame(confidence_rows)

    globals_dict[
        "mean_confidence_correct"
    ] = (
        confidence_correct_sum
        / confidence_correct_count
        if confidence_correct_count > 0
        else np.nan
    )
    globals_dict[
        "mean_confidence_incorrect"
    ] = (
        confidence_wrong_sum
        / confidence_wrong_count
        if confidence_wrong_count > 0
        else np.nan
    )
    globals_dict[
        "high_confidence_incorrect_pixels"
    ] = int(high_confidence_wrong)
    globals_dict[
        "expected_calibration_error"
    ] = float(expected_calibration_error)

    return {
        "global_metrics": globals_dict,
        "per_class": per_class,
        "confusion": confusion_df,
        "confusion_array": confusion,
        "confusion_normalized": confusion_normalized,
        "confidence": confidence_df,
    }


# =============================================================================
# 8. DIAGNÓSTICO DE PERMUTACIÓN DE CÓDIGOS
# =============================================================================

def permutation_diagnostic(
    confusion: np.ndarray,
    codes: np.ndarray,
    names: list[str],
) -> tuple[dict, pd.DataFrame]:
    confusion = np.asarray(confusion, dtype=np.int64)
    total = confusion.sum()

    if total == 0:
        return (
            {
                "identity_accuracy": np.nan,
                "optimal_relabel_accuracy": np.nan,
                "relabel_gain": np.nan,
                "many_to_one_upper_bound": np.nan,
                "method": "sin datos",
            },
            pd.DataFrame(),
        )

    identity_accuracy = np.trace(confusion) / total

    if SCIPY_AVAILABLE:
        row_indexes, column_indexes = linear_sum_assignment(
            -confusion
        )
        method = "Hungarian exacto"
    else:
        method = "Greedy aproximado"
        available_rows = set(range(confusion.shape[0]))
        available_cols = set(range(confusion.shape[1]))
        pairs = []

        while available_rows and available_cols:
            best_pair = None
            best_value = -1
            for row in available_rows:
                for col in available_cols:
                    value = confusion[row, col]
                    if value > best_value:
                        best_value = value
                        best_pair = (row, col)
            row, col = best_pair
            pairs.append((row, col))
            available_rows.remove(row)
            available_cols.remove(col)

        row_indexes = np.asarray(
            [pair[0] for pair in pairs],
            dtype=int,
        )
        column_indexes = np.asarray(
            [pair[1] for pair in pairs],
            dtype=int,
        )

    optimal_correct = confusion[
        row_indexes,
        column_indexes,
    ].sum()
    optimal_accuracy = optimal_correct / total

    many_to_one_correct = confusion.max(axis=0).sum()
    many_to_one_accuracy = many_to_one_correct / total

    rows = []
    for reference_index, predicted_index in zip(
        row_indexes,
        column_indexes,
    ):
        predicted_total = confusion[:, predicted_index].sum()
        matched = confusion[
            reference_index,
            predicted_index,
        ]
        rows.append(
            {
                "predicted_code": int(codes[predicted_index]),
                "predicted_name": names[predicted_index],
                "best_reference_code": int(codes[reference_index]),
                "best_reference_name": names[reference_index],
                "is_identity_mapping": (
                    int(codes[predicted_index])
                    == int(codes[reference_index])
                ),
                "matched_pixels": int(matched),
                "predicted_pixels": int(predicted_total),
                "purity_percentage": (
                    matched / predicted_total * 100
                    if predicted_total > 0
                    else np.nan
                ),
            }
        )

    summary = {
        "identity_accuracy": float(identity_accuracy),
        "optimal_relabel_accuracy": float(optimal_accuracy),
        "relabel_gain": float(
            optimal_accuracy - identity_accuracy
        ),
        "many_to_one_upper_bound": float(
            many_to_one_accuracy
        ),
        "method": method,
        "non_identity_assignments": int(
            sum(
                not row["is_identity_mapping"]
                for row in rows
            )
        ),
    }

    return summary, pd.DataFrame(rows)


# =============================================================================
# 9. DIAGNÓSTICO DE DESPLAZAMIENTO ESPACIAL
# =============================================================================

def shifted_slices(
    height: int,
    width: int,
    dy: int,
    dx: int,
):
    if dy >= 0:
        ref_y = slice(0, height - dy)
        pred_y = slice(dy, height)
    else:
        ref_y = slice(-dy, height)
        pred_y = slice(0, height + dy)

    if dx >= 0:
        ref_x = slice(0, width - dx)
        pred_x = slice(dx, width)
    else:
        ref_x = slice(-dx, width)
        pred_x = slice(0, width + dx)

    return ref_y, ref_x, pred_y, pred_x


def reduced_aligned_arrays(
    context: dict,
) -> tuple[np.ndarray, np.ndarray, float, float]:
    prediction_path = context["prediction_path"]
    reference_path = context["reference_path"]

    with rasterio.open(prediction_path) as pred, rasterio.open(
        reference_path
    ) as ref_src:
        scale = min(
            1.0,
            SHIFT_MAX_DIMENSION
            / max(pred.width, pred.height),
        )
        out_width = max(1, int(round(pred.width * scale)))
        out_height = max(1, int(round(pred.height * scale)))

        predicted = pred.read(
            1,
            out_shape=(out_height, out_width),
            resampling=Resampling.nearest,
        ).astype(np.int64)

        reference_vrt = WarpedVRT(
            ref_src,
            crs=pred.crs,
            transform=pred.transform,
            width=pred.width,
            height=pred.height,
            resampling=Resampling.nearest,
            nodata=ref_src.nodata,
        )
        try:
            reference = reference_vrt.read(
                1,
                out_shape=(out_height, out_width),
                resampling=Resampling.nearest,
            ).astype(np.int64)
        finally:
            reference_vrt.close()

        original_pixels_per_reduced_x = (
            pred.width / out_width
        )
        original_pixels_per_reduced_y = (
            pred.height / out_height
        )

    return (
        predicted,
        reference,
        original_pixels_per_reduced_x,
        original_pixels_per_reduced_y,
    )


def shift_diagnostic(context: dict) -> tuple[pd.DataFrame, dict]:
    (
        predicted,
        reference,
        factor_x,
        factor_y,
    ) = reduced_aligned_arrays(context)

    codes = context["codes"]
    code_set = set(int(code) for code in codes)

    rows = []
    height, width = predicted.shape

    for dy in range(
        -SHIFT_MAX_PIXELS,
        SHIFT_MAX_PIXELS + 1,
    ):
        for dx in range(
            -SHIFT_MAX_PIXELS,
            SHIFT_MAX_PIXELS + 1,
        ):
            (
                ref_y,
                ref_x,
                pred_y,
                pred_x,
            ) = shifted_slices(
                height,
                width,
                dy,
                dx,
            )

            true_values = reference[ref_y, ref_x]
            pred_values = predicted[pred_y, pred_x]

            valid = np.isin(
                true_values,
                list(code_set),
            ) & np.isin(
                pred_values,
                list(code_set),
            )

            total = int(valid.sum())
            if total == 0:
                accuracy = np.nan
                kappa = np.nan
            else:
                true_valid = true_values[valid]
                pred_valid = pred_values[valid]
                accuracy = float(
                    np.mean(true_valid == pred_valid)
                )

                true_counts = np.asarray(
                    [
                        np.count_nonzero(true_valid == code)
                        for code in codes
                    ],
                    dtype=float,
                )
                pred_counts = np.asarray(
                    [
                        np.count_nonzero(pred_valid == code)
                        for code in codes
                    ],
                    dtype=float,
                )
                expected = float(
                    np.sum(
                        (true_counts / total)
                        * (pred_counts / total)
                    )
                )
                kappa = (
                    (accuracy - expected)
                    / (1 - expected)
                    if expected < 1
                    else np.nan
                )

            rows.append(
                {
                    "dx_reduced_pixels": dx,
                    "dy_reduced_pixels": dy,
                    "dx_original_pixels": dx * factor_x,
                    "dy_original_pixels": dy * factor_y,
                    "evaluated_pixels": total,
                    "accuracy": accuracy,
                    "kappa": kappa,
                }
            )

    dataframe = pd.DataFrame(rows)

    base_row = dataframe.loc[
        (dataframe["dx_reduced_pixels"] == 0)
        & (dataframe["dy_reduced_pixels"] == 0)
    ].iloc[0]

    valid_rows = dataframe.dropna(subset=["accuracy"])
    best_row = valid_rows.loc[
        valid_rows["accuracy"].idxmax()
    ]

    summary = {
        "base_accuracy_reduced": float(
            base_row["accuracy"]
        ),
        "best_shift_accuracy": float(
            best_row["accuracy"]
        ),
        "shift_accuracy_gain": float(
            best_row["accuracy"]
            - base_row["accuracy"]
        ),
        "best_dx_reduced_pixels": int(
            best_row["dx_reduced_pixels"]
        ),
        "best_dy_reduced_pixels": int(
            best_row["dy_reduced_pixels"]
        ),
        "best_dx_original_pixels": float(
            best_row["dx_original_pixels"]
        ),
        "best_dy_original_pixels": float(
            best_row["dy_original_pixels"]
        ),
        "downsampling_factor_x": float(factor_x),
        "downsampling_factor_y": float(factor_y),
    }

    return dataframe, summary


# =============================================================================
# 10. DIAGNÓSTICO RADIOMÉTRICO Y NORMALIZACIÓN
# =============================================================================

def radiometric_diagnostic(
    context: dict,
) -> tuple[pd.DataFrame, dict]:
    input_raster = context["input_raster"]
    config = context["config"]

    if input_raster is None or not input_raster.exists():
        return (
            pd.DataFrame(
                [
                    {
                        "estado": "NO EVALUADO",
                        "detalle": (
                            "No se encontró INPUT_RASTER ni pudo "
                            "recuperarse desde PREDICTION_COMPLETED.json."
                        ),
                    }
                ]
            ),
            {
                "available": False,
                "reason": "input raster no localizado",
            },
        )

    band_indexes = [
        int(value)
        for value in config.get(
            "band_indexes",
            [1, 2, 3, 4],
        )
    ]
    band_order = config.get(
        "band_order",
        ["B2", "B3", "B4", "B8"],
    )
    train_means = np.asarray(
        config.get("means", [np.nan] * len(band_indexes)),
        dtype=float,
    )
    train_stds = np.asarray(
        config.get("stds", [np.nan] * len(band_indexes)),
        dtype=float,
    )

    use_raster_scale_offset = bool(
        config.get("use_raster_scale_offset", True)
    )
    auto_scale = bool(
        config.get("auto_scale_reflectance", True)
    )
    reflectance_scale = safe_float(
        config.get("reflectance_scale", 10000.0),
        10000.0,
    )
    reflectance_offset = safe_float(
        config.get("reflectance_add_offset", 0.0),
        0.0,
    )

    with rasterio.open(input_raster) as src:
        if max(band_indexes) > src.count:
            raise ValueError(
                f"La imagen tiene {src.count} bandas, pero el config "
                f"solicita {band_indexes}."
            )

        scale = min(
            1.0,
            RADIOMETRY_MAX_DIMENSION
            / max(src.width, src.height),
        )
        out_width = max(1, int(round(src.width * scale)))
        out_height = max(1, int(round(src.height * scale)))

        raw = src.read(
            indexes=band_indexes,
            out_shape=(
                len(band_indexes),
                out_height,
                out_width,
            ),
            resampling=Resampling.nearest,
        ).astype(np.float32)

        nodata = src.nodata
        valid = np.isfinite(raw)
        if nodata is not None and np.isfinite(nodata):
            valid &= raw != nodata

        raw_nan = np.where(valid, raw, np.nan)

        scales = np.asarray(
            [
                src.scales[index - 1]
                for index in band_indexes
            ],
            dtype=np.float32,
        )[:, None, None]
        offsets = np.asarray(
            [
                src.offsets[index - 1]
                for index in band_indexes
            ],
            dtype=np.float32,
        )[:, None, None]

        descriptions = [
            src.descriptions[index - 1]
            for index in band_indexes
        ]

    finite_values = raw_nan[np.isfinite(raw_nan)]
    raw_p995_global = (
        float(np.nanpercentile(finite_values, 99.5))
        if finite_values.size > 0
        else np.nan
    )

    raster_transform_used = (
        use_raster_scale_offset
        and (
            not np.allclose(scales, 1.0)
            or not np.allclose(offsets, 0.0)
        )
    )

    if raster_transform_used:
        transformed = raw_nan * scales + offsets
        scale_method = "scale/offset del GeoTIFF"
    elif auto_scale and raw_p995_global > 2:
        transformed = (
            raw_nan + reflectance_offset
        ) / reflectance_scale
        scale_method = (
            f"división automática entre {reflectance_scale}"
        )
    else:
        transformed = raw_nan
        scale_method = "sin escalamiento adicional"

    transformed = np.clip(
        transformed,
        0.0,
        1.0,
    )

    rows = []
    max_mean_shift = 0.0
    maximum_std_ratio_deviation = 0.0

    for position, (
        band_index,
        band_name,
    ) in enumerate(zip(band_indexes, band_order)):
        raw_band = raw_nan[position]
        transformed_band = transformed[position]

        finite_raw = raw_band[np.isfinite(raw_band)]
        finite_transformed = transformed_band[
            np.isfinite(transformed_band)
        ]

        scene_mean = (
            float(np.mean(finite_transformed))
            if finite_transformed.size
            else np.nan
        )
        scene_std = (
            float(np.std(finite_transformed))
            if finite_transformed.size
            else np.nan
        )

        train_mean = (
            float(train_means[position])
            if position < len(train_means)
            else np.nan
        )
        train_std = (
            float(train_stds[position])
            if position < len(train_stds)
            else np.nan
        )

        mean_shift_sd = (
            abs(scene_mean - train_mean) / train_std
            if (
                np.isfinite(scene_mean)
                and np.isfinite(train_mean)
                and np.isfinite(train_std)
                and train_std > 0
            )
            else np.nan
        )
        std_ratio = (
            scene_std / train_std
            if (
                np.isfinite(scene_std)
                and np.isfinite(train_std)
                and train_std > 0
            )
            else np.nan
        )

        if np.isfinite(mean_shift_sd):
            max_mean_shift = max(
                max_mean_shift,
                mean_shift_sd,
            )
        if np.isfinite(std_ratio):
            maximum_std_ratio_deviation = max(
                maximum_std_ratio_deviation,
                abs(std_ratio - 1),
            )

        rows.append(
            {
                "band_position": position + 1,
                "raster_band_index": band_index,
                "expected_band": band_name,
                "raster_description": descriptions[position],
                "raw_min": (
                    float(np.min(finite_raw))
                    if finite_raw.size
                    else np.nan
                ),
                "raw_p01": (
                    float(np.percentile(finite_raw, 1))
                    if finite_raw.size
                    else np.nan
                ),
                "raw_p50": (
                    float(np.percentile(finite_raw, 50))
                    if finite_raw.size
                    else np.nan
                ),
                "raw_p995": (
                    float(np.percentile(finite_raw, 99.5))
                    if finite_raw.size
                    else np.nan
                ),
                "raw_max": (
                    float(np.max(finite_raw))
                    if finite_raw.size
                    else np.nan
                ),
                "scene_reflectance_mean": scene_mean,
                "scene_reflectance_std": scene_std,
                "scene_reflectance_p01": (
                    float(
                        np.percentile(
                            finite_transformed,
                            1,
                        )
                    )
                    if finite_transformed.size
                    else np.nan
                ),
                "scene_reflectance_p50": (
                    float(
                        np.percentile(
                            finite_transformed,
                            50,
                        )
                    )
                    if finite_transformed.size
                    else np.nan
                ),
                "scene_reflectance_p995": (
                    float(
                        np.percentile(
                            finite_transformed,
                            99.5,
                        )
                    )
                    if finite_transformed.size
                    else np.nan
                ),
                "training_mean": train_mean,
                "training_std": train_std,
                "mean_shift_in_training_std": mean_shift_sd,
                "scene_to_training_std_ratio": std_ratio,
                "valid_sample_pixels": int(
                    finite_transformed.size
                ),
            }
        )

    summary = {
        "available": True,
        "input_raster": str(input_raster),
        "scale_method": scale_method,
        "raw_global_p995": raw_p995_global,
        "max_mean_shift_in_training_std": float(
            max_mean_shift
        ),
        "max_std_ratio_deviation": float(
            maximum_std_ratio_deviation
        ),
        "band_indexes": band_indexes,
        "band_order": band_order,
        "raster_descriptions": descriptions,
    }

    return pd.DataFrame(rows), summary


# =============================================================================
# 11. VALIDACIÓN DE SPLITS Y POSIBLE FUGA DE INFORMACIÓN
# =============================================================================

def find_split_csv(
    splits_dir: Path,
    split_name: str,
) -> Path | None:
    candidates = {
        "train": [
            "train_samples.csv",
            "train.csv",
        ],
        "val": [
            "val_samples.csv",
            "validation_samples.csv",
            "val.csv",
            "validation.csv",
        ],
        "test": [
            "test_samples.csv",
            "test.csv",
        ],
    }
    for name in candidates[split_name]:
        path = splits_dir / name
        if path.exists():
            return path
    return None


def detect_path_column(
    dataframe: pd.DataFrame,
    kind: str,
) -> str | None:
    options = {
        "image": [
            "image_path",
            "image",
            "img_path",
            "raster_path",
            "chip_path",
        ],
        "mask": [
            "mask_path",
            "mask",
            "label_path",
            "target_path",
        ],
    }[kind]

    lower_map = {
        str(column).lower(): column
        for column in dataframe.columns
    }
    for option in options:
        if option in lower_map:
            return lower_map[option]
    return None


def normalize_path_text(value) -> str:
    try:
        return str(Path(str(value))).replace(
            "\\",
            "/",
        ).lower()
    except Exception:
        return str(value).lower()


def chip_spatial_keys(
    dataframe: pd.DataFrame,
    image_column: str | None,
) -> set:
    if image_column is None:
        return set()

    paths = dataframe[image_column].dropna().tolist()
    if len(paths) > MAX_CHIPS_FOR_SPATIAL_SPLIT_CHECK:
        rng = np.random.default_rng(42)
        paths = rng.choice(
            paths,
            size=MAX_CHIPS_FOR_SPATIAL_SPLIT_CHECK,
            replace=False,
        ).tolist()

    keys = set()
    for value in paths:
        path = Path(str(value))
        if not path.exists():
            continue
        try:
            with rasterio.open(path) as src:
                key = (
                    str(src.crs),
                    round(src.bounds.left, 6),
                    round(src.bounds.bottom, 6),
                    round(src.bounds.right, 6),
                    round(src.bounds.top, 6),
                )
                keys.add(key)
        except Exception:
            continue
    return keys


def split_diagnostic(context: dict) -> dict:
    config = context["config"]
    splits_dir_value = config.get("splits_dir")
    if not splits_dir_value:
        return {
            "summary": pd.DataFrame(
                [
                    {
                        "estado": "NO EVALUADO",
                        "detalle": "config.json no contiene splits_dir.",
                    }
                ]
            ),
            "duplicates": pd.DataFrame(),
            "spatial_overlap": pd.DataFrame(),
            "metrics": {
                "available": False,
            },
        }

    splits_dir = Path(splits_dir_value)
    if not splits_dir.exists():
        return {
            "summary": pd.DataFrame(
                [
                    {
                        "estado": "NO EVALUADO",
                        "detalle": (
                            f"No existe la carpeta de splits: "
                            f"{splits_dir}"
                        ),
                    }
                ]
            ),
            "duplicates": pd.DataFrame(),
            "spatial_overlap": pd.DataFrame(),
            "metrics": {
                "available": False,
            },
        }

    split_frames = {}
    summary_rows = []

    for split_name in ("train", "val", "test"):
        csv_path = find_split_csv(
            splits_dir,
            split_name,
        )
        if csv_path is None:
            summary_rows.append(
                {
                    "split": split_name,
                    "csv": "",
                    "rows": 0,
                    "image_column": "",
                    "mask_column": "",
                    "status": "NO ENCONTRADO",
                }
            )
            continue

        frame = pd.read_csv(csv_path)
        image_column = detect_path_column(
            frame,
            "image",
        )
        mask_column = detect_path_column(
            frame,
            "mask",
        )
        split_frames[split_name] = {
            "frame": frame,
            "csv": csv_path,
            "image_column": image_column,
            "mask_column": mask_column,
        }
        summary_rows.append(
            {
                "split": split_name,
                "csv": str(csv_path),
                "rows": len(frame),
                "image_column": image_column,
                "mask_column": mask_column,
                "status": "LEÍDO",
            }
        )

    duplicate_rows = []
    spatial_rows = []

    split_names = list(split_frames)
    for position, split_a in enumerate(split_names):
        for split_b in split_names[position + 1:]:
            data_a = split_frames[split_a]
            data_b = split_frames[split_b]

            for kind in ("image", "mask"):
                column_a = data_a[f"{kind}_column"]
                column_b = data_b[f"{kind}_column"]

                if column_a is None or column_b is None:
                    continue

                paths_a = set(
                    data_a["frame"][column_a]
                    .dropna()
                    .map(normalize_path_text)
                )
                paths_b = set(
                    data_b["frame"][column_b]
                    .dropna()
                    .map(normalize_path_text)
                )
                exact = sorted(paths_a & paths_b)

                basenames_a = {
                    Path(path).name: path
                    for path in paths_a
                }
                basenames_b = {
                    Path(path).name: path
                    for path in paths_b
                }
                same_names = sorted(
                    set(basenames_a)
                    & set(basenames_b)
                )

                duplicate_rows.append(
                    {
                        "split_a": split_a,
                        "split_b": split_b,
                        "type": kind,
                        "exact_path_duplicates": len(exact),
                        "same_filename_count": len(same_names),
                        "exact_examples": " | ".join(
                            exact[:10]
                        ),
                        "filename_examples": " | ".join(
                            same_names[:10]
                        ),
                    }
                )

            spatial_keys_a = chip_spatial_keys(
                data_a["frame"],
                data_a["image_column"],
            )
            spatial_keys_b = chip_spatial_keys(
                data_b["frame"],
                data_b["image_column"],
            )
            spatial_overlap = (
                spatial_keys_a & spatial_keys_b
            )

            spatial_rows.append(
                {
                    "split_a": split_a,
                    "split_b": split_b,
                    "sampled_georeferenced_chips_a": len(
                        spatial_keys_a
                    ),
                    "sampled_georeferenced_chips_b": len(
                        spatial_keys_b
                    ),
                    "exact_same_extent_count": len(
                        spatial_overlap
                    ),
                    "examples": " | ".join(
                        map(str, list(spatial_overlap)[:5])
                    ),
                }
            )

    duplicates_df = pd.DataFrame(duplicate_rows)
    spatial_df = pd.DataFrame(spatial_rows)

    exact_duplicates = (
        int(
            duplicates_df[
                "exact_path_duplicates"
            ].sum()
        )
        if not duplicates_df.empty
        else 0
    )
    same_extent_count = (
        int(
            spatial_df[
                "exact_same_extent_count"
            ].sum()
        )
        if not spatial_df.empty
        else 0
    )

    return {
        "summary": pd.DataFrame(summary_rows),
        "duplicates": duplicates_df,
        "spatial_overlap": spatial_df,
        "metrics": {
            "available": True,
            "exact_path_duplicates": exact_duplicates,
            "exact_same_extent_count": same_extent_count,
            "note": (
                "La ausencia de duplicados exactos no demuestra "
                "independencia espacial completa. Para ello se requieren "
                "bloques geográficos separados."
            ),
        },
    }


# =============================================================================
# 12. COMPARACIÓN ENTRE ENTRENAMIENTO Y PREDICCIÓN EXTERNA
# =============================================================================

def get_metric(data: dict, names, default=np.nan):
    for name in names:
        if name in data:
            return safe_float(data[name])
    return float(default)


def training_external_comparison(
    training_info: dict,
    external_metrics: dict,
) -> tuple[pd.DataFrame, dict]:
    test_report = training_info["test_report"]
    best_history = training_info["best_history"]

    rows = []

    metric_specs = [
        (
            "accuracy",
            ["accuracy"],
            ["accuracy"],
        ),
        (
            "macro_f1",
            ["macro_f1", "f1_macro"],
            ["macro_f1", "f1_macro"],
        ),
        (
            "mean_iou",
            ["mean_iou", "miou"],
            ["mean_iou", "miou"],
        ),
        (
            "kappa",
            ["kappa"],
            ["kappa"],
        ),
    ]

    drops = {}

    for metric_name, internal_names, external_names in metric_specs:
        internal_value = get_metric(
            test_report,
            internal_names,
        )
        external_value = get_metric(
            external_metrics,
            external_names,
        )
        difference = (
            external_value - internal_value
            if (
                np.isfinite(internal_value)
                and np.isfinite(external_value)
            )
            else np.nan
        )

        rows.append(
            {
                "metric": metric_name,
                "internal_test": internal_value,
                "external_prediction": external_value,
                "external_minus_internal": difference,
            }
        )
        drops[metric_name] = difference

    best_val_iou = get_metric(
        best_history,
        ["val_mean_iou"],
    )
    best_val_f1 = get_metric(
        best_history,
        ["val_macro_f1"],
    )

    summary = {
        "best_validation_mean_iou": best_val_iou,
        "best_validation_macro_f1": best_val_f1,
        "test_accuracy": get_metric(
            test_report,
            ["accuracy"],
        ),
        "test_macro_f1": get_metric(
            test_report,
            ["macro_f1", "f1_macro"],
        ),
        "test_mean_iou": get_metric(
            test_report,
            ["mean_iou", "miou"],
        ),
        "test_kappa": get_metric(
            test_report,
            ["kappa"],
        ),
        "external_accuracy": get_metric(
            external_metrics,
            ["accuracy"],
        ),
        "external_macro_f1": get_metric(
            external_metrics,
            ["macro_f1"],
        ),
        "external_mean_iou": get_metric(
            external_metrics,
            ["mean_iou"],
        ),
        "external_kappa": get_metric(
            external_metrics,
            ["kappa"],
        ),
        "accuracy_drop": drops.get(
            "accuracy",
            np.nan,
        ),
        "macro_f1_drop": drops.get(
            "macro_f1",
            np.nan,
        ),
        "mean_iou_drop": drops.get(
            "mean_iou",
            np.nan,
        ),
    }

    return pd.DataFrame(rows), summary


# =============================================================================
# 13. GENERACIÓN AUTOMÁTICA DE HALLAZGOS
# =============================================================================

def make_finding(
    severity,
    component,
    evidence,
    interpretation,
    solution,
):
    return {
        "severidad": severity,
        "componente": component,
        "evidencia": evidence,
        "interpretacion": interpretation,
        "solucion_recomendada": solution,
    }


def build_findings(
    context: dict,
    alignment_summary: dict,
    evaluation: dict,
    mapping_summary: dict,
    shift_summary: dict,
    radiometry_summary: dict,
    split_metrics: dict,
    comparison_summary: dict,
) -> pd.DataFrame:
    findings = []
    metrics = evaluation["global_metrics"]
    per_class = evaluation["per_class"]

    kappa = safe_float(metrics.get("kappa"))
    accuracy = safe_float(metrics.get("accuracy"))

    if np.isfinite(kappa) and kappa < 0:
        findings.append(
            make_finding(
                "ALTO",
                "Kappa negativo",
                (
                    f"Kappa={kappa:.4f}; "
                    f"Po={metrics.get('observed_agreement_po', np.nan):.4f}; "
                    f"Pe={metrics.get('expected_agreement_pe', np.nan):.4f}"
                ),
                (
                    "La coincidencia observada es menor que la esperada "
                    "según las proporciones marginales. No es únicamente "
                    "un problema de la fórmula de Kappa."
                ),
                (
                    "Revisar primero el mapeo de códigos, la alineación, "
                    "la normalización y la distribución de clases."
                ),
            )
        )

    unknown_prediction = int(
        metrics.get(
            "unknown_prediction_pixels",
            0,
        )
    )
    unknown_reference = int(
        metrics.get(
            "unknown_reference_pixels",
            0,
        )
    )
    if unknown_prediction > 0 or unknown_reference > 0:
        findings.append(
            make_finding(
                "CRÍTICO",
                "Códigos temáticos desconocidos",
                (
                    f"Predicción desconocida={unknown_prediction}; "
                    f"referencia desconocida={unknown_reference}"
                ),
                (
                    "Existen valores que no aparecen en el mapeo de "
                    "clases del modelo."
                ),
                (
                    "Corregir idx_to_code/code_to_idx o reclasificar el "
                    "ráster antes de interpretar las métricas."
                ),
            )
        )

    relabel_gain = safe_float(
        mapping_summary.get("relabel_gain")
    )
    if (
        np.isfinite(relabel_gain)
        and relabel_gain >= MAPPING_GAIN_ALERT
    ):
        findings.append(
            make_finding(
                "CRÍTICO",
                "Posible permutación de códigos",
                (
                    f"La exactitud subiría "
                    f"{relabel_gain * 100:.2f} puntos porcentuales "
                    "con un remapeo uno a uno."
                ),
                (
                    "Las formas espaciales pueden ser razonables, pero "
                    "las salidas estar etiquetadas con códigos distintos."
                ),
                (
                    "Comparar la tabla mapeo_optimo con idx_to_code y "
                    "regenerar el GeoTIFF usando la correspondencia correcta."
                ),
            )
        )

    shift_gain = safe_float(
        shift_summary.get("shift_accuracy_gain")
    )
    if (
        np.isfinite(shift_gain)
        and shift_gain >= SHIFT_GAIN_ALERT
    ):
        findings.append(
            make_finding(
                "ALTO",
                "Posible desplazamiento espacial",
                (
                    f"La exactitud mejora "
                    f"{shift_gain * 100:.2f} puntos con desplazamiento "
                    f"dx={shift_summary.get('best_dx_original_pixels', np.nan):.2f}, "
                    f"dy={shift_summary.get('best_dy_original_pixels', np.nan):.2f} "
                    "píxeles originales."
                ),
                (
                    "La predicción y la referencia podrían no estar "
                    "comparándose sobre la misma cuadrícula."
                ),
                (
                    "Alinear la referencia exactamente con el CRS, "
                    "transformación, origen y resolución de la predicción "
                    "mediante vecino más cercano."
                ),
            )
        )

    if not alignment_summary.get("same_crs", False):
        findings.append(
            make_finding(
                "ALTO",
                "CRS diferente",
                "El CRS de la predicción y la referencia no es idéntico.",
                (
                    "El código reproyecta la referencia, pero la diferencia "
                    "incrementa el riesgo de desplazamientos o recortes."
                ),
                "Crear una referencia alineada previamente con la predicción.",
            )
        )

    if not alignment_summary.get("same_transform", False):
        findings.append(
            make_finding(
                "MEDIO",
                "Transformación afín diferente",
                (
                    f"Offset columna="
                    f"{alignment_summary.get('pixel_offset_col')}; "
                    f"offset fila="
                    f"{alignment_summary.get('pixel_offset_row')}"
                ),
                (
                    "Aunque ambos rásteres tengan el mismo CRS y resolución, "
                    "el origen de la malla puede ser diferente."
                ),
                (
                    "Usar la predicción como plantilla exacta y remuestrear "
                    "la referencia con vecino más cercano."
                ),
            )
        )

    max_radiometric_shift = safe_float(
        radiometry_summary.get(
            "max_mean_shift_in_training_std"
        )
    )
    if (
        radiometry_summary.get("available")
        and np.isfinite(max_radiometric_shift)
        and max_radiometric_shift
        >= RADIOMETRIC_MEAN_SHIFT_ALERT
    ):
        findings.append(
            make_finding(
                "ALTO",
                "Cambio radiométrico o de dominio",
                (
                    f"Máximo desplazamiento de media="
                    f"{max_radiometric_shift:.2f} desviaciones "
                    "del entrenamiento."
                ),
                (
                    "La imagen completa tiene una distribución espectral "
                    "diferente a la de los chips usados para entrenar."
                ),
                (
                    "Confirmar escala de reflectancia, orden B2-B3-B4-B8, "
                    "fecha/estación y considerar ajuste fino con muestras "
                    "representativas de 2025."
                ),
            )
        )

    if not per_class.empty:
        dominant_row = per_class.loc[
            per_class["predicted_percentage"].idxmax()
        ]
        dominant_percentage = float(
            dominant_row["predicted_percentage"]
        )
        if dominant_percentage / 100 >= DOMINANT_CLASS_ALERT:
            findings.append(
                make_finding(
                    "ALTO",
                    "Colapso hacia una clase",
                    (
                        f"La clase {int(dominant_row['class_code'])} "
                        f"concentra {dominant_percentage:.2f} % "
                        "de la predicción."
                    ),
                    (
                        "El modelo está asignando gran parte del territorio "
                        "a una sola categoría."
                    ),
                    (
                        "Revisar la distribución de entrenamiento, pesos "
                        "por clase, normalización y confusiones de esa clase."
                    ),
                )
            )

    missing_reference = metrics.get(
        "classes_without_reference_pixels",
        [],
    )
    if missing_reference:
        findings.append(
            make_finding(
                "MEDIO",
                "Clases ausentes en la referencia",
                str(missing_reference),
                (
                    "Las métricas de esas clases no son interpretables y "
                    "sus predicciones se contabilizan como errores si la "
                    "referencia no las contiene."
                ),
                (
                    "Verificar si realmente no existen en el área/año o "
                    "si hubo un problema de reclasificación."
                ),
            )
        )

    exact_duplicates = int(
        split_metrics.get(
            "exact_path_duplicates",
            0,
        )
    )
    same_extents = int(
        split_metrics.get(
            "exact_same_extent_count",
            0,
        )
    )
    if exact_duplicates > 0 or same_extents > 0:
        findings.append(
            make_finding(
                "CRÍTICO",
                "Fuga entre splits",
                (
                    f"Rutas duplicadas={exact_duplicates}; "
                    f"extensiones geográficas idénticas={same_extents}"
                ),
                (
                    "Las métricas internas pueden estar infladas porque "
                    "las mismas muestras aparecen en más de un conjunto."
                ),
                (
                    "Reconstruir train/validation/test con separación "
                    "geográfica estricta."
                ),
            )
        )

    test_accuracy = safe_float(
        comparison_summary.get("test_accuracy")
    )
    external_accuracy = safe_float(
        comparison_summary.get("external_accuracy")
    )
    if (
        np.isfinite(test_accuracy)
        and np.isfinite(external_accuracy)
        and test_accuracy - external_accuracy
        >= EXTERNAL_METRIC_DROP_ALERT
    ):
        findings.append(
            make_finding(
                "ALTO",
                "Brecha entre prueba interna y escena completa",
                (
                    f"Accuracy interna={test_accuracy:.4f}; "
                    f"externa={external_accuracy:.4f}; "
                    f"caída={(test_accuracy - external_accuracy):.4f}"
                ),
                (
                    "El modelo funciona sobre chips similares al "
                    "entrenamiento, pero no generaliza a la escena completa."
                ),
                (
                    "Revisar independencia espacial, cambio temporal, "
                    "distribución radiométrica y representatividad del dataset."
                ),
            )
        )

    mean_wrong_confidence = safe_float(
        metrics.get("mean_confidence_incorrect")
    )
    if (
        np.isfinite(mean_wrong_confidence)
        and mean_wrong_confidence
        >= HIGH_CONFIDENCE_THRESHOLD
    ):
        findings.append(
            make_finding(
                "ALTO",
                "Errores con alta confianza",
                (
                    f"Confianza media de errores="
                    f"{mean_wrong_confidence:.4f}"
                ),
                (
                    "El modelo no solo se equivoca, sino que está "
                    "sobreconfiado en sus asignaciones."
                ),
                (
                    "Aplicar calibración, revisar cambio de dominio y "
                    "considerar ajuste fino con datos del año objetivo."
                ),
            )
        )

    image_year = IMAGE_YEAR
    reference_year = REFERENCE_YEAR_BY_LEVEL.get(
        context["level"]
    )
    if (
        image_year is not None
        and reference_year is not None
        and image_year != reference_year
    ):
        findings.append(
            make_finding(
                "MEDIO",
                "Diferencia temporal",
                (
                    f"Imagen={image_year}; referencia={reference_year}"
                ),
                (
                    "Las métricas mezclan errores del modelo con cambios "
                    "reales de cobertura ocurridos entre las fechas."
                ),
                (
                    "Reportar el resultado como concordancia espacial con "
                    "referencia histórica y construir una muestra "
                    "independiente del mismo año para exactitud 2025."
                ),
            )
        )

    if context["validity_mask"] is None:
        findings.append(
            make_finding(
                "INFORMATIVO",
                "Máscara de nubes/sombras",
                "No se proporcionó una máscara de validez.",
                (
                    "Nubes, sombras o píxeles atmosféricamente inválidos "
                    "pueden entrar en la evaluación."
                ),
                (
                    "Incorporar una máscara de nubes y sombras cuando "
                    "esté disponible."
                ),
            )
        )

    if not findings:
        findings.append(
            make_finding(
                "CORRECTO",
                "Diagnóstico",
                "No se detectaron alertas con los umbrales configurados.",
                (
                    "No se identifica una causa automática dominante."
                ),
                (
                    "Revisar visualmente la matriz de confusión y validar "
                    "con muestras contemporáneas independientes."
                ),
            )
        )

    dataframe = pd.DataFrame(findings)
    dataframe["_rank"] = dataframe[
        "severidad"
    ].map(severity_rank)
    dataframe = (
        dataframe.sort_values(
            ["_rank", "componente"]
        )
        .drop(columns="_rank")
        .reset_index(drop=True)
    )
    return dataframe


# =============================================================================
# 14. GRÁFICAS
# =============================================================================

def save_distribution_plot(
    per_class: pd.DataFrame,
    output_path: Path,
    title: str,
):
    if per_class.empty:
        return

    x = np.arange(len(per_class))
    width = 0.38

    plt.figure(figsize=(12, 6))
    plt.bar(
        x - width / 2,
        per_class["reference_percentage"],
        width,
        label="Referencia",
    )
    plt.bar(
        x + width / 2,
        per_class["predicted_percentage"],
        width,
        label="Predicción",
    )
    plt.xticks(
        x,
        per_class["class_code"].astype(str),
        rotation=0,
    )
    plt.xlabel("Código de clase")
    plt.ylabel("Porcentaje de píxeles")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close()


def save_confusion_plot(
    normalized_confusion: pd.DataFrame,
    output_path: Path,
    title: str,
):
    if normalized_confusion.empty:
        return

    matrix = normalized_confusion.to_numpy()

    plt.figure(figsize=(10, 8))
    image = plt.imshow(
        matrix,
        aspect="auto",
        vmin=0,
        vmax=1,
    )
    plt.colorbar(
        image,
        label="Proporción por clase de referencia",
    )
    plt.xticks(
        np.arange(len(normalized_confusion.columns)),
        [
            str(value).split(" - ")[0]
            for value in normalized_confusion.columns
        ],
        rotation=90,
    )
    plt.yticks(
        np.arange(len(normalized_confusion.index)),
        [
            str(value).split(" - ")[0]
            for value in normalized_confusion.index
        ],
    )
    plt.xlabel("Clase predicha")
    plt.ylabel("Clase de referencia")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close()


def save_shift_plot(
    shift_df: pd.DataFrame,
    output_path: Path,
    title: str,
):
    if shift_df.empty:
        return

    pivot = shift_df.pivot(
        index="dy_reduced_pixels",
        columns="dx_reduced_pixels",
        values="accuracy",
    )

    plt.figure(figsize=(8, 7))
    image = plt.imshow(
        pivot.to_numpy(),
        aspect="auto",
        origin="lower",
    )
    plt.colorbar(
        image,
        label="Exactitud",
    )
    plt.xticks(
        np.arange(len(pivot.columns)),
        pivot.columns,
    )
    plt.yticks(
        np.arange(len(pivot.index)),
        pivot.index,
    )
    plt.xlabel("Desplazamiento X")
    plt.ylabel("Desplazamiento Y")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close()


def save_training_comparison_plot(
    comparison_df: pd.DataFrame,
    output_path: Path,
    title: str,
):
    if comparison_df.empty:
        return

    valid = comparison_df.dropna(
        subset=[
            "internal_test",
            "external_prediction",
        ],
        how="all",
    )
    if valid.empty:
        return

    x = np.arange(len(valid))
    width = 0.38

    plt.figure(figsize=(10, 6))
    plt.bar(
        x - width / 2,
        valid["internal_test"],
        width,
        label="Prueba interna",
    )
    plt.bar(
        x + width / 2,
        valid["external_prediction"],
        width,
        label="Predicción externa",
    )
    plt.xticks(
        x,
        valid["metric"],
    )
    plt.ylim(
        min(
            -0.5,
            float(
                np.nanmin(
                    valid[
                        [
                            "internal_test",
                            "external_prediction",
                        ]
                    ].to_numpy()
                )
            )
            - 0.05,
        ),
        1.05,
    )
    plt.ylabel("Valor")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        output_path,
        dpi=180,
        bbox_inches="tight",
    )
    plt.close()


# =============================================================================
# 15. GUARDADO DEL INFORME
# =============================================================================

def write_text_report(
    level_name: str,
    context: dict,
    globals_dict: dict,
    mapping_summary: dict,
    shift_summary: dict,
    radiometry_summary: dict,
    findings: pd.DataFrame,
    output_path: Path,
):
    lines = [
        "=" * 80,
        f"DIAGNÓSTICO INTEGRAL RESNET-50 — {level_name}",
        "=" * 80,
        "",
        f"Predicción: {context['prediction_path']}",
        f"Referencia: {context['reference_path']}",
        f"Modelo: {context['run_dir']}",
        f"Imagen de entrada: {context['input_raster']}",
        "",
        "MÉTRICAS PRINCIPALES",
        "-" * 80,
        f"Exactitud: {safe_float(globals_dict.get('accuracy')):.6f}",
        f"Macro F1: {safe_float(globals_dict.get('macro_f1')):.6f}",
        f"IoU media: {safe_float(globals_dict.get('mean_iou')):.6f}",
        f"Kappa: {safe_float(globals_dict.get('kappa')):.6f}",
        (
            f"Acuerdo observado Po: "
            f"{safe_float(globals_dict.get('observed_agreement_po')):.6f}"
        ),
        (
            f"Acuerdo esperado Pe: "
            f"{safe_float(globals_dict.get('expected_agreement_pe')):.6f}"
        ),
        (
            f"Desacuerdo de cantidad: "
            f"{safe_float(globals_dict.get('quantity_disagreement')):.6f}"
        ),
        (
            f"Desacuerdo de asignación: "
            f"{safe_float(globals_dict.get('allocation_disagreement')):.6f}"
        ),
        "",
        "PRUEBAS ESPECÍFICAS",
        "-" * 80,
        (
            f"Ganancia por remapeo de códigos: "
            f"{safe_float(mapping_summary.get('relabel_gain')):.6f}"
        ),
        (
            f"Ganancia por desplazamiento: "
            f"{safe_float(shift_summary.get('shift_accuracy_gain')):.6f}"
        ),
        (
            f"Máximo cambio radiométrico: "
            f"{safe_float(radiometry_summary.get('max_mean_shift_in_training_std')):.6f} "
            "desviaciones del entrenamiento"
        ),
        "",
        "HALLAZGOS PRIORIZADOS",
        "-" * 80,
    ]

    for _, row in findings.iterrows():
        lines.extend(
            [
                f"[{row['severidad']}] {row['componente']}",
                f"Evidencia: {row['evidencia']}",
                f"Interpretación: {row['interpretacion']}",
                f"Solución: {row['solucion_recomendada']}",
                "",
            ]
        )

    output_path.write_text(
        "\n".join(lines),
        encoding="utf-8",
    )


def write_level_outputs(
    context: dict,
    config_checks: pd.DataFrame,
    metadata_df: pd.DataFrame,
    alignment_df: pd.DataFrame,
    evaluation: dict,
    mapping_summary: dict,
    mapping_df: pd.DataFrame,
    shift_df: pd.DataFrame,
    radiometry_df: pd.DataFrame,
    training_info: dict,
    split_info: dict,
    comparison_df: pd.DataFrame,
    findings: pd.DataFrame,
) -> Path:
    level_name = context["level"]
    output_dir = ensure_dir(
        DIAGNOSTIC_OUTPUT / level_name
    )

    global_df = pd.DataFrame(
        [evaluation["global_metrics"]]
    )
    mapping_summary_df = pd.DataFrame(
        [mapping_summary]
    )

    tables = {
        "hallazgos": findings,
        "configuracion": config_checks,
        "metadatos_raster": metadata_df,
        "alineacion": alignment_df,
        "metricas_globales": global_df,
        "metricas_clase": evaluation["per_class"],
        "confusion": evaluation["confusion"],
        "confusion_normalizada": evaluation[
            "confusion_normalized"
        ],
        "mapeo_resumen": mapping_summary_df,
        "mapeo_optimo": mapping_df,
        "desplazamientos": shift_df,
        "radiometria": radiometry_df,
        "confianza": evaluation["confidence"],
        "comparacion_entrenamiento": comparison_df,
        "historial_entrenamiento": training_info["history"],
        "dataset_splits": training_info["dataset_splits"],
        "dataset_pixeles": training_info["dataset_pixels"],
        "split_validacion": split_info["summary"],
        "split_duplicados": split_info["duplicates"],
        "split_solape_espacial": split_info[
            "spatial_overlap"
        ],
    }

    for name, dataframe in tables.items():
        if isinstance(dataframe, pd.DataFrame):
            dataframe.to_csv(
                output_dir / f"{name}.csv",
                index=(
                    name
                    in {
                        "confusion",
                        "confusion_normalizada",
                    }
                ),
                encoding="utf-8-sig",
            )

    workbook_path = (
        output_dir
        / f"diagnostico_integral_{level_name}.xlsx"
    )
    try:
        with pd.ExcelWriter(
            workbook_path,
            engine="openpyxl",
        ) as writer:
            for name, dataframe in tables.items():
                if not isinstance(dataframe, pd.DataFrame):
                    continue
                sheet_name = name[:31]
                dataframe.to_excel(
                    writer,
                    sheet_name=sheet_name,
                    index=(
                        name
                        not in {
                            "confusion",
                            "confusion_normalizada",
                        }
                    ),
                )
    except Exception as error:
        warnings.warn(
            f"No se pudo crear el Excel de {level_name}: {error}. "
            "Los CSV sí fueron guardados.",
            RuntimeWarning,
        )

    save_distribution_plot(
        evaluation["per_class"],
        output_dir / "distribucion_clases.png",
        f"Distribución de clases — {level_name}",
    )
    save_confusion_plot(
        evaluation["confusion_normalized"],
        output_dir / "matriz_confusion_normalizada.png",
        f"Matriz de confusión normalizada — {level_name}",
    )
    save_shift_plot(
        shift_df,
        output_dir / "diagnostico_desplazamiento.png",
        f"Exactitud por desplazamiento — {level_name}",
    )
    save_training_comparison_plot(
        comparison_df,
        output_dir / "entrenamiento_vs_prediccion.png",
        f"Prueba interna frente a predicción externa — {level_name}",
    )

    return output_dir


# =============================================================================
# 16. DIAGNÓSTICO COMPLETO POR NIVEL
# =============================================================================

def diagnose_level(level_name: str) -> dict:
    print("\n" + "=" * 80)
    print(f"DIAGNÓSTICO {level_name.upper()}")
    print("=" * 80)

    context = load_level_context(level_name)

    if not context["reference_path"].exists():
        raise FileNotFoundError(
            f"No existe la referencia de {level_name}: "
            f"{context['reference_path']}"
        )

    print("1/9 Validando configuración del modelo...")
    config_checks = validate_model_configuration(
        context
    )

    print("2/9 Leyendo métricas del entrenamiento...")
    training_info = load_training_information(
        context
    )

    print("3/9 Revisando metadatos y alineación...")
    metadata_records = [
        raster_metadata(
            context["prediction_path"],
            "prediccion",
        ),
        raster_metadata(
            context["reference_path"],
            "referencia",
        ),
    ]
    if context["confidence_path"] is not None:
        metadata_records.append(
            raster_metadata(
                context["confidence_path"],
                "confianza",
            )
        )
    if context["input_raster"] is not None:
        metadata_records.append(
            raster_metadata(
                context["input_raster"],
                "imagen_entrada",
            )
        )
    metadata_df = pd.DataFrame(metadata_records)

    alignment_df, alignment_summary = (
        spatial_alignment_diagnostic(
            context["prediction_path"],
            context["reference_path"],
        )
    )

    print(
        "4/9 Calculando métricas censales y distribución "
        "de clases..."
    )
    evaluation = evaluate_prediction_censally(
        context
    )

    print("5/9 Probando correspondencia de códigos...")
    (
        mapping_summary,
        mapping_df,
    ) = permutation_diagnostic(
        evaluation["confusion_array"],
        context["codes"],
        context["names"],
    )

    print("6/9 Probando desplazamientos espaciales...")
    shift_df, shift_summary = shift_diagnostic(
        context
    )

    print("7/9 Comparando radiometría y normalización...")
    (
        radiometry_df,
        radiometry_summary,
    ) = radiometric_diagnostic(context)

    print("8/9 Revisando train, validation y test...")
    split_info = split_diagnostic(context)

    print("9/9 Comparando prueba interna y escena completa...")
    (
        comparison_df,
        comparison_summary,
    ) = training_external_comparison(
        training_info,
        evaluation["global_metrics"],
    )

    findings = build_findings(
        context=context,
        alignment_summary=alignment_summary,
        evaluation=evaluation,
        mapping_summary=mapping_summary,
        shift_summary=shift_summary,
        radiometry_summary=radiometry_summary,
        split_metrics=split_info["metrics"],
        comparison_summary=comparison_summary,
    )

    output_dir = write_level_outputs(
        context=context,
        config_checks=config_checks,
        metadata_df=metadata_df,
        alignment_df=alignment_df,
        evaluation=evaluation,
        mapping_summary=mapping_summary,
        mapping_df=mapping_df,
        shift_df=shift_df,
        radiometry_df=radiometry_df,
        training_info=training_info,
        split_info=split_info,
        comparison_df=comparison_df,
        findings=findings,
    )

    write_text_report(
        level_name=level_name,
        context=context,
        globals_dict=evaluation["global_metrics"],
        mapping_summary=mapping_summary,
        shift_summary=shift_summary,
        radiometry_summary=radiometry_summary,
        findings=findings,
        output_path=(
            output_dir / "REPORTE_DIAGNOSTICO.txt"
        ),
    )

    summary = {
        "level": level_name,
        "status": "completed",
        "output_dir": str(output_dir),
        **evaluation["global_metrics"],
        **{
            f"mapping_{key}": value
            for key, value in mapping_summary.items()
        },
        **{
            f"shift_{key}": value
            for key, value in shift_summary.items()
        },
        **{
            f"radiometry_{key}": value
            for key, value in radiometry_summary.items()
            if not isinstance(value, (list, dict))
        },
        "critical_findings": int(
            np.count_nonzero(
                findings["severidad"] == "CRÍTICO"
            )
        ),
        "high_findings": int(
            np.count_nonzero(
                findings["severidad"] == "ALTO"
            )
        ),
        "first_priority": (
            findings.iloc[0]["componente"]
            if not findings.empty
            else ""
        ),
    }

    print("\nResultados principales:")
    print(
        pd.DataFrame(
            [
                {
                    "nivel": level_name,
                    "accuracy": summary.get("accuracy"),
                    "macro_f1": summary.get("macro_f1"),
                    "mean_iou": summary.get("mean_iou"),
                    "kappa": summary.get("kappa"),
                    "ganancia_remapeo": summary.get(
                        "mapping_relabel_gain"
                    ),
                    "ganancia_desplazamiento": summary.get(
                        "shift_shift_accuracy_gain"
                    ),
                    "prioridad": summary.get(
                        "first_priority"
                    ),
                }
            ]
        )
    )
    print(f"Informe guardado en: {output_dir}")

    # Liberación explícita de objetos antes de pasar al siguiente nivel.
    del (
        config_checks,
        training_info,
        metadata_df,
        alignment_df,
        evaluation,
        mapping_df,
        shift_df,
        radiometry_df,
        split_info,
        comparison_df,
        findings,
    )
    gc.collect()

    return summary


# =============================================================================
# 17. EJECUCIÓN: NIVEL 1 Y DESPUÉS NIVEL 2
# =============================================================================

def main():
    ensure_dir(DIAGNOSTIC_OUTPUT)

    print("=" * 80)
    print("DIAGNÓSTICO INTEGRAL DE RESNET-50 — CPU ONLY")
    print("Primero nivel 1 y después nivel 2")
    print("No se cargará PyTorch, ROCm ni la GPU.")
    print("=" * 80)
    print(f"Salida: {DIAGNOSTIC_OUTPUT}")

    summaries = []

    for level_name in LEVELS:
        try:
            summary = diagnose_level(level_name)
            summaries.append(summary)
        except KeyboardInterrupt:
            print(
                f"\nProceso interrumpido durante {level_name}."
            )
            summaries.append(
                {
                    "level": level_name,
                    "status": "interrupted",
                }
            )
            break
        except Exception as error:
            print(
                f"\nNo fue posible completar {level_name}: "
                f"{error}"
            )
            traceback.print_exc()
            summaries.append(
                {
                    "level": level_name,
                    "status": "error",
                    "error": str(error),
                }
            )
            print(
                "El error quedó registrado. Se intentará continuar "
                "con el siguiente nivel."
            )
        finally:
            # CPU-only: se libera memoria de Python sin tocar GPU/ROCm.
            gc.collect()

    summary_df = pd.DataFrame(summaries)
    summary_csv = (
        DIAGNOSTIC_OUTPUT
        / "RESUMEN_DIAGNOSTICO_AMBOS_NIVELES.csv"
    )
    summary_xlsx = (
        DIAGNOSTIC_OUTPUT
        / "RESUMEN_DIAGNOSTICO_AMBOS_NIVELES.xlsx"
    )
    summary_df.to_csv(
        summary_csv,
        index=False,
        encoding="utf-8-sig",
    )
    try:
        summary_df.to_excel(
            summary_xlsx,
            index=False,
            engine="openpyxl",
        )
    except Exception as error:
        warnings.warn(
            f"No se pudo crear el resumen Excel: {error}",
            RuntimeWarning,
        )

    save_json(
        {
            "created_at": datetime.now().isoformat(
                timespec="seconds"
            ),
            "results": summaries,
        },
        DIAGNOSTIC_OUTPUT
        / "RESUMEN_DIAGNOSTICO_AMBOS_NIVELES.json",
    )

    print("\n" + "=" * 80)
    print("RESUMEN FINAL")
    print("=" * 80)
    print(summary_df)
    print(f"\nResumen guardado en: {summary_csv}")

    return summary_df


# Ejecución directa al copiar todo este TXT en una sola celda.
resultados_diagnostico = main()


DIAGNÓSTICO INTEGRAL DE RESNET-50 — CPU ONLY
Primero nivel 1 y después nivel 2
No se cargará PyTorch, ROCm ni la GPU.
Salida: C:\Users\Ivan sua\anaconda3\Predicciones_ResNet50_Segmentacion\DIAGNOSTICO_INTEGRAL_RESNET50

DIAGNÓSTICO NIVEL_1
1/9 Validando configuración del modelo...
2/9 Leyendo métricas del entrenamiento...
3/9 Revisando metadatos y alineación...
4/9 Calculando métricas censales y distribución de clases...
5/9 Probando correspondencia de códigos...
6/9 Probando desplazamientos espaciales...
7/9 Comparando radiometría y normalización...
8/9 Revisando train, validation y test...
9/9 Comparando prueba interna y escena completa...

Resultados principales:
     nivel  accuracy  macro_f1  mean_iou     kappa  ganancia_remapeo  \
0  nivel_1  0.096341  0.074859  0.040404 -0.416293          0.792811   

   ganancia_desplazamiento                       prioridad  
0                 0.089775  Códigos temáticos desconocidos  
Informe guardado en: C:\Users\Ivan sua\anaconda3\Prediccio

In [1]:
from __future__ import annotations

# =============================================================================
# CONFIGURACIÓN DE RECURSOS — Ryzen 5 5600X, 16 GB RAM y GPU AMD de 16 GB
# Debe definirse antes de importar NumPy, PyTorch, GDAL o Rasterio.
# =============================================================================
import os

os.environ.setdefault("OMP_NUM_THREADS", "3")
os.environ.setdefault("MKL_NUM_THREADS", "3")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "3")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "3")
os.environ.setdefault("GDAL_NUM_THREADS", "1")
os.environ.setdefault("GDAL_CACHEMAX", "128")

# -------------------------------------------------------------------------
# MODO ULTRAESTABLE: GPU/ROCm DESACTIVADOS
# -------------------------------------------------------------------------
# Se ocultan todos los aceleradores antes de importar PyTorch. De esta forma
# la predicción se ejecuta exclusivamente en CPU y un reinicio del controlador
# AMD no puede tumbar el kernel de Jupyter.
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["HIP_VISIBLE_DEVICES"] = "-1"
os.environ["ROCR_VISIBLE_DEVICES"] = "-1"

import gc
import json
import math
import os
import shutil
import time
import traceback
import warnings
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import torch
import torch.nn as nn
import torch.nn.functional as F
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window
from torchvision import models
from tqdm import tqdm

# =============================================================================
# 1. CONFIGURACIÓN
# =============================================================================
MODEL_NAME = "ResNet50-UNet"
# =============================================================================
# 1. RUTAS — MODIFICAR ÚNICAMENTE ESTA SECCIÓN
# =============================================================================
MODEL_ROOT = Path(r"C:\Users\Ivan sua\anaconda3\Modelos_ResNet50_Segmentacion")
INPUT_RASTER = Path(r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif")
OUTPUT_DIR = Path(r"C:\Users\Ivan sua\anaconda3\Predicciones_ResNet50_Segmentacion_CORREGIDA_CPU")

# Se procesan en este orden dentro de una sola celda:
# primero nivel 1 y después nivel 2.
LEVELS_TO_PREDICT = ["nivel_1", "nivel_2"]

# None selecciona automáticamente la ejecución final más reciente.
# También puede escribirse el nombre exacto de una carpeta de entrenamiento.
RUN_BY_LEVEL = {"nivel_1": None, "nivel_2": None}

REFERENCE_BY_LEVEL = {
    # Ajusta únicamente estas dos rutas si en tu equipo están en otra carpeta.
    "nivel_1": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel1.tif"
    ),
    "nivel_2": Path(
        r"C:\Users\Ivan sua\anaconda3\Rasters_CLC\Raster_Almeidas_Nivel2.tif"
    ),
}

# Los TIFF de referencia inspeccionados contienen:
# Nivel 1 -> 0,1,2,3,4,5
# Nivel 2 -> 0,1,2,...,12
# El valor 0 corresponde al fondo/NoData temático.
REFERENCE_BACKGROUND_CODE = 0
REFERENCE_CODES_BY_LEVEL = {
    "nivel_1": [1, 2, 3, 4, 5],
    "nivel_2": list(range(1, 13)),
}

# El código 13 NO aparece en el ráster de referencia de Nivel 2.
# El modelo sí tiene 13 salidas. La clase del modelo que no tenga un
# equivalente presente en la referencia se conservará como un código extra.
AUTO_HARMONIZE_REFERENCE_CODES = True

# La correspondencia de códigos se estima SOLO con 20 % de los bloques.
# El 80 % restante se utiliza para la evaluación principal, evitando usar
# exactamente los mismos píxeles para decidir el remapeo y para evaluarlo.
HARMONIZATION_BLOCK_MODULUS = 5
HARMONIZATION_CALIBRATION_REMAINDER = 0

# Guarda además las métricas sobre todos los píxeles válidos como información
# descriptiva. La métrica principal de verificación del remapeo será HOLDOUT.
SAVE_FULL_HARMONIZED_METRICS = False

# Mapeo de Nivel 1 inferido a partir de la predicción previa y el ráster de
# referencia proporcionado. Se usa únicamente como control de coherencia.
EXPECTED_LEVEL1_REFERENCE_MAPPING = {
    1: 2,  # salida modelo 1 -> código referencia 2
    2: 1,  # salida modelo 2 -> código referencia 1
    3: 5,  # salida modelo 3 -> código referencia 5
    4: 3,  # salida modelo 4 -> código referencia 3
    5: 4,  # salida modelo 5 -> código referencia 4
}

# Reutiliza una predicción final existente y continúa con el siguiente nivel.
# Cambiar a False cuando se necesite regenerar los GeoTIFF.
REUSE_COMPLETED_PREDICTIONS = True

# Para la predicción final se recomienda usar únicamente best_model.pt.
# Cambiar a True solo para permitir last_checkpoint.pt cuando no exista
# un mejor modelo guardado.
ALLOW_LAST_CHECKPOINT = False

# Máscara opcional con 1 para píxeles válidos y 0 para nubes, sombras o exclusiones.
VALIDITY_MASK_RASTER = None

# Predicción exclusivamente en CPU.
DEVICE = torch.device("cpu")

torch.set_num_threads(3)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

USE_AMP = False

# Teselas más pequeñas para reducir el pico de RAM.
TILE_SIZE_BY_LEVEL = {"nivel_1": 128, "nivel_2": 128}
OVERLAP_BY_LEVEL = {"nivel_1": 16, "nivel_2": 16}

# Los acumuladores permanecen en disco mediante np.memmap.
RESUME_PREDICTION = True
FLUSH_EVERY_TILES = 3

# En CPU esta variable controla cada cuántas teselas se fuerza gc.collect().
EMPTY_CACHE_EVERY_TILES = 25
SAVE_CONFUSION_PLOT = False
OUTPUT_NODATA = 0
TEMP_DIR = OUTPUT_DIR / "_temporales"

# El código evalúa todos los píxeles válidos en la zona de coincidencia entre
# la predicción y el ráster de referencia. Cuando la referencia corresponde
# a un año diferente de la imagen, los resultados representan concordancia
# espacial con esa referencia histórica y no exactitud temática del año 2025.
EVALUATION_NOTE = (
    "Evaluación censal de todos los píxeles válidos coincidentes. "
    "Si la referencia es de un año diferente, las métricas expresan "
    "concordancia espacial con la referencia histórica."
)

CLASES_NIVEL_1 = {
    0: "NoData",
    1: "2. Territorios agrícolas",
    2: "3. Bosques y áreas seminaturales",
    3: "5. Superficies de agua",
    4: "1. Territorios artificializados",
    5: "4. Áreas húmedas",
}
CLASES_NIVEL_2 = {
    0: "NoData",
    1: "2.4 Áreas agrícolas heterogéneas",
    2: "3.2 Áreas con vegetación herbácea o arbustiva",
    3: "3.1 Bosques",
    4: "2.3 Pastos",
    5: "5.1 Aguas continentales",
    6: "1.1 Zonas urbanizadas",
    7: "1.3 Zonas de extracción minera y escombreras",
    8: "4.1 Áreas húmedas continentales",
    9: "1.2 Zonas industriales o comerciales y redes de comunicación",
    10: "1.4 Zonas verdes artificializadas no agrícolas",
    11: "2.1 Cultivos transitorios",
    12: "2.2 Cultivos permanentes",
    13: "3.3 Áreas abiertas sin o con poca vegetación",
}


def get_class_dict(level_name: str) -> dict[int, str]:
    if level_name == "nivel_1":
        return CLASES_NIVEL_1
    if level_name == "nivel_2":
        return CLASES_NIVEL_2
    raise ValueError(level_name)


# =============================================================================
# 2. ARQUITECTURA
# =============================================================================

class ConvNormAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        groups = min(32, out_channels)
        while out_channels % groups != 0:
            groups -= 1
        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.insert(3, nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, dropout: float = 0.0):
        super().__init__()
        self.conv = ConvNormAct(in_channels + skip_channels, out_channels, dropout)

    def forward(self, decoder, skip):
        decoder = F.interpolate(
            decoder,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return self.conv(torch.cat([decoder, skip], dim=1))



def _valid_group_count(channels: int, maximum_groups: int = 32) -> int:
    """Selecciona el mayor número de grupos que divide exactamente los canales."""
    for groups in (maximum_groups, 16, 8, 4, 2, 1):
        if groups <= channels and channels % groups == 0:
            return groups
    return 1


def replace_batchnorm_with_groupnorm(module: nn.Module) -> nn.Module:
    """
    Sustituye recursivamente BatchNorm2d por GroupNorm.

    Esta conversión evita la operación BatchNorm de MIOpen que produjo
    miopenStatusUnknownError y es más estable con batch size 1–4.
    Los parámetros afines de los pesos preentrenados se conservan.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.BatchNorm2d):
            replacement = nn.GroupNorm(
                num_groups=_valid_group_count(child.num_features),
                num_channels=child.num_features,
                eps=child.eps,
                affine=child.affine,
            )
            if child.affine:
                with torch.no_grad():
                    replacement.weight.copy_(child.weight)
                    replacement.bias.copy_(child.bias)
            setattr(module, name, replacement)
        else:
            replace_batchnorm_with_groupnorm(child)
    return module


def adapt_first_convolution(old_conv: nn.Conv2d, in_channels: int, pretrained_loaded: bool) -> nn.Conv2d:
    new_conv = nn.Conv2d(
        in_channels,
        old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=old_conv.stride,
        padding=old_conv.padding,
        bias=False,
    )
    with torch.no_grad():
        if pretrained_loaded:
            # Entrada Sentinel-2: B2, B3, B4, B8. Los pesos ImageNet vienen en RGB.
            if in_channels >= 3:
                new_conv.weight[:, 0] = old_conv.weight[:, 2]  # B2 <- azul
                new_conv.weight[:, 1] = old_conv.weight[:, 1]  # B3 <- verde
                new_conv.weight[:, 2] = old_conv.weight[:, 0]  # B4 <- rojo
                if in_channels > 3:
                    mean_weight = old_conv.weight.mean(dim=1, keepdim=True)
                    new_conv.weight[:, 3:] = mean_weight.repeat(1, in_channels - 3, 1, 1)
            else:
                new_conv.weight[:, :in_channels] = old_conv.weight[:, :in_channels]
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")
    return new_conv


class ResNet50UNet(nn.Module):
    """Segmentación semántica con ResNet-50 como codificador y decodificador tipo U-Net."""

    def __init__(
        self,
        in_channels: int,
        num_classes: int,
        pretrained: bool = True,
        freeze_encoder: bool = False,
        decoder_dropout: float = 0.10,
    ):
        super().__init__()
        pretrained_loaded = False
        weights = None
        if pretrained:
            try:
                weights = models.ResNet50_Weights.DEFAULT
                encoder = models.resnet50(weights=weights)
                pretrained_loaded = True
            except Exception as exc:
                warnings.warn(
                    f"No fue posible cargar pesos ImageNet de ResNet-50 ({exc}); se inicializa sin pesos.",
                    RuntimeWarning,
                )
                encoder = models.resnet50(weights=None)
        else:
            encoder = models.resnet50(weights=None)

        encoder.conv1 = adapt_first_convolution(
            encoder.conv1,
            in_channels,
            pretrained_loaded,
        )
        encoder = replace_batchnorm_with_groupnorm(encoder)

        self.conv1 = encoder.conv1
        self.bn1 = encoder.bn1
        self.relu = encoder.relu
        self.maxpool = encoder.maxpool
        self.layer1 = encoder.layer1   # 256 canales, 1/4
        self.layer2 = encoder.layer2   # 512 canales, 1/8
        self.layer3 = encoder.layer3   # 1024 canales, 1/16
        self.layer4 = encoder.layer4   # 2048 canales, 1/32

        self.decoder4 = DecoderBlock(2048, 1024, 512, decoder_dropout)
        self.decoder3 = DecoderBlock(512, 512, 256, decoder_dropout)
        self.decoder2 = DecoderBlock(256, 256, 128, decoder_dropout)
        self.decoder1 = DecoderBlock(128, 64, 64, decoder_dropout)
        self.final_refine = ConvNormAct(64, 32, decoder_dropout)
        self.segmentation_head = nn.Conv2d(32, num_classes, kernel_size=1)
        self.pretrained_loaded = pretrained_loaded

        if freeze_encoder:
            for module in (
                self.conv1, self.bn1, self.layer1, self.layer2, self.layer3, self.layer4
            ):
                for parameter in module.parameters():
                    parameter.requires_grad = False

    def forward(self, x):
        input_size = x.shape[-2:]
        skip0 = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(skip0)
        skip1 = self.layer1(x)
        skip2 = self.layer2(skip1)
        skip3 = self.layer3(skip2)
        x = self.layer4(skip3)

        x = self.decoder4(x, skip3)
        x = self.decoder3(x, skip2)
        x = self.decoder2(x, skip1)
        x = self.decoder1(x, skip0)
        x = F.interpolate(x, size=input_size, mode="bilinear", align_corners=False)
        x = self.final_refine(x)
        return self.segmentation_head(x)


def build_segmentation_model(
    in_channels: int,
    num_classes: int,
    pretrained: bool = False,
    freeze_encoder: bool = False,
):
    model = ResNet50UNet(
        in_channels=in_channels,
        num_classes=num_classes,
        pretrained=pretrained,
        freeze_encoder=freeze_encoder,
        decoder_dropout=0.10,
    )
    return model, bool(model.pretrained_loaded)


# =============================================================================
# 3. UTILIDADES DE MODELO Y PREPROCESAMIENTO
# =============================================================================
def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data: dict, path: str | Path) -> Path:
    """
    Guarda un diccionario en JSON de forma segura.

    Primero escribe un archivo temporal y después lo reemplaza para evitar
    estados de reanudación dañados tras un cierre inesperado.
    """
    path = Path(path)
    ensure_dir(path.parent)
    temporary_path = path.with_suffix(path.suffix + ".tmp")

    with temporary_path.open("w", encoding="utf-8") as file:
        json.dump(
            data,
            file,
            indent=2,
            ensure_ascii=False,
        )

    temporary_path.replace(path)
    return path


def safe_cuda_cleanup() -> None:
    """
    Compatibilidad con el resto del código.

    En esta versión la GPU está desactivada. La función solo libera objetos
    de Python y memoria de CPU; no consulta ni inicializa CUDA/ROCm.
    """
    gc.collect()


def close_memmap(array) -> None:
    """
    Cierra explícitamente un np.memmap.

    En Windows no basta con usar del; el descriptor del archivo puede seguir
    abierto y producir WinError 32 al intentar eliminarlo.
    """
    if array is None:
        return

    try:
        array.flush()
    except Exception:
        pass

    mmap_object = getattr(array, "_mmap", None)
    if mmap_object is not None:
        try:
            mmap_object.close()
        except Exception as close_error:
            warnings.warn(
                f"No fue posible cerrar completamente un archivo temporal: "
                f"{close_error}",
                RuntimeWarning,
            )


def safe_unlink(
    path: str | Path,
    retries: int = 6,
    wait_seconds: float = 1.0,
) -> bool:
    """
    Intenta eliminar un archivo varias veces.

    Si Windows mantiene un bloqueo temporal, el código no se detiene:
    muestra una advertencia y continúa.
    """
    path = Path(path)

    if not path.exists():
        return True

    last_error = None

    for attempt in range(1, retries + 1):
        try:
            path.unlink()
            return True
        except FileNotFoundError:
            return True
        except (PermissionError, OSError) as error:
            last_error = error
            gc.collect()
            time.sleep(wait_seconds)

    warnings.warn(
        f"No se pudo eliminar el archivo temporal bloqueado: {path}. "
        f"Se conservará sin afectar la predicción. Error: {last_error}",
        RuntimeWarning,
    )
    return False


def load_active_temp_paths(
    temp_dir: Path,
) -> tuple[Path, Path, Path, Path]:
    """
    Recupera los nombres de los archivos temporales activos.

    Esto permite reanudar una predicción aunque, por un bloqueo anterior,
    se haya tenido que crear un archivo temporal con nombre alternativo.
    """
    manifest_path = temp_dir / "archivos_temporales_activos.json"

    default_probability = temp_dir / "probability_sum.dat"
    default_weight = temp_dir / "weight_sum.dat"
    default_state = temp_dir / "estado_prediccion.json"

    if not manifest_path.exists():
        return (
            default_probability,
            default_weight,
            default_state,
            manifest_path,
        )

    try:
        manifest = load_json(manifest_path)
        probability_file = Path(
            manifest.get("probability_file", default_probability)
        )
        weight_file = Path(
            manifest.get("weight_file", default_weight)
        )
        state_path = Path(
            manifest.get("state_path", default_state)
        )

        # Seguridad: todos los archivos deben permanecer dentro del temp_dir.
        resolved_temp = temp_dir.resolve()
        for candidate in (
            probability_file,
            weight_file,
            state_path,
        ):
            if resolved_temp not in candidate.resolve().parents:
                raise ValueError(
                    "El manifiesto contiene una ruta fuera del directorio "
                    "temporal."
                )

        return (
            probability_file,
            weight_file,
            state_path,
            manifest_path,
        )

    except Exception as error:
        warnings.warn(
            f"No fue posible leer el manifiesto temporal: {error}. "
            "Se usarán los nombres predeterminados.",
            RuntimeWarning,
        )
        return (
            default_probability,
            default_weight,
            default_state,
            manifest_path,
        )


def save_active_temp_paths(
    manifest_path: Path,
    probability_file: Path,
    weight_file: Path,
    state_path: Path,
) -> None:
    save_json(
        {
            "probability_file": str(probability_file),
            "weight_file": str(weight_file),
            "state_path": str(state_path),
        },
        manifest_path,
    )


def _validate_run_config(run_dir: Path, level_name: str) -> dict:
    config_path = run_dir / "config.json"
    if not config_path.exists():
        raise FileNotFoundError(
            f"La ejecución no contiene config.json: {run_dir}"
        )

    config = load_json(config_path)

    if str(config.get("task", "")).lower() != "segmentacion_semantica":
        raise ValueError(
            f"La ejecución no corresponde a segmentación semántica: {run_dir}"
        )

    if str(config.get("level", "")).lower() != level_name:
        raise ValueError(
            f"El nivel del config.json no coincide con {level_name}: {run_dir}"
        )

    model_name = str(config.get("model", "")).lower()
    encoder_name = str(config.get("encoder", "")).lower()
    if "resnet50" not in model_name and encoder_name != "resnet50":
        raise ValueError(
            f"La ejecución no corresponde a ResNet-50: {run_dir}"
        )

    band_indexes = tuple(
        int(value)
        for value in config.get("band_indexes", [])
    )
    if len(band_indexes) != 4:
        raise ValueError(
            "La ejecución debe usar cuatro bandas B2, B3, B4 y B8. "
            f"Se encontró: {band_indexes}"
        )

    return config


def find_run_dir(level_name: str) -> Path:
    forced = RUN_BY_LEVEL.get(level_name)

    if forced:
        candidate = MODEL_ROOT / forced
        if not candidate.exists():
            raise FileNotFoundError(candidate)
        _validate_run_config(candidate, level_name)
        return candidate

    candidates = []

    for run_dir in MODEL_ROOT.glob(f"ResNet50Seg_{level_name}_*"):
        try:
            _validate_run_config(run_dir, level_name)
        except Exception:
            continue

        best_model = run_dir / "best_model.pt"
        last_checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
        completed_marker = run_dir / "LEVEL_COMPLETED.json"
        final_report = run_dir / "reports" / "resumen_metricas_test.json"

        if not best_model.exists() and not (
            ALLOW_LAST_CHECKPOINT and last_checkpoint.exists()
        ):
            continue

        completed = completed_marker.exists() or final_report.exists()
        evidence = [
            path.stat().st_mtime
            for path in (
                completed_marker,
                final_report,
                best_model,
                last_checkpoint,
            )
            if path.exists()
        ]
        modified = max(evidence) if evidence else run_dir.stat().st_mtime

        # Se priorizan las ejecuciones finalizadas y después la más reciente.
        candidates.append((1 if completed else 0, modified, run_dir))

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró un modelo de segmentación válido para "
            f"{level_name} en {MODEL_ROOT}. Verifica que exista "
            "best_model.pt y config.json."
        )

    return max(candidates, key=lambda item: (item[0], item[1]))[2]


def checkpoint_path(run_dir: Path) -> Path:
    best_model = run_dir / "best_model.pt"
    if best_model.exists():
        return best_model

    last_checkpoint = run_dir / "checkpoints" / "last_checkpoint.pt"
    if ALLOW_LAST_CHECKPOINT and last_checkpoint.exists():
        warnings.warn(
            "No existe best_model.pt; se utilizará last_checkpoint.pt.",
            RuntimeWarning,
        )
        return last_checkpoint

    raise FileNotFoundError(
        f"No existe best_model.pt en {run_dir}. "
        "Finaliza el entrenamiento o cambia ALLOW_LAST_CHECKPOINT a True."
    )


def parse_mapping(mapping) -> dict[int, int]:
    return {int(key): int(value) for key, value in mapping.items()}


def load_model_for_level(level_name: str):
    run_dir = find_run_dir(level_name)
    config = load_json(run_dir / "config.json")
    idx_to_code = parse_mapping(config["idx_to_code"])
    idx_to_name = {int(k): str(v) for k, v in config["idx_to_name"].items()}
    band_indexes = tuple(int(v) for v in config.get("band_indexes", [1, 2, 3, 4]))
    num_classes = int(config["num_classes"])

    model, _ = build_segmentation_model(
        in_channels=len(band_indexes),
        num_classes=num_classes,
        pretrained=False,
        freeze_encoder=False,
    )
    selected_checkpoint = checkpoint_path(run_dir)
    try:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(
            selected_checkpoint,
            map_location="cpu",
        )

    state = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state, strict=True)
    del state, checkpoint
    gc.collect()
    expected_indices = list(range(len(idx_to_code)))
    if sorted(idx_to_code) != expected_indices:
        raise ValueError(
            "El idx_to_code del checkpoint no es consecutivo desde cero: "
            f"{idx_to_code}"
        )
    if len(set(idx_to_code.values())) != len(idx_to_code):
        raise ValueError(
            f"El checkpoint contiene códigos temáticos duplicados: {idx_to_code}"
        )
    model = model.to(DEVICE).eval()
    return model, config, run_dir, idx_to_code, idx_to_name, band_indexes


def amp_context():
    # CPU estable: sin autocast de GPU.
    return nullcontext()


def reflectance_window(src, window: Window, band_indexes, config):
    array = src.read(indexes=band_indexes, window=window).astype(np.float32)
    finite = np.isfinite(array)
    if src.nodata is not None and np.isfinite(src.nodata):
        finite &= array != src.nodata
    valid = finite.all(axis=0)
    array = np.where(finite, array, np.nan)

    scales = np.asarray([src.scales[i - 1] for i in band_indexes], dtype=np.float32)[:, None, None]
    offsets = np.asarray([src.offsets[i - 1] for i in band_indexes], dtype=np.float32)[:, None, None]
    use_transform = bool(config.get("use_raster_scale_offset", True)) and (
        not np.allclose(scales, 1.0) or not np.allclose(offsets, 0.0)
    )
    finite_values = array[np.isfinite(array)]
    if use_transform:
        array = array * scales + offsets
    elif bool(config.get("auto_scale_reflectance", True)) and finite_values.size and float(np.nanpercentile(finite_values, 99.5)) > 2.0:
        array = (array + float(config.get("reflectance_add_offset", 0.0))) / float(config.get("reflectance_scale", 10000.0))

    array = np.clip(array, 0.0, 1.0)
    means = np.asarray(config["means"], dtype=np.float32)[:, None, None]
    stds = np.asarray(config["stds"], dtype=np.float32)[:, None, None]
    array = np.nan_to_num((array - means) / np.maximum(stds, 1e-6), nan=0.0)
    array[:, ~valid] = 0.0
    return array.astype(np.float32), valid


def pad_to_multiple(tensor: torch.Tensor, multiple: int = 32):
    height, width = tensor.shape[-2:]
    pad_h = (multiple - height % multiple) % multiple
    pad_w = (multiple - width % multiple) % multiple
    padded = F.pad(tensor, (0, pad_w, 0, pad_h), mode="replicate")
    return padded, height, width


def starts(length: int, tile_size: int, overlap: int) -> list[int]:
    if tile_size <= overlap:
        raise ValueError("TILE_SIZE debe ser mayor que OVERLAP")
    if length <= tile_size:
        return [0]
    stride = tile_size - overlap
    values = list(range(0, length - tile_size + 1, stride))
    final = length - tile_size
    if values[-1] != final:
        values.append(final)
    return values


def blend_weight(height: int, width: int) -> np.ndarray:
    wy = np.hanning(height) if height > 2 else np.ones(height)
    wx = np.hanning(width) if width > 2 else np.ones(width)
    weight = np.outer(wy, wx).astype(np.float32)
    return np.maximum(weight, 0.05)


# =============================================================================
# 4. PREDICCIÓN PÍXEL A PÍXEL
# =============================================================================
@torch.inference_mode()
def predict_level(level_name: str):
    (
        model,
        config,
        run_dir,
        idx_to_code,
        idx_to_name,
        band_indexes,
    ) = load_model_for_level(level_name)

    if not INPUT_RASTER.exists():
        raise FileNotFoundError(
            f"No existe INPUT_RASTER: {INPUT_RASTER}"
        )

    tile_size = int(TILE_SIZE_BY_LEVEL[level_name])
    overlap = int(OVERLAP_BY_LEVEL[level_name])

    if tile_size <= overlap:
        raise ValueError(
            "El tamaño de tesela debe ser mayor que el solapamiento."
        )

    level_dir = ensure_dir(OUTPUT_DIR / level_name)
    temp_dir = ensure_dir(TEMP_DIR / level_name)

    prediction_path = (
        level_dir / f"{MODEL_NAME}_{level_name}_prediccion.tif"
    )
    confidence_path = (
        level_dir / f"{MODEL_NAME}_{level_name}_confianza.tif"
    )
    completed_path = level_dir / "PREDICTION_COMPLETED.json"

    if (
        REUSE_COMPLETED_PREDICTIONS
        and prediction_path.exists()
        and confidence_path.exists()
        and completed_path.exists()
    ):
        print(
            f"{level_name} ya tiene una predicción final completa. "
            "Se reutilizará para calcular las métricas."
        )
        del model
        safe_cuda_cleanup()

        return (
            prediction_path,
            confidence_path,
            run_dir,
            idx_to_code,
            idx_to_name,
        )

    (
        probability_file,
        weight_file,
        state_path,
        manifest_path,
    ) = load_active_temp_paths(temp_dir)

    probability_sum = None
    weight_sum = None
    validity_vrt = None
    validity_src = None

    try:
        with rasterio.open(INPUT_RASTER) as src:
            height = src.height
            width = src.width
            num_classes = len(idx_to_code)

            if max(band_indexes) > src.count:
                raise ValueError(
                    f"La imagen tiene {src.count} bandas y se solicitaron "
                    f"{band_indexes}"
                )

            row_starts = starts(
                height,
                tile_size,
                overlap,
            )
            col_starts = starts(
                width,
                tile_size,
                overlap,
            )
            tile_positions = [
                (row, col)
                for row in row_starts
                for col in col_starts
            ]
            total_tiles = len(tile_positions)

            selected_checkpoint = checkpoint_path(run_dir)

            expected_state = {
                "height": height,
                "width": width,
                "num_classes": num_classes,
                "tile_size": tile_size,
                "overlap": overlap,
                "level": level_name,
                "input_raster": str(INPUT_RASTER.resolve()),
                "run_dir": str(run_dir.resolve()),
                "checkpoint": str(selected_checkpoint.resolve()),
                "band_indexes": list(band_indexes),
            }

            next_tile_index = 0

            can_resume = (
                RESUME_PREDICTION
                and state_path.exists()
                and probability_file.exists()
                and weight_file.exists()
            )

            if can_resume:
                try:
                    state = load_json(state_path)
                    compatible = all(
                        state.get(key) == value
                        for key, value in expected_state.items()
                    )

                    if compatible:
                        next_tile_index = int(
                            state.get("next_tile_index", 0)
                        )

                        probability_sum = np.memmap(
                            probability_file,
                            dtype="float32",
                            mode="r+",
                            shape=(
                                num_classes,
                                height,
                                width,
                            ),
                        )
                        weight_sum = np.memmap(
                            weight_file,
                            dtype="float32",
                            mode="r+",
                            shape=(height, width),
                        )

                        print(
                            f"Reanudando {level_name} desde la tesela "
                            f"{next_tile_index} de {total_tiles}."
                        )
                    else:
                        can_resume = False

                except Exception as resume_error:
                    warnings.warn(
                        f"No se pudo reanudar la predicción: "
                        f"{resume_error}. Se reiniciará.",
                        RuntimeWarning,
                    )
                    close_memmap(probability_sum)
                    close_memmap(weight_sum)
                    probability_sum = None
                    weight_sum = None
                    can_resume = False

            if not can_resume:
                close_memmap(probability_sum)
                close_memmap(weight_sum)
                probability_sum = None
                weight_sum = None
                gc.collect()

                deletion_results = [
                    safe_unlink(probability_file),
                    safe_unlink(weight_file),
                    safe_unlink(state_path),
                ]

                if not all(deletion_results):
                    token = str(time.time_ns())
                    probability_file = (
                        temp_dir / f"probability_sum_{token}.dat"
                    )
                    weight_file = (
                        temp_dir / f"weight_sum_{token}.dat"
                    )
                    state_path = (
                        temp_dir / f"estado_prediccion_{token}.json"
                    )

                    print(
                        "Windows mantuvo bloqueado un archivo temporal "
                        "anterior. Se usarán archivos nuevos para continuar."
                    )

                save_active_temp_paths(
                    manifest_path,
                    probability_file,
                    weight_file,
                    state_path,
                )

                probability_sum = np.memmap(
                    probability_file,
                    dtype="float32",
                    mode="w+",
                    shape=(
                        num_classes,
                        height,
                        width,
                    ),
                )
                weight_sum = np.memmap(
                    weight_file,
                    dtype="float32",
                    mode="w+",
                    shape=(height, width),
                )

                probability_sum[:] = 0
                weight_sum[:] = 0
                probability_sum.flush()
                weight_sum.flush()
                next_tile_index = 0

            if VALIDITY_MASK_RASTER is not None:
                validity_src = rasterio.open(
                    VALIDITY_MASK_RASTER
                )
                validity_vrt = WarpedVRT(
                    validity_src,
                    crs=src.crs,
                    transform=src.transform,
                    width=src.width,
                    height=src.height,
                    resampling=Resampling.nearest,
                    nodata=0,
                )

            progress = tqdm(
                range(
                    next_tile_index,
                    total_tiles,
                ),
                total=total_tiles,
                initial=next_tile_index,
                desc=f"Predicción {level_name}",
                leave=False,
                mininterval=5.0,
                dynamic_ncols=False,
            )

            try:
                for tile_index in progress:
                    row, col = tile_positions[tile_index]
                    tile_h = min(
                        tile_size,
                        height - row,
                    )
                    tile_w = min(
                        tile_size,
                        width - col,
                    )
                    window = Window(
                        col,
                        row,
                        tile_w,
                        tile_h,
                    )

                    image, valid = reflectance_window(
                        src,
                        window,
                        band_indexes,
                        config,
                    )

                    if validity_vrt is not None:
                        valid &= (
                            validity_vrt.read(
                                1,
                                window=window,
                            )
                            > 0
                        )

                    if valid.any():
                        tensor = (
                            torch.from_numpy(image)
                            .unsqueeze(0)
                            .to(
                                DEVICE,
                                non_blocking=False,
                            )
                        )

                        (
                            tensor,
                            original_h,
                            original_w,
                        ) = pad_to_multiple(
                            tensor,
                            32,
                        )

                        with amp_context():
                            logits = model(tensor)
                            tile_probabilities = torch.softmax(
                                logits,
                                dim=1,
                            )

                        tile_probabilities = (
                            tile_probabilities[
                                0,
                                :,
                                :original_h,
                                :original_w,
                            ]
                            .float()
                            .cpu()
                            .numpy()
                        )

                        weight = (
                            blend_weight(
                                tile_h,
                                tile_w,
                            )
                            * valid.astype(np.float32)
                        )

                        probability_sum[
                            :,
                            row:row + tile_h,
                            col:col + tile_w,
                        ] += (
                            tile_probabilities
                            * weight[None, :, :]
                        )

                        weight_sum[
                            row:row + tile_h,
                            col:col + tile_w,
                        ] += weight

                        del (
                            tensor,
                            logits,
                            tile_probabilities,
                            weight,
                        )

                    next_index = tile_index + 1

                    if (
                        next_index % FLUSH_EVERY_TILES == 0
                        or next_index == total_tiles
                    ):
                        probability_sum.flush()
                        weight_sum.flush()

                        save_json(
                            {
                                **expected_state,
                                "next_tile_index": next_index,
                                "total_tiles": total_tiles,
                                "model": MODEL_NAME,
                                "level": level_name,
                                "probability_file": str(
                                    probability_file
                                ),
                                "weight_file": str(
                                    weight_file
                                ),
                            },
                            state_path,
                        )

                    if (
                        next_index
                        % EMPTY_CACHE_EVERY_TILES
                        == 0
                    ):
                        # Libera objetos temporales sin tocar GPU.
                        safe_cuda_cleanup()

            finally:
                progress.close()

                if probability_sum is not None:
                    try:
                        probability_sum.flush()
                    except Exception:
                        pass

                if weight_sum is not None:
                    try:
                        weight_sum.flush()
                    except Exception:
                        pass

                if validity_vrt is not None:
                    validity_vrt.close()
                    validity_vrt = None

                if validity_src is not None:
                    validity_src.close()
                    validity_src = None

            class_profile = src.profile.copy()
            class_profile.update(
                count=1,
                dtype="uint16",
                nodata=OUTPUT_NODATA,
                compress="lzw",
                tiled=True,
            )

            confidence_profile = src.profile.copy()
            confidence_profile.update(
                count=1,
                dtype="float32",
                nodata=0.0,
                compress="lzw",
                tiled=True,
            )

            code_array = np.asarray(
                [
                    idx_to_code[index]
                    for index in range(num_classes)
                ],
                dtype=np.uint16,
            )

            writing_progress = tqdm(
                range(0, height, 256),
                desc="Escritura GeoTIFF",
                leave=False,
                mininterval=3.0,
                dynamic_ncols=False,
            )

            try:
                with rasterio.open(
                    prediction_path,
                    "w",
                    **class_profile,
                ) as dst_class, rasterio.open(
                    confidence_path,
                    "w",
                    **confidence_profile,
                ) as dst_conf:
                    for row in writing_progress:
                        block_height = min(
                            256,
                            height - row,
                        )

                        probabilities = np.asarray(
                            probability_sum[
                                :,
                                row:row + block_height,
                                :,
                            ],
                            dtype=np.float32,
                        )

                        weights = np.asarray(
                            weight_sum[
                                row:row + block_height,
                                :,
                            ],
                            dtype=np.float32,
                        )

                        valid = weights > 0

                        probabilities = np.divide(
                            probabilities,
                            weights[None, :, :],
                            out=np.zeros_like(
                                probabilities
                            ),
                            where=valid[None, :, :],
                        )

                        indices = probabilities.argmax(
                            axis=0
                        )
                        confidence = (
                            probabilities.max(axis=0)
                            .astype(np.float32)
                        )

                        prediction = np.full(
                            (
                                block_height,
                                width,
                            ),
                            OUTPUT_NODATA,
                            dtype=np.uint16,
                        )

                        prediction[valid] = (
                            code_array[
                                indices[valid]
                            ]
                        )
                        confidence[~valid] = 0.0

                        output_window = Window(
                            0,
                            row,
                            width,
                            block_height,
                        )

                        dst_class.write(
                            prediction,
                            1,
                            window=output_window,
                        )
                        dst_conf.write(
                            confidence,
                            1,
                            window=output_window,
                        )

                        del (
                            probabilities,
                            weights,
                            valid,
                            indices,
                            confidence,
                            prediction,
                        )
                        gc.collect()

            finally:
                writing_progress.close()

        # Guardar primero el marcador de finalización.
        # Así, un archivo temporal bloqueado nunca invalida la predicción.
        save_json(
            {
                "completed": True,
                "model": MODEL_NAME,
                "level": level_name,
                "input_raster": str(INPUT_RASTER),
                "run_dir": str(run_dir),
                "checkpoint": str(
                    checkpoint_path(run_dir)
                ),
                "prediction": str(prediction_path),
                "confidence": str(confidence_path),
                "tile_size": tile_size,
                "overlap": overlap,
                "band_indexes": list(band_indexes),
            },
            completed_path,
        )

    finally:
        if validity_vrt is not None:
            try:
                validity_vrt.close()
            except Exception:
                pass

        if validity_src is not None:
            try:
                validity_src.close()
            except Exception:
                pass

        close_memmap(probability_sum)
        close_memmap(weight_sum)
        probability_sum = None
        weight_sum = None

        try:
            del model
        except Exception:
            pass

        safe_cuda_cleanup()

    # La eliminación de temporales es opcional.
    # Un bloqueo de Windows ya no detiene la predicción ni las métricas.
    safe_unlink(probability_file)
    safe_unlink(weight_file)
    safe_unlink(state_path)
    safe_unlink(manifest_path)

    return (
        prediction_path,
        confidence_path,
        run_dir,
        idx_to_code,
        idx_to_name,
    )



# =============================================================================
# 5. ARMONIZACIÓN AUTOMÁTICA DE CÓDIGOS CON LA REFERENCIA
# =============================================================================

def _maximum_weight_assignment(matrix: np.ndarray):
    """
    Asignación exacta de máximo peso entre filas (códigos de referencia)
    y columnas (códigos de salida del modelo).

    Implementación por programación dinámica con bitmask. Para 5 y 13 clases
    es pequeña, exacta y evita depender de SciPy.
    """
    matrix = np.asarray(matrix, dtype=np.int64)
    rows, cols = matrix.shape

    transposed = False
    if rows > cols:
        matrix = matrix.T
        rows, cols = matrix.shape
        transposed = True

    # dp[(row_index, used_mask)] = (score, tuple(columns))
    from functools import lru_cache

    @lru_cache(maxsize=None)
    def solve(row_index: int, used_mask: int):
        if row_index >= rows:
            return 0, ()

        best_score = -1
        best_assignment = None

        for column in range(cols):
            if used_mask & (1 << column):
                continue

            tail_score, tail_assignment = solve(
                row_index + 1,
                used_mask | (1 << column),
            )
            score = int(matrix[row_index, column]) + int(tail_score)

            if score > best_score:
                best_score = score
                best_assignment = (column,) + tuple(tail_assignment)

        return best_score, best_assignment

    score, assignment = solve(0, 0)

    if not transposed:
        row_indexes = np.arange(rows, dtype=np.int64)
        column_indexes = np.asarray(assignment, dtype=np.int64)
    else:
        # La matriz se transpusó: filas actuales eran columnas originales.
        row_indexes = np.asarray(assignment, dtype=np.int64)
        column_indexes = np.arange(rows, dtype=np.int64)

    return row_indexes, column_indexes, int(score)


def _mapping_calibration_block(block_index: int) -> bool:
    return (
        block_index % HARMONIZATION_BLOCK_MODULUS
        == HARMONIZATION_CALIBRATION_REMAINDER
    )


def _collect_reference_codes(reference_path: Path) -> list[int]:
    observed = set()

    with rasterio.open(reference_path) as src:
        for _, window in src.block_windows(1):
            values = src.read(1, window=window)
            unique_values = np.unique(values)
            observed.update(int(value) for value in unique_values.tolist())

    observed.discard(int(REFERENCE_BACKGROUND_CODE))
    return sorted(observed)


def derive_reference_code_mapping(
    prediction_path: Path,
    reference_path: Path,
    model_code_to_name: dict[int, str],
    level_name: str,
):
    """
    Calcula la correspondencia entre los códigos que produce el modelo y los
    códigos numéricos presentes en el ráster de referencia.

    IMPORTANTE:
    - La red neuronal no se modifica.
    - No se cambian las probabilidades.
    - Solo se armonizan identificadores numéricos de clase.
    - La correspondencia se estima con un subconjunto de bloques.
    """
    expected_reference_codes = [
        int(value)
        for value in REFERENCE_CODES_BY_LEVEL[level_name]
    ]

    observed_reference_codes = _collect_reference_codes(reference_path)

    missing_expected = sorted(
        set(expected_reference_codes) - set(observed_reference_codes)
    )
    unexpected_reference = sorted(
        set(observed_reference_codes) - set(expected_reference_codes)
    )

    if unexpected_reference:
        warnings.warn(
            f"{level_name}: la referencia contiene códigos no esperados: "
            f"{unexpected_reference}. Se incluirán en el análisis.",
            RuntimeWarning,
        )

    reference_codes = sorted(
        set(expected_reference_codes) & set(observed_reference_codes)
    )

    if not reference_codes:
        raise ValueError(
            f"{level_name}: no se encontraron códigos temáticos válidos "
            f"en la referencia {reference_path}."
        )

    model_codes = sorted(int(code) for code in model_code_to_name)

    reference_to_row = {
        code: index
        for index, code in enumerate(reference_codes)
    }
    model_to_column = {
        code: index
        for index, code in enumerate(model_codes)
    }

    confusion = np.zeros(
        (len(reference_codes), len(model_codes)),
        dtype=np.int64,
    )

    calibration_pixels = 0

    with rasterio.open(prediction_path) as pred, rasterio.open(
        reference_path
    ) as reference_src:
        reference_vrt = WarpedVRT(
            reference_src,
            crs=pred.crs,
            transform=pred.transform,
            width=pred.width,
            height=pred.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for block_index, (_, window) in enumerate(
                pred.block_windows(1)
            ):
                if not _mapping_calibration_block(block_index):
                    continue

                predicted = pred.read(
                    1,
                    window=window,
                ).astype(np.int64)

                reference = reference_vrt.read(
                    1,
                    window=window,
                ).astype(np.int64)

                valid = np.isin(
                    predicted,
                    model_codes,
                ) & np.isin(
                    reference,
                    reference_codes,
                )

                if not valid.any():
                    continue

                true_values = reference[valid]
                pred_values = predicted[valid]
                calibration_pixels += int(valid.sum())

                for true_code in reference_codes:
                    true_mask = true_values == true_code
                    if not true_mask.any():
                        continue

                    corresponding_predictions = pred_values[true_mask]
                    values, counts = np.unique(
                        corresponding_predictions,
                        return_counts=True,
                    )

                    row = reference_to_row[true_code]
                    for predicted_code, count in zip(
                        values.tolist(),
                        counts.tolist(),
                    ):
                        if int(predicted_code) in model_to_column:
                            confusion[
                                row,
                                model_to_column[int(predicted_code)],
                            ] += int(count)
        finally:
            reference_vrt.close()

    if calibration_pixels == 0:
        raise ValueError(
            f"{level_name}: no se encontraron píxeles válidos para "
            "estimar el remapeo."
        )

    row_indexes, column_indexes, optimal_score = (
        _maximum_weight_assignment(confusion)
    )

    mapping = {}
    rows = []

    matched_model_codes = set()
    matched_reference_codes = set()

    for row_index, column_index in zip(
        row_indexes.tolist(),
        column_indexes.tolist(),
    ):
        reference_code = int(reference_codes[row_index])
        model_code = int(model_codes[column_index])

        matched_model_codes.add(model_code)
        matched_reference_codes.add(reference_code)

        matched_pixels = int(
            confusion[row_index, column_index]
        )
        predicted_total = int(
            confusion[:, column_index].sum()
        )
        reference_total = int(
            confusion[row_index, :].sum()
        )

        mapping[model_code] = reference_code

        rows.append(
            {
                "model_output_code": model_code,
                "model_class_name": model_code_to_name.get(
                    model_code,
                    f"Clase modelo {model_code}",
                ),
                "reference_code": reference_code,
                "matched_calibration_pixels": matched_pixels,
                "model_code_calibration_pixels": predicted_total,
                "reference_code_calibration_pixels": reference_total,
                "mapping_purity": (
                    matched_pixels / predicted_total
                    if predicted_total > 0
                    else np.nan
                ),
                "mapping_recall": (
                    matched_pixels / reference_total
                    if reference_total > 0
                    else np.nan
                ),
                "matched": True,
            }
        )

    # Una referencia puede no contener todas las clases del modelo.
    # En Nivel 2 hay 13 salidas del modelo y solo 12 códigos temáticos
    # observados en el ráster proporcionado.
    unmatched_model_codes = [
        code
        for code in model_codes
        if code not in matched_model_codes
    ]

    next_extra_code = max(reference_codes) + 1

    for model_code in unmatched_model_codes:
        while next_extra_code in matched_reference_codes:
            next_extra_code += 1

        mapping[model_code] = int(next_extra_code)

        column_index = model_to_column[model_code]
        predicted_total = int(confusion[:, column_index].sum())

        rows.append(
            {
                "model_output_code": int(model_code),
                "model_class_name": model_code_to_name.get(
                    model_code,
                    f"Clase modelo {model_code}",
                ),
                "reference_code": int(next_extra_code),
                "matched_calibration_pixels": 0,
                "model_code_calibration_pixels": predicted_total,
                "reference_code_calibration_pixels": 0,
                "mapping_purity": 0.0,
                "mapping_recall": np.nan,
                "matched": False,
            }
        )

        next_extra_code += 1

    mapping_df = (
        pd.DataFrame(rows)
        .sort_values("model_output_code")
        .reset_index(drop=True)
    )

    identity_score = 0
    for reference_code in reference_codes:
        if reference_code in model_to_column:
            identity_score += int(
                confusion[
                    reference_to_row[reference_code],
                    model_to_column[reference_code],
                ]
            )

    identity_accuracy = (
        identity_score / calibration_pixels
        if calibration_pixels > 0
        else np.nan
    )
    optimal_accuracy = (
        optimal_score / calibration_pixels
        if calibration_pixels > 0
        else np.nan
    )

    summary = {
        "level": level_name,
        "calibration_pixels": int(calibration_pixels),
        "reference_codes_observed": observed_reference_codes,
        "reference_codes_used": reference_codes,
        "reference_codes_expected_but_absent": missing_expected,
        "model_codes": model_codes,
        "identity_accuracy_calibration": float(identity_accuracy),
        "optimal_mapping_accuracy_calibration": float(optimal_accuracy),
        "mapping_gain_calibration": float(
            optimal_accuracy - identity_accuracy
        ),
        "unmatched_model_codes": unmatched_model_codes,
        "mapping": {
            int(key): int(value)
            for key, value in mapping.items()
        },
    }

    if level_name == "nivel_1":
        inferred = {
            int(key): int(value)
            for key, value in mapping.items()
        }
        expected = {
            int(key): int(value)
            for key, value in EXPECTED_LEVEL1_REFERENCE_MAPPING.items()
        }
        summary["matches_expected_level1_mapping"] = (
            inferred == expected
        )

        if inferred != expected:
            warnings.warn(
                "El remapeo de Nivel 1 obtenido en esta ejecución difiere "
                f"del remapeo observado previamente. Obtenido={inferred}; "
                f"esperado={expected}. Se conservará el remapeo calculado "
                "con la predicción actual.",
                RuntimeWarning,
            )

    return mapping, mapping_df, summary


def write_harmonized_prediction(
    raw_prediction_path: Path,
    output_path: Path,
    mapping: dict[int, int],
    model_code_to_name: dict[int, str],
):
    """
    Reescribe únicamente los códigos temáticos. No modifica la geometría,
    resolución, extensión ni confianza del modelo.
    """
    ensure_dir(output_path.parent)

    mapping = {
        int(key): int(value)
        for key, value in mapping.items()
    }

    max_input_code = max(mapping) if mapping else 0
    lookup = np.arange(
        max_input_code + 1,
        dtype=np.uint16,
    )

    for source_code, destination_code in mapping.items():
        if source_code >= len(lookup):
            continue
        lookup[source_code] = np.uint16(destination_code)

    class_name_by_reference_code = {}
    for model_code, reference_code in mapping.items():
        class_name_by_reference_code[int(reference_code)] = (
            model_code_to_name.get(
                int(model_code),
                f"Clase modelo {model_code}",
            )
        )

    with rasterio.open(raw_prediction_path) as src:
        profile = src.profile.copy()
        profile.update(
            dtype="uint16",
            nodata=OUTPUT_NODATA,
            count=1,
            compress="lzw",
            tiled=True,
        )

        with rasterio.open(
            output_path,
            "w",
            **profile,
        ) as dst:
            for _, window in src.block_windows(1):
                raw = src.read(
                    1,
                    window=window,
                ).astype(np.int64)

                corrected = np.full(
                    raw.shape,
                    OUTPUT_NODATA,
                    dtype=np.uint16,
                )

                valid = (
                    raw > 0
                ) & (
                    raw <= max_input_code
                )

                corrected[valid] = lookup[
                    raw[valid]
                ]

                dst.write(
                    corrected,
                    1,
                    window=window,
                )

            dst.update_tags(
                CLASS_MAPPING_JSON=json.dumps(
                    {
                        str(code): name
                        for code, name
                        in class_name_by_reference_code.items()
                    },
                    ensure_ascii=False,
                ),
                SOURCE_CODE_MAPPING_JSON=json.dumps(
                    {
                        str(key): int(value)
                        for key, value in mapping.items()
                    },
                    ensure_ascii=False,
                ),
                MAPPING_NOTE=(
                    "Códigos numéricos armonizados con el ráster de "
                    "referencia. Las probabilidades del modelo no se alteraron."
                ),
            )

    return class_name_by_reference_code


def _evaluation_codes_and_names(
    reference_codes: list[int],
    mapping: dict[int, int],
    model_code_to_name: dict[int, str],
):
    code_to_name = {}

    inverse_mapping = {}
    for model_code, corrected_code in mapping.items():
        inverse_mapping[int(corrected_code)] = int(model_code)

    all_codes = sorted(
        set(int(code) for code in reference_codes)
        | set(int(value) for value in mapping.values())
    )

    for code in all_codes:
        if code in inverse_mapping:
            model_code = inverse_mapping[code]
            code_to_name[code] = model_code_to_name.get(
                model_code,
                f"Clase modelo {model_code}",
            )
        else:
            code_to_name[code] = f"Código referencia {code}"

    return all_codes, code_to_name


def evaluate_harmonized_prediction(
    prediction_path: Path,
    reference_path: Path,
    evaluation_codes: list[int],
    code_to_name: dict[int, str],
    selection: str,
):
    """
    selection:
      - 'calibration': solo bloques usados para estimar el remapeo
      - 'holdout': bloques NO usados para estimar el remapeo
      - 'all': todos los bloques
    """
    codes = [int(code) for code in evaluation_codes]
    code_to_index = {
        code: index
        for index, code in enumerate(codes)
    }

    confusion = np.zeros(
        (len(codes), len(codes)),
        dtype=np.int64,
    )

    unknown_prediction = 0
    unknown_reference = 0
    evaluated = 0

    max_code = max(codes)
    lookup = np.full(
        max_code + 1,
        -1,
        dtype=np.int16,
    )
    for code, index in code_to_index.items():
        lookup[code] = index

    with rasterio.open(prediction_path) as pred, rasterio.open(
        reference_path
    ) as reference_src:
        reference_vrt = WarpedVRT(
            reference_src,
            crs=pred.crs,
            transform=pred.transform,
            width=pred.width,
            height=pred.height,
            resampling=Resampling.nearest,
            nodata=REFERENCE_BACKGROUND_CODE,
        )

        try:
            for block_index, (_, window) in enumerate(
                pred.block_windows(1)
            ):
                is_calibration = _mapping_calibration_block(
                    block_index
                )

                if selection == "calibration" and not is_calibration:
                    continue
                if selection == "holdout" and is_calibration:
                    continue

                predicted = pred.read(
                    1,
                    window=window,
                ).astype(np.int64)
                reference = reference_vrt.read(
                    1,
                    window=window,
                ).astype(np.int64)

                valid = (
                    predicted != OUTPUT_NODATA
                ) & (
                    reference != REFERENCE_BACKGROUND_CODE
                )

                if not valid.any():
                    continue

                pred_valid = predicted[valid]
                true_valid = reference[valid]

                pred_known = (
                    (pred_valid >= 0)
                    & (pred_valid <= max_code)
                )
                true_known = (
                    (true_valid >= 0)
                    & (true_valid <= max_code)
                )

                pred_idx = np.full(
                    pred_valid.shape,
                    -1,
                    dtype=np.int16,
                )
                true_idx = np.full(
                    true_valid.shape,
                    -1,
                    dtype=np.int16,
                )

                pred_idx[pred_known] = lookup[
                    pred_valid[pred_known]
                ]
                true_idx[true_known] = lookup[
                    true_valid[true_known]
                ]

                unknown_prediction += int(
                    np.count_nonzero(pred_idx < 0)
                )
                unknown_reference += int(
                    np.count_nonzero(true_idx < 0)
                )

                metric_valid = (
                    pred_idx >= 0
                ) & (
                    true_idx >= 0
                )

                if not metric_valid.any():
                    continue

                true_idx = true_idx[metric_valid].astype(
                    np.int64
                )
                pred_idx = pred_idx[metric_valid].astype(
                    np.int64
                )

                evaluated += int(metric_valid.sum())

                encoded = (
                    true_idx * len(codes)
                    + pred_idx
                )
                confusion += np.bincount(
                    encoded,
                    minlength=len(codes) ** 2,
                ).reshape(len(codes), len(codes))
        finally:
            reference_vrt.close()

    metrics, precision, recall, f1, iou, support, predicted_support, tp = (
        metrics_from_confusion(confusion)
    )

    metrics["evaluated_pixels"] = int(evaluated)
    metrics["unknown_prediction_pixels"] = int(
        unknown_prediction
    )
    metrics["unknown_reference_pixels"] = int(
        unknown_reference
    )
    metrics["selection"] = selection
    metrics["evaluation_note"] = (
        "Concordancia espacial con la referencia histórica. "
        "El remapeo de códigos se estima solo en los bloques de calibración; "
        "la evaluación HOLDOUT usa bloques distintos."
    )

    per_class = pd.DataFrame(
        {
            "class_code": codes,
            "class_name": [
                code_to_name.get(
                    code,
                    f"Clase {code}",
                )
                for code in codes
            ],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "iou": iou,
            "reference_pixels": support.astype(
                np.int64
            ),
            "predicted_pixels": predicted_support.astype(
                np.int64
            ),
            "true_positive": tp.astype(np.int64),
        }
    )

    confusion_df = pd.DataFrame(
        confusion,
        index=[
            f"{code} - {code_to_name.get(code, f'Clase {code}')}"
            for code in codes
        ],
        columns=[
            f"{code} - {code_to_name.get(code, f'Clase {code}')}"
            for code in codes
        ],
    )

    row_totals = confusion.sum(
        axis=1,
        keepdims=True,
    )
    normalized = np.divide(
        confusion,
        row_totals,
        out=np.zeros_like(
            confusion,
            dtype=np.float64,
        ),
        where=row_totals > 0,
    )

    normalized_df = pd.DataFrame(
        normalized,
        index=confusion_df.index,
        columns=confusion_df.columns,
    )

    return metrics, per_class, confusion_df, normalized_df


def save_harmonization_reports(
    level_name: str,
    level_dir: Path,
    mapping_df: pd.DataFrame,
    mapping_summary: dict,
    raw_prediction_path: Path,
    harmonized_prediction_path: Path,
    reference_path: Path,
    model_code_to_name: dict[int, str],
):
    report_dir = ensure_dir(
        level_dir / "armonizacion_codigos"
    )

    mapping_df.to_csv(
        report_dir / "mapeo_codigos.csv",
        index=False,
        encoding="utf-8-sig",
    )
    save_json(
        mapping_summary,
        report_dir / "mapeo_codigos.json",
    )

    reference_codes = [
        int(value)
        for value in mapping_summary[
            "reference_codes_used"
        ]
    ]

    mapping = {
        int(key): int(value)
        for key, value in mapping_summary["mapping"].items()
    }

    evaluation_codes, code_to_name = (
        _evaluation_codes_and_names(
            reference_codes,
            mapping,
            model_code_to_name,
        )
    )

    # Métricas ANTES del remapeo, usando el mismo conjunto HOLDOUT.
    raw_codes = sorted(
        set(reference_codes)
        | set(int(code) for code in model_code_to_name)
    )
    raw_code_to_name = {
        int(code): model_code_to_name.get(
            int(code),
            f"Código {code}",
        )
        for code in raw_codes
    }

    (
        raw_holdout_metrics,
        raw_holdout_per_class,
        raw_holdout_confusion,
        raw_holdout_confusion_norm,
    ) = evaluate_harmonized_prediction(
        raw_prediction_path,
        reference_path,
        raw_codes,
        raw_code_to_name,
        selection="holdout",
    )

    (
        corrected_holdout_metrics,
        corrected_holdout_per_class,
        corrected_holdout_confusion,
        corrected_holdout_confusion_norm,
    ) = evaluate_harmonized_prediction(
        harmonized_prediction_path,
        reference_path,
        evaluation_codes,
        code_to_name,
        selection="holdout",
    )

    save_json(
        raw_holdout_metrics,
        report_dir / "metricas_ANTES_holdout.json",
    )
    save_json(
        corrected_holdout_metrics,
        report_dir / "metricas_CORREGIDAS_holdout.json",
    )

    raw_holdout_per_class.to_csv(
        report_dir / "metricas_ANTES_por_clase_holdout.csv",
        index=False,
        encoding="utf-8-sig",
    )
    corrected_holdout_per_class.to_csv(
        report_dir / "metricas_CORREGIDAS_por_clase_holdout.csv",
        index=False,
        encoding="utf-8-sig",
    )
    raw_holdout_confusion.to_csv(
        report_dir / "confusion_ANTES_holdout.csv",
        encoding="utf-8-sig",
    )
    corrected_holdout_confusion.to_csv(
        report_dir / "confusion_CORREGIDA_holdout.csv",
        encoding="utf-8-sig",
    )
    corrected_holdout_confusion_norm.to_csv(
        report_dir / "confusion_CORREGIDA_normalizada_holdout.csv",
        encoding="utf-8-sig",
    )

    full_metrics = None

    if SAVE_FULL_HARMONIZED_METRICS:
        (
            full_metrics,
            full_per_class,
            full_confusion,
            full_confusion_norm,
        ) = evaluate_harmonized_prediction(
            harmonized_prediction_path,
            reference_path,
            evaluation_codes,
            code_to_name,
            selection="all",
        )

        save_json(
            full_metrics,
            report_dir / "metricas_CORREGIDAS_todos_los_pixeles.json",
        )
        full_per_class.to_csv(
            report_dir / "metricas_CORREGIDAS_por_clase_todos_los_pixeles.csv",
            index=False,
            encoding="utf-8-sig",
        )
        full_confusion.to_csv(
            report_dir / "confusion_CORREGIDA_todos_los_pixeles.csv",
            encoding="utf-8-sig",
        )
        full_confusion_norm.to_csv(
            report_dir / "confusion_CORREGIDA_normalizada_todos_los_pixeles.csv",
            encoding="utf-8-sig",
        )

    comparison = pd.DataFrame(
        [
            {
                "level": level_name,
                "metric": "accuracy",
                "before_holdout": raw_holdout_metrics.get(
                    "accuracy"
                ),
                "after_holdout": corrected_holdout_metrics.get(
                    "accuracy"
                ),
            },
            {
                "level": level_name,
                "metric": "macro_f1",
                "before_holdout": raw_holdout_metrics.get(
                    "macro_f1"
                ),
                "after_holdout": corrected_holdout_metrics.get(
                    "macro_f1"
                ),
            },
            {
                "level": level_name,
                "metric": "mean_iou",
                "before_holdout": raw_holdout_metrics.get(
                    "mean_iou"
                ),
                "after_holdout": corrected_holdout_metrics.get(
                    "mean_iou"
                ),
            },
            {
                "level": level_name,
                "metric": "kappa",
                "before_holdout": raw_holdout_metrics.get(
                    "kappa"
                ),
                "after_holdout": corrected_holdout_metrics.get(
                    "kappa"
                ),
            },
        ]
    )

    comparison.to_csv(
        report_dir / "comparacion_antes_despues.csv",
        index=False,
        encoding="utf-8-sig",
    )

    try:
        with pd.ExcelWriter(
            report_dir / "informe_armonizacion.xlsx",
            engine="openpyxl",
        ) as writer:
            mapping_df.to_excel(
                writer,
                sheet_name="mapeo_codigos",
                index=False,
            )
            comparison.to_excel(
                writer,
                sheet_name="comparacion",
                index=False,
            )
            corrected_holdout_per_class.to_excel(
                writer,
                sheet_name="metricas_holdout",
                index=False,
            )
            corrected_holdout_confusion.to_excel(
                writer,
                sheet_name="confusion_holdout",
            )
    except Exception as error:
        warnings.warn(
            f"No se pudo crear el Excel de armonización: {error}",
            RuntimeWarning,
        )

    return {
        "mapping_summary": mapping_summary,
        "raw_holdout_metrics": raw_holdout_metrics,
        "corrected_holdout_metrics": corrected_holdout_metrics,
        "corrected_full_metrics": full_metrics,
        "report_dir": str(report_dir),
    }


# =============================================================================
# 5. MÉTRICAS GLOBALES Y POR CLASE
# =============================================================================
def metrics_from_confusion(matrix: np.ndarray):
    matrix = matrix.astype(np.float64)
    tp = np.diag(matrix)
    support = matrix.sum(axis=1)
    predicted = matrix.sum(axis=0)
    total = matrix.sum()
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, support, out=np.zeros_like(tp), where=support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) > 0)
    iou = np.divide(tp, support + predicted - tp, out=np.zeros_like(tp), where=(support + predicted - tp) > 0)
    active = support > 0
    accuracy = float(tp.sum() / total) if total else 0.0
    expected = float((support * predicted).sum() / (total ** 2)) if total else 0.0
    kappa = float((accuracy - expected) / (1.0 - expected)) if total and expected < 1.0 else 0.0
    weights = support / total if total else np.zeros_like(support)
    global_metrics = {
        "accuracy": accuracy,
        "macro_precision": float(precision[active].mean()) if active.any() else 0.0,
        "macro_recall": float(recall[active].mean()) if active.any() else 0.0,
        "macro_f1": float(f1[active].mean()) if active.any() else 0.0,
        "mean_iou": float(iou[active].mean()) if active.any() else 0.0,
        "weighted_f1": float((f1 * weights).sum()) if total else 0.0,
        "kappa": kappa,
        "evaluated_pixels": int(total),
    }
    return global_metrics, precision, recall, f1, iou, support, predicted, tp


def evaluate_prediction(prediction_path: Path, reference_path: Path, idx_to_code, idx_to_name, output_dir: Path):
    if not reference_path.exists():
        warnings.warn(f"No existe la referencia; se omiten métricas: {reference_path}", RuntimeWarning)
        return None

    codes = np.asarray([idx_to_code[i] for i in sorted(idx_to_code)], dtype=np.int64)
    names = [idx_to_name[i] for i in sorted(idx_to_code)]
    code_to_index = {int(code): i for i, code in enumerate(codes)}
    confusion = np.zeros((len(codes), len(codes)), dtype=np.int64)
    unknown_reference = 0

    maximum_code = int(codes.max())
    lookup = np.full(maximum_code + 1, -1, dtype=np.int16)
    lookup[codes] = np.arange(len(codes), dtype=np.int16)

    with rasterio.open(prediction_path) as pred, rasterio.open(reference_path) as ref:
        with WarpedVRT(ref, crs=pred.crs, transform=pred.transform, width=pred.width, height=pred.height, resampling=Resampling.nearest, nodata=ref.nodata) as aligned:
            for _, window in pred.block_windows(1):
                y_pred = pred.read(1, window=window)
                y_true = aligned.read(1, window=window)
                valid = y_pred != OUTPUT_NODATA
                valid &= np.isfinite(y_true)
                if aligned.nodata is not None:
                    valid &= y_true != aligned.nodata
                if not valid.any():
                    continue
                true_values = y_true[valid].astype(np.int64)
                pred_values = y_pred[valid].astype(np.int64)
                known = np.isin(true_values, codes) & np.isin(pred_values, codes)
                unknown_reference += int((~np.isin(true_values, codes)).sum())
                true_values = true_values[known]
                pred_values = pred_values[known]
                if true_values.size == 0:
                    continue
                if (
                    true_values.max(initial=0) > maximum_code
                    or pred_values.max(initial=0) > maximum_code
                ):
                    raise ValueError(
                        "Se encontraron códigos fuera del rango definido "
                        "después de aplicar el filtro de clases conocidas."
                    )

                true_idx = lookup[true_values].astype(np.int64, copy=False)
                pred_idx = lookup[pred_values].astype(np.int64, copy=False)
                encoded = len(codes) * true_idx + pred_idx
                confusion += np.bincount(encoded, minlength=len(codes) ** 2).reshape(len(codes), len(codes))

    global_metrics, precision, recall, f1, iou, support, predicted, tp = metrics_from_confusion(confusion)
    global_metrics["unknown_reference_pixels"] = int(unknown_reference)
    global_metrics["evaluation_note"] = EVALUATION_NOTE

    missing_reference = [
        int(codes[index])
        for index, value in enumerate(support)
        if int(value) == 0
    ]
    global_metrics["classes_without_reference_pixels"] = missing_reference

    class_metrics = pd.DataFrame({
        "class_code": codes,
        "class_name": names,
        "support_reference": support.astype(np.int64),
        "support_prediction": predicted.astype(np.int64),
        "true_positive": tp.astype(np.int64),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "iou": iou,
    })
    confusion_df = pd.DataFrame(confusion, index=names, columns=names)

    row_totals = confusion.sum(axis=1, keepdims=True)
    confusion_normalized = np.divide(
        confusion,
        row_totals,
        out=np.zeros_like(confusion, dtype=np.float64),
        where=row_totals > 0,
    )
    confusion_normalized_df = pd.DataFrame(
        confusion_normalized,
        index=names,
        columns=names,
    )

    ensure_dir(output_dir)
    class_metrics.to_csv(
        output_dir / "metricas_por_clase_prediccion.csv",
        index=False,
        encoding="utf-8-sig",
    )
    confusion_df.to_csv(
        output_dir / "matriz_confusion_prediccion.csv",
        encoding="utf-8-sig",
    )
    confusion_normalized_df.to_csv(
        output_dir / "matriz_confusion_normalizada_prediccion.csv",
        encoding="utf-8-sig",
    )
    with (
        output_dir / "metricas_globales_prediccion.json"
    ).open("w", encoding="utf-8") as file:
        json.dump(global_metrics, file, indent=2, ensure_ascii=False)
    try:
        with pd.ExcelWriter(output_dir / "metricas_prediccion.xlsx", engine="openpyxl") as writer:
            pd.DataFrame([global_metrics]).to_excel(
                writer,
                sheet_name="globales",
                index=False,
            )
            class_metrics.to_excel(
                writer,
                sheet_name="por_clase",
                index=False,
            )
            confusion_df.to_excel(
                writer,
                sheet_name="confusion",
            )
            confusion_normalized_df.to_excel(
                writer,
                sheet_name="confusion_normalizada",
            )
    except Exception as exc:
        warnings.warn(f"No se pudo crear Excel: {exc}", RuntimeWarning)

    if missing_reference:
        warnings.warn(
            "La muestra de referencia no contiene píxeles para las clases: "
            f"{missing_reference}. Sus métricas no son interpretables.",
            RuntimeWarning,
        )

    if SAVE_CONFUSION_PLOT:
        figure, axis = plt.subplots(figsize=(10, 9))
        image = axis.imshow(confusion, interpolation="nearest")
        axis.set_title(f"Matriz de confusión - {MODEL_NAME}")
        axis.set_xlabel("Predicción")
        axis.set_ylabel("Referencia")
        axis.set_xticks(range(len(names)), names, rotation=90, fontsize=7)
        axis.set_yticks(range(len(names)), names, fontsize=7)
        figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
        figure.tight_layout()
        figure.savefig(
            output_dir / "matriz_confusion_prediccion.png",
            dpi=200,
            bbox_inches="tight",
        )
        plt.close(figure)
    return global_metrics


# =============================================================================
# 6. EJECUCIÓN
# =============================================================================
def main():
    ensure_dir(OUTPUT_DIR)

    print("=" * 80)
    print("RESNET-50 — PREDICCIÓN SEMÁNTICA CON ARMONIZACIÓN DE CÓDIGOS")
    print("MODO CPU ONLY — GPU/ROCm DESACTIVADOS")
    print("Orden: nivel_1 -> nivel_2")
    print("=" * 80)
    print(f"Entrada: {INPUT_RASTER}")
    print(f"Salida: {OUTPUT_DIR}")
    print(
        "Los códigos finales se ajustarán automáticamente a los valores "
        "numéricos presentes en cada ráster de referencia."
    )

    summaries = []

    for position, level_name in enumerate(
        LEVELS_TO_PREDICT,
        start=1,
    ):
        print("\n" + "=" * 80)
        print(
            f"NIVEL {position}/{len(LEVELS_TO_PREDICT)}: "
            f"{level_name}"
        )
        print("=" * 80)

        try:
            reference_path = REFERENCE_BY_LEVEL[level_name]
            if not reference_path.exists():
                # En Nivel 1 también se prueba automáticamente el nombre "(1)".
                alternate = (
                    reference_path.with_name(
                        "Raster_Almeidas_Nivel1(1).tif"
                    )
                    if level_name == "nivel_1"
                    else None
                )

                if alternate is not None and alternate.exists():
                    reference_path = alternate
                else:
                    raise FileNotFoundError(
                        f"No existe la referencia para {level_name}: "
                        f"{REFERENCE_BY_LEVEL[level_name]}"
                    )

            (
                raw_prediction_path,
                confidence_path,
                run_dir,
                idx_to_code,
                idx_to_name,
            ) = predict_level(level_name)

            model_code_to_name = {
                int(idx_to_code[index]): str(
                    idx_to_name.get(
                        index,
                        f"Clase interna {index}",
                    )
                )
                for index in sorted(idx_to_code)
            }

            if AUTO_HARMONIZE_REFERENCE_CODES:
                print(
                    f"\nDerivando correspondencia de códigos para "
                    f"{level_name}..."
                )

                mapping, mapping_df, mapping_summary = (
                    derive_reference_code_mapping(
                        raw_prediction_path,
                        reference_path,
                        model_code_to_name,
                        level_name,
                    )
                )

                print("\nMAPEO OBTENIDO:")
                print(
                    mapping_df[
                        [
                            "model_output_code",
                            "model_class_name",
                            "reference_code",
                            "mapping_purity",
                            "matched",
                        ]
                    ].to_string(index=False)
                )

                level_dir = ensure_dir(
                    OUTPUT_DIR / level_name
                )
                harmonized_path = (
                    level_dir
                    / f"{MODEL_NAME}_{level_name}_prediccion_ARMONIZADA.tif"
                )

                code_to_name = write_harmonized_prediction(
                    raw_prediction_path,
                    harmonized_path,
                    mapping,
                    model_code_to_name,
                )

                reports = save_harmonization_reports(
                    level_name=level_name,
                    level_dir=level_dir,
                    mapping_df=mapping_df,
                    mapping_summary=mapping_summary,
                    raw_prediction_path=raw_prediction_path,
                    harmonized_prediction_path=harmonized_path,
                    reference_path=reference_path,
                    model_code_to_name=model_code_to_name,
                )

                corrected = reports[
                    "corrected_holdout_metrics"
                ]
                before = reports[
                    "raw_holdout_metrics"
                ]

                print("\n" + "-" * 80)
                print("MÉTRICAS HOLDOUT — ANTES Y DESPUÉS")
                print("-" * 80)
                print(
                    f"Accuracy: "
                    f"{before.get('accuracy', np.nan):.4f} -> "
                    f"{corrected.get('accuracy', np.nan):.4f}"
                )
                print(
                    f"Macro F1: "
                    f"{before.get('macro_f1', np.nan):.4f} -> "
                    f"{corrected.get('macro_f1', np.nan):.4f}"
                )
                print(
                    f"Mean IoU: "
                    f"{before.get('mean_iou', np.nan):.4f} -> "
                    f"{corrected.get('mean_iou', np.nan):.4f}"
                )
                print(
                    f"Kappa: "
                    f"{before.get('kappa', np.nan):.4f} -> "
                    f"{corrected.get('kappa', np.nan):.4f}"
                )

                summaries.append(
                    {
                        "model": MODEL_NAME,
                        "level": level_name,
                        "status": "completed",
                        "run_dir": str(run_dir),
                        "raw_prediction": str(
                            raw_prediction_path
                        ),
                        "harmonized_prediction": str(
                            harmonized_path
                        ),
                        "confidence": str(
                            confidence_path
                        ),
                        "reference": str(
                            reference_path
                        ),
                        "mapping": json.dumps(
                            {
                                str(key): int(value)
                                for key, value in mapping.items()
                            },
                            ensure_ascii=False,
                        ),
                        "accuracy_before_holdout": before.get(
                            "accuracy"
                        ),
                        "accuracy_after_holdout": corrected.get(
                            "accuracy"
                        ),
                        "macro_f1_before_holdout": before.get(
                            "macro_f1"
                        ),
                        "macro_f1_after_holdout": corrected.get(
                            "macro_f1"
                        ),
                        "mean_iou_before_holdout": before.get(
                            "mean_iou"
                        ),
                        "mean_iou_after_holdout": corrected.get(
                            "mean_iou"
                        ),
                        "kappa_before_holdout": before.get(
                            "kappa"
                        ),
                        "kappa_after_holdout": corrected.get(
                            "kappa"
                        ),
                        "mapping_report_dir": reports[
                            "report_dir"
                        ],
                    }
                )

            else:
                summaries.append(
                    {
                        "model": MODEL_NAME,
                        "level": level_name,
                        "status": "completed_raw_only",
                        "run_dir": str(run_dir),
                        "raw_prediction": str(
                            raw_prediction_path
                        ),
                        "confidence": str(
                            confidence_path
                        ),
                    }
                )

        except KeyboardInterrupt:
            print(
                f"\nEjecución interrumpida durante {level_name}. "
                "Los temporales de predicción se conservaron."
            )
            summaries.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "interrupted",
                }
            )
            break

        except Exception as error:
            print(
                f"\nNo fue posible completar {level_name}: "
                f"{error}"
            )
            traceback.print_exc()
            summaries.append(
                {
                    "model": MODEL_NAME,
                    "level": level_name,
                    "status": "error",
                    "error": str(error),
                }
            )
            print(
                "Se intentará continuar con el siguiente nivel."
            )

        finally:
            safe_cuda_cleanup()

        if level_name != LEVELS_TO_PREDICT[-1]:
            print(
                "\nNivel 1 finalizado. Liberando memoria antes de nivel 2..."
            )
            gc.collect()
            time.sleep(10)
            print("Continuando con nivel 2...")

    summary_df = pd.DataFrame(summaries)

    summary_csv = (
        OUTPUT_DIR
        / "RESUMEN_PREDICCION_ARMONIZADA_AMBOS_NIVELES.csv"
    )
    summary_excel = (
        OUTPUT_DIR
        / "RESUMEN_PREDICCION_ARMONIZADA_AMBOS_NIVELES.xlsx"
    )

    summary_df.to_csv(
        summary_csv,
        index=False,
        encoding="utf-8-sig",
    )

    try:
        summary_df.to_excel(
            summary_excel,
            index=False,
            engine="openpyxl",
        )
    except Exception as error:
        warnings.warn(
            f"No se pudo crear el resumen Excel: {error}",
            RuntimeWarning,
        )

    print("\n" + "=" * 80)
    print("RESUMEN FINAL")
    print("=" * 80)
    print(summary_df)
    print(f"\nResumen: {summary_csv}")

    return summary_df


# Ejecución directa en una sola celda de Jupyter.
summary_df = main()


RESNET-50 — PREDICCIÓN SEMÁNTICA CON ARMONIZACIÓN DE CÓDIGOS
MODO CPU ONLY — GPU/ROCm DESACTIVADOS
Orden: nivel_1 -> nivel_2
Entrada: C:\Users\Ivan sua\anaconda3\Rasters_CLC\zona_estudio.tif
Salida: C:\Users\Ivan sua\anaconda3\Predicciones_ResNet50_Segmentacion_CORREGIDA_CPU
Los códigos finales se ajustarán automáticamente a los valores numéricos presentes en cada ráster de referencia.

NIVEL 1/2: nivel_1



Derivando correspondencia de códigos para nivel_1...

MAPEO OBTENIDO:
 model_output_code                 model_class_name  reference_code  mapping_purity  matched
                 1         2. Territorios agrícolas               2        0.903330     True
                 2 3. Bosques y áreas seminaturales               1        0.900926     True
                 3           5. Superficies de agua               5        0.787691     True
                 4  1. Territorios artificializados               3        0.416664     True
                 5                 4. Áreas húmedas               4        0.000000     True

--------------------------------------------------------------------------------
MÉTRICAS HOLDOUT — ANTES Y DESPUÉS
--------------------------------------------------------------------------------
Accuracy: 0.1041 -> 0.8810
Macro F1: 0.0544 -> 0.6910
Mean IoU: 0.0288 -> 0.5609
Kappa: -0.4006 -> 0.7111

Nivel 1 finalizado. Liberando memoria antes de nivel 2...
Continua


Derivando correspondencia de códigos para nivel_2...

MAPEO OBTENIDO:
 model_output_code                                             model_class_name  reference_code  mapping_purity  matched
                 1                             2.4 Áreas agrícolas heterogéneas               2        0.703186     True
                 2                3.2 Áreas con vegetación herbácea o arbustiva               1        0.803758     True
                 3                                                  3.1 Bosques               7        0.571614     True
                 4                                                   2.3 Pastos               3        0.591304     True
                 5                                      5.1 Aguas continentales              10        0.681335     True
                 6                                        1.1 Zonas urbanizadas               4        0.557353     True
                 7                 1.3 Zonas de extracción minera y escombreras   